Simulamos nulos

In [3]:
# =============================================================================
# 01_simular_nulos_corregido_v2.py
#
# Paso 1: Simulación de Datos Faltantes
# CORRECCIÓN: Convierte las fechas de UTC a la hora local de cada ciudad.
# =============================================================================

import pandas as pd
import numpy as np
import os
import random
import time

# --- 0. Carga de Datos y Preparación Inicial ---
try:
    file_path_load = os.path.join('CSVs', 'Air_Quality.csv')
    if not os.path.exists(file_path_load):
        script_dir_parent = os.path.dirname(os.getcwd())
        alt_file_path_load = os.path.join(script_dir_parent, 'CSVs', 'Air_Quality.csv')
        if os.path.exists(alt_file_path_load):
            file_path_load = alt_file_path_load
        else:
            raise FileNotFoundError("Air_Quality.csv no encontrado.")
    df_air_quality_raw = pd.read_csv(file_path_load)
    print(f"Datos cargados desde: {file_path_load}")
except FileNotFoundError:
    print(f"Error: No se encontró el archivo Air_Quality.csv. Creando DataFrame de ejemplo...");
    # ... (código de ejemplo) ...
    # (El código de ejemplo no se modificará, ya que es solo un fallback)
    df_air_quality_raw = pd.DataFrame() # Simplificado para el ejemplo

if not df_air_quality_raw.empty:
    # Asegurar que la columna 'Date' es de tipo datetime y está localizada en UTC
    df_air_quality_raw['Date'] = pd.to_datetime(df_air_quality_raw['Date'], utc=True)

    if 'CO2' in df_air_quality_raw.columns:
        df_air_quality_raw = df_air_quality_raw.drop(columns=['CO2'])
        print("Columna 'CO2' descartada.")

    # --- 1. Conversión de Zona Horaria ---
    print("\nConvirtiendo fechas de UTC a la hora local de cada ciudad...")

    city_timezones = {
        'Brasilia': 'America/Sao_Paulo', # GMT-3
        'Cairo': 'Africa/Cairo',         # GMT+2
        'Dubai': 'Asia/Dubai',           # GMT+4
        'London': 'Europe/London',       # GMT+0/GMT+1 (Pandas maneja DST, pero aquí será consistente)
        'New York': 'America/New_York',    # GMT-5 (EST)
        'Sydney': 'Australia/Sydney'     # GMT+10 (AEST)
    }

    # Reemplazar "New_York" por "New York" si es necesario
    df_air_quality_raw['City'] = df_air_quality_raw['City'].replace('New_York', 'New York')

    df_parts = []
    for city, tz in city_timezones.items():
        df_city = df_air_quality_raw[df_air_quality_raw['City'] == city].copy()
        if not df_city.empty:
            df_city['Date'] = df_city['Date'].dt.tz_convert(tz)
            df_parts.append(df_city)
        else:
            print(f"  Advertencia: No se encontraron datos para la ciudad '{city}' en el mapeo de zonas horarias.")
    
    # Concatenar de nuevo y ordenar por ciudad y fecha
    df_air_quality = pd.concat(df_parts).sort_values(by=['City', 'Date']).reset_index(drop=True)

    print("Conversión de zona horaria completada.")
    # Verificación visual
    print("\nEjemplo de fechas convertidas (Brasilia):")
    print(df_air_quality[df_air_quality['City'] == 'Brasilia'].head(2)[['Date', 'City']])
    print("\nEjemplo de fechas convertidas (Sydney):")
    print(df_air_quality[df_air_quality['City'] == 'Sydney'].head(2)[['Date', 'City']])
else:
    # Si usamos el fallback de datos de ejemplo
    df_air_quality = df_air_quality_raw


# --- Parámetros de Simulación ---
CONTAMINANT_COLS = ['CO', 'NO2', 'SO2', 'O3', 'PM2.5', 'PM10']
AQI_COL = ['AQI']
ALL_TARGET_COLS = CONTAMINANT_COLS + AQI_COL
N_REPETITIONS = 5

NAN_SIMULATION_SCENARIOS = [
    {'name': 'RandomRows_20percent', 'type': 'row_percentage', 'config': {'percentage': 0.20}},
    {'name': 'BlockRows_12h_20percent', 'type': 'block_rows', 'config': {'length': 12, 'percentage': 0.20}},
    {'name': 'BlockRows_24h_20percent', 'type': 'block_rows', 'config': {'length': 24, 'percentage': 0.20}},
    {'name': 'BlockRows_36h_20percent', 'type': 'block_rows', 'config': {'length': 36, 'percentage': 0.20}},
    {'name': 'BlockRows_48h_20percent', 'type': 'block_rows', 'config': {'length': 48, 'percentage': 0.20}},
    {'name': 'BlockRows_60h_20percent', 'type': 'block_rows', 'config': {'length': 60, 'percentage': 0.20}}
]

PROJECT_ROOT_DIR = os.getcwd()
SIMULATED_DATA_ROOT_DIR = os.path.join(PROJECT_ROOT_DIR, 'CSVs', 'simulados') # Nueva carpeta de salida
os.makedirs(SIMULATED_DATA_ROOT_DIR, exist_ok=True)

# --- Funciones Auxiliares para Simulación de NaNs (sin cambios)---
def introduce_nans_by_row(df_original_city_input, scenario_config_input, random_seed_input):
    np.random.seed(random_seed_input); random.seed(random_seed_input)
    df_with_nans = df_original_city_input.copy().reset_index(drop=True)
    n_rows = len(df_with_nans); row_indices_to_nan_positions = []
    scenario_type, config = scenario_config_input['type'], scenario_config_input['config']
    if scenario_type == 'row_percentage':
        num_rows_to_nan = int(n_rows * config['percentage'])
        if num_rows_to_nan > 0 and n_rows > 0: row_indices_to_nan_positions = random.sample(range(n_rows), min(num_rows_to_nan, n_rows))
    elif scenario_type == 'block_rows':
        block_length, percentage = config['length'], config.get('percentage', 0.20); total_rows_to_nan = int(n_rows * percentage)
        num_blocks = max(1, total_rows_to_nan // block_length) if block_length > 0 else 0
        used_indices, possible_starts = set(), list(range(n_rows - block_length + 1)); random.shuffle(possible_starts)
        blocks_created = 0
        for start_pos in possible_starts:
            if blocks_created >= num_blocks: break
            block_range = set(range(start_pos, start_pos + block_length))
            if not block_range.intersection(used_indices):
                row_indices_to_nan_positions.extend(list(block_range)); used_indices.update(block_range); blocks_created += 1
        row_indices_to_nan_positions = sorted(list(set(row_indices_to_nan_positions)))
    if row_indices_to_nan_positions:
        col_positions_to_nan = df_with_nans.columns.get_indexer(ALL_TARGET_COLS)
        col_positions_to_nan = [pos for pos in col_positions_to_nan if pos != -1]
        if col_positions_to_nan: df_with_nans.iloc[row_indices_to_nan_positions, col_positions_to_nan] = np.nan
    return df_with_nans

# --- Script Principal de Simulación ---
start_time_script = time.time()

if 'df_air_quality' not in locals() or df_air_quality.empty:
    print("Error: df_air_quality no está cargado o está vacío. Saliendo."); exit()

for city_name_original in df_air_quality['City'].unique():
    city_name_safe = city_name_original.replace(" ", "_")
    print(f"\nProcesando Ciudad: {city_name_original} (Directorio seguro: {city_name_safe})")
    city_sim_data_dir_root = os.path.join(SIMULATED_DATA_ROOT_DIR, city_name_safe)
    os.makedirs(city_sim_data_dir_root, exist_ok=True)

    df_city_original_for_sim = df_air_quality[df_air_quality['City'] == city_name_original].copy()
    if df_city_original_for_sim.empty: print(f"  No hay datos para la ciudad {city_name_original}."); continue
    if df_city_original_for_sim[ALL_TARGET_COLS].isnull().any().any():
        print(f"  Advertencia: NaNs preexistentes en {city_name_original}. Rellenando con ffill/bfill.");
        for col_fill in ALL_TARGET_COLS:
             if col_fill in df_city_original_for_sim.columns: df_city_original_for_sim[col_fill] = df_city_original_for_sim[col_fill].ffill().bfill()
        if df_city_original_for_sim[ALL_TARGET_COLS].isnull().any().any():
            print(f"  Error: No se pudieron rellenar NaNs en {city_name_original}. Omitiendo."); continue

    for scenario in NAN_SIMULATION_SCENARIOS:
        scenario_name_safe = scenario['name'].replace("%", "percent")
        print(f"  Generando escenario: {scenario_name_safe}")
        scenario_city_sim_data_dir_specific = os.path.join(city_sim_data_dir_root, scenario_name_safe)
        os.makedirs(scenario_city_sim_data_dir_specific, exist_ok=True)
        for rep in range(1, N_REPETITIONS + 1):
            current_seed = int(time.time() * 1000 + random.randint(0,100000) + rep + hash(city_name_original) + hash(scenario_name_safe)) % (2**32 - 1)
            df_city_with_nans = introduce_nans_by_row(df_city_original_for_sim, scenario, random_seed_input=current_seed)
            sim_filename = f"{city_name_safe}_{scenario_name_safe}_rep{rep}_seed{current_seed}_with_nans.csv"
            df_city_with_nans.to_csv(os.path.join(scenario_city_sim_data_dir_specific, sim_filename), index=False)
    print(f"  Simulación de escenarios para {city_name_original} completada.")

end_time_script = time.time()
print(f"\nProceso de simulación de NaNs completado.")
print(f"Archivos de datos simulados guardados en subcarpetas dentro de: {SIMULATED_DATA_ROOT_DIR}")
print(f"Tiempo total de ejecución: {(end_time_script - start_time_script) / 60:.2f} minutos.")

Datos cargados desde: CSVs\Air_Quality.csv
Columna 'CO2' descartada.

Convirtiendo fechas de UTC a la hora local de cada ciudad...
Conversión de zona horaria completada.

Ejemplo de fechas convertidas (Brasilia):
                        Date      City
0  2023-12-31 21:00:00-03:00  Brasilia
1  2023-12-31 22:00:00-03:00  Brasilia

Ejemplo de fechas convertidas (Sydney):
                            Date    City
43920  2024-01-01 11:00:00+11:00  Sydney
43921  2024-01-01 12:00:00+11:00  Sydney

Procesando Ciudad: Brasilia (Directorio seguro: Brasilia)
  Generando escenario: RandomRows_20percent
  Generando escenario: BlockRows_12h_20percent
  Generando escenario: BlockRows_24h_20percent
  Generando escenario: BlockRows_36h_20percent
  Generando escenario: BlockRows_48h_20percent
  Generando escenario: BlockRows_60h_20percent
  Simulación de escenarios para Brasilia completada.

Procesando Ciudad: Cairo (Directorio seguro: Cairo)
  Generando escenario: RandomRows_20percent
  Generando escena

Interpolación Lineal

In [4]:
# =============================================================================
# 02_imputacion_lineal.py
#
# Paso 2: Imputación con Interpolación Lineal y Evaluación
#
# Esta celda lee los archivos con NaNs generados, les aplica interpolación 
# lineal y evalúa su rendimiento.
# =============================================================================

import pandas as pd
import numpy as np
import os
import random
from sklearn.metrics import mean_absolute_error, mean_squared_error
import time
import glob

# --- 0. Carga de Datos ORIGINALES (Ground Truth) ---
try:
    file_path_load = os.path.join('CSVs', 'Air_Quality.csv')
    if not os.path.exists(file_path_load):
        script_dir_parent = os.path.dirname(os.getcwd())
        alt_file_path_load = os.path.join(script_dir_parent, 'CSVs', 'Air_Quality.csv')
        if os.path.exists(alt_file_path_load):
            file_path_load = alt_file_path_load
        else:
            raise FileNotFoundError("Air_Quality.csv no encontrado.")
    df_air_quality_ground_truth = pd.read_csv(file_path_load)
    print(f"Datos 'Ground Truth' cargados desde: {file_path_load}")
except FileNotFoundError:
    print(f"Error: No se encontró el archivo Air_Quality.csv. Es necesario para la evaluación.")
    exit()

df_air_quality_ground_truth['Date'] = pd.to_datetime(df_air_quality_ground_truth['Date'])
if 'CO2' in df_air_quality_ground_truth.columns:
    df_air_quality_ground_truth = df_air_quality_ground_truth.drop(columns=['CO2'])

# --- Parámetros de Imputación y Evaluación ---
CONTAMINANT_COLS = ['CO', 'NO2', 'SO2', 'O3', 'PM2.5', 'PM10']
AQI_COL = ['AQI']
ALL_TARGET_COLS = CONTAMINANT_COLS + AQI_COL

# --- Lista de escenarios a procesar (DEBE COINCIDIR CON LA CELDA 1) ---
NAN_SIMULATION_SCENARIOS = [
    {'name': 'RandomRows_20percent', 'type': 'row_percentage', 'config': {'percentage': 0.20}},
    {'name': 'BlockRows_12h_20percent', 'type': 'block_rows', 'config': {'length': 12, 'percentage': 0.20}},
    {'name': 'BlockRows_24h_20percent', 'type': 'block_rows', 'config': {'length': 24, 'percentage': 0.20}},
    {'name': 'BlockRows_36h_20percent', 'type': 'block_rows', 'config': {'length': 36, 'percentage': 0.20}},
    {'name': 'BlockRows_48h_20percent', 'type': 'block_rows', 'config': {'length': 48, 'percentage': 0.20}},
    {'name': 'BlockRows_60h_20percent', 'type': 'block_rows', 'config': {'length': 60, 'percentage': 0.20}}
]

PROJECT_ROOT_DIR = os.getcwd()
SIMULATED_DATA_ROOT_DIR = os.path.join(PROJECT_ROOT_DIR, 'CSVs', 'simulados')
IMPUTATIONS_BASE_DIR = os.path.join(PROJECT_ROOT_DIR, 'imputaciones')
INTERPOLATION_METHOD_NAME = 'interpolacionLineal'
METRICS_OUTPUT_DIR = os.path.join(IMPUTATIONS_BASE_DIR, INTERPOLATION_METHOD_NAME, 'metricas')
IMPUTED_CSVS_OUTPUT_DIR = os.path.join(IMPUTATIONS_BASE_DIR, INTERPOLATION_METHOD_NAME, 'CSVs')

os.makedirs(METRICS_OUTPUT_DIR, exist_ok=True)
os.makedirs(IMPUTED_CSVS_OUTPUT_DIR, exist_ok=True)

# --- Funciones Auxiliares para Métricas ---
def calculate_metrics_with_nmae_nrmse(y_true_df_reset, y_pred_df_reset, nan_mask_for_metrics_df_reset):
    results = {}
    for col in ALL_TARGET_COLS:
        results[col] = {'MAE': np.nan, 'RMSE': np.nan, 'NRMAE_range': np.nan, 'NRMSE_range': np.nan}
        if col not in y_true_df_reset.columns or col not in y_pred_df_reset.columns or col not in nan_mask_for_metrics_df_reset.columns:
            continue

        true_col_vals = y_true_df_reset[col].values
        pred_col_vals = y_pred_df_reset[col].values
        mask_col_flags = nan_mask_for_metrics_df_reset[col].values

        true_at_nans = true_col_vals[mask_col_flags]
        pred_at_nans = pred_col_vals[mask_col_flags]

        if len(true_at_nans) == 0: continue
        valid_pred_mask = ~np.isnan(pred_at_nans)
        true_at_nans_valid = true_at_nans[valid_pred_mask]
        pred_at_nans_valid = pred_at_nans[valid_pred_mask]

        if len(true_at_nans_valid) == 0: continue
        try:
            mae = mean_absolute_error(true_at_nans_valid, pred_at_nans_valid)
            rmse = np.sqrt(mean_squared_error(true_at_nans_valid, pred_at_nans_valid))
            original_col_range = y_true_df_reset[col].max() - y_true_df_reset[col].min()
            
            nrmae_range = np.nan
            nrmse_range = np.nan
            if original_col_range > 1e-9:
                nrmae_range = mae / original_col_range
                nrmse_range = rmse / original_col_range

            results[col]['MAE'] = mae; results[col]['RMSE'] = rmse
            results[col]['NRMAE_range'] = nrmae_range; results[col]['NRMSE_range'] = nrmse_range
        except Exception as e: pass
    return results

# --- Script Principal de Imputación y Evaluación ---
all_detailed_metrics_results = []
start_time_script = time.time()

for city_name_original in df_air_quality_ground_truth['City'].unique():
    city_name_safe = city_name_original.replace(" ", "_").replace("-", "_")
    print(f"\nProcesando Ciudad: {city_name_original}")
    
    df_city_gt = df_air_quality_ground_truth[df_air_quality_ground_truth['City'] == city_name_original].copy()
    if df_city_gt[ALL_TARGET_COLS].isnull().any().any():
        for col_fill in ALL_TARGET_COLS:
            if col_fill in df_city_gt.columns: df_city_gt[col_fill] = df_city_gt[col_fill].ffill().bfill()
    if df_city_gt.isnull().values.any():
        print(f"  No se pudieron rellenar NaNs en el ground truth de {city_name_original}. Omitiendo ciudad.")
        continue
    df_city_original_ground_truth_reset = df_city_gt.reset_index(drop=True)

    sim_data_city_dir = os.path.join(SIMULATED_DATA_ROOT_DIR, city_name_safe)
    if not os.path.exists(sim_data_city_dir):
        print(f"  Directorio de datos simulados no encontrado para {city_name_original}. Omitiendo.")
        continue

    for scenario in NAN_SIMULATION_SCENARIOS:
        scenario_name_safe = scenario['name'].replace("%", "percent").replace(" ", "_")
        print(f"  Evaluando escenario: {scenario_name_safe}")
        
        sim_data_scenario_dir = os.path.join(sim_data_city_dir, scenario_name_safe)
        if not os.path.exists(sim_data_scenario_dir):
            print(f"    Directorio de escenario no encontrado: {sim_data_scenario_dir}. Omitiendo.")
            continue
            
        imputed_csv_scenario_dir = os.path.join(IMPUTED_CSVS_OUTPUT_DIR, city_name_safe, scenario_name_safe)
        os.makedirs(imputed_csv_scenario_dir, exist_ok=True)

        repetition_files = glob.glob(os.path.join(sim_data_scenario_dir, "*_with_nans.csv"))

        for sim_file_path in repetition_files:
            file_name = os.path.basename(sim_file_path)
            try:
                df_city_with_nans_reset = pd.read_csv(sim_file_path)
                rep_str = file_name.split('_rep')[-1].split('_')[0]
                seed_str = file_name.split('_seed')[-1].split('_')[0]
                rep = int(rep_str) if rep_str.isdigit() else 0
                current_seed = int(seed_str) if seed_str.isdigit() else 0
            except Exception as e:
                print(f"      Error cargando {file_name}: {e}")
                continue
            
            # 1. Realizar imputación
            df_city_imputed_linear_reset = df_city_with_nans_reset.copy()
            for col in ALL_TARGET_COLS:
                if col in df_city_imputed_linear_reset.columns:
                    df_city_imputed_linear_reset[col] = df_city_imputed_linear_reset[col].interpolate(
                        method='linear', limit_direction='both', limit_area=None)
            for col in ALL_TARGET_COLS:
                if col in df_city_imputed_linear_reset.columns:
                    df_city_imputed_linear_reset[col] = df_city_imputed_linear_reset[col].ffill().bfill()

            # 2. Guardar archivo imputado
            imputed_filename = file_name.replace('_with_nans.csv', f'_imputed_{INTERPOLATION_METHOD_NAME}.csv')
            df_city_imputed_linear_reset.to_csv(os.path.join(imputed_csv_scenario_dir, imputed_filename), index=False)

            # 3. Calcular métricas
            nan_mask = df_city_with_nans_reset[ALL_TARGET_COLS].isnull() & ~df_city_original_ground_truth_reset[ALL_TARGET_COLS].isnull()
            metrics_scores = calculate_metrics_with_nmae_nrmse(df_city_original_ground_truth_reset, df_city_imputed_linear_reset, nan_mask)

            # 4. Almacenar resultados detallados
            for var_name in ALL_TARGET_COLS:
                all_detailed_metrics_results.append({
                    'City': city_name_original, 'Scenario': scenario_name_safe, 'Repetition': rep,
                    'Seed': current_seed, 'Imputation_Method': INTERPOLATION_METHOD_NAME,
                    'Variable': var_name, 
                    'MAE': metrics_scores[var_name]['MAE'], 
                    'RMSE': metrics_scores[var_name]['RMSE'],
                    'NRMAE_range': metrics_scores[var_name]['NRMAE_range'],
                    'NRMSE_range': metrics_scores[var_name]['NRMSE_range']
                })
    print(f"  Procesamiento de escenarios para {city_name_original} completado.")

# --- Guardado de Métricas Resumidas ---
if all_detailed_metrics_results:
    df_all_detailed_metrics = pd.DataFrame(all_detailed_metrics_results)
    
    print("\nGenerando resúmenes de métricas por variable y ciudad...")
    grouped_by_city_var = df_all_detailed_metrics.groupby(['City', 'Variable'])[['MAE', 'RMSE', 'NRMAE_range', 'NRMSE_range']].mean().reset_index()
    grouped_by_city_var.rename(columns={'MAE': 'Avg_MAE', 'RMSE': 'Avg_RMSE', 'NRMAE_range': 'Avg_NRMAE_range', 'NRMSE_range': 'Avg_NRMSE_range'}, inplace=True)
    for city_name_original in grouped_by_city_var['City'].unique():
        city_name_safe = city_name_original.replace(" ", "_").replace("-", "_")
        city_metrics_df = grouped_by_city_var[grouped_by_city_var['City'] == city_name_original]
        variables_summary_path = os.path.join(METRICS_OUTPUT_DIR, f'metrics_variables_summary_{city_name_safe}.csv')
        city_metrics_df.to_csv(variables_summary_path, index=False)
        print(f"  Resumen de variables para {city_name_original} guardado en: {variables_summary_path}")
        if city_name_original == df_air_quality_ground_truth['City'].unique()[0]: print(city_metrics_df)

    print("\nGenerando resumen combinado de contaminantes...")
    df_contaminants_only = df_all_detailed_metrics[df_all_detailed_metrics['Variable'].isin(CONTAMINANT_COLS)]
    contaminants_combined_summary = df_contaminants_only.groupby('City')[['MAE', 'RMSE', 'NRMAE_range', 'NRMSE_range']].mean().reset_index()
    contaminants_combined_summary.rename(columns={'MAE': 'Avg_MAE_Conts', 'RMSE': 'Avg_RMSE_Conts', 'NRMAE_range': 'Avg_NRMAE_range_Conts', 'NRMSE_range': 'Avg_NRMSE_range_Conts'}, inplace=True)
    contaminants_summary_path = os.path.join(METRICS_OUTPUT_DIR, 'metrics_contaminants_combined_summary.csv')
    contaminants_combined_summary.to_csv(contaminants_summary_path, index=False)
    print(f"  Resumen combinado de contaminantes guardado en: {contaminants_summary_path}")
    print(contaminants_combined_summary)

    print("\nGenerando resumen combinado de AQI...")
    df_aqi_only = df_all_detailed_metrics[df_all_detailed_metrics['Variable'] == 'AQI']
    aqi_combined_summary = df_aqi_only.groupby('City')[['MAE', 'RMSE', 'NRMAE_range', 'NRMSE_range']].mean().reset_index()
    aqi_combined_summary.rename(columns={'MAE': 'Avg_MAE_AQI', 'RMSE': 'Avg_RMSE_AQI', 'NRMAE_range': 'Avg_NRMAE_range_AQI', 'NRMSE_range': 'Avg_NRMSE_range_AQI'}, inplace=True)
    aqi_summary_path = os.path.join(METRICS_OUTPUT_DIR, 'metrics_aqi_combined_summary.csv')
    aqi_combined_summary.to_csv(aqi_summary_path, index=False)
    print(f"  Resumen combinado de AQI guardado en: {aqi_summary_path}")
    print(aqi_combined_summary)
else:
    print("No se generaron resultados de métricas detalladas.")

end_time_script = time.time()
print(f"\nProceso de imputación y evaluación completado.")
print(f"Archivos de datos simulados leídos desde: {SIMULATED_DATA_ROOT_DIR}")
print(f"Archivos CSV imputados guardados en subcarpetas dentro de: {IMPUTED_CSVS_OUTPUT_DIR}")
print(f"Archivos de métricas resumidas guardados en: {METRICS_OUTPUT_DIR}")
print(f"Tiempo total de ejecución: {(end_time_script - start_time_script) / 60:.2f} minutos.")

Datos 'Ground Truth' cargados desde: CSVs\Air_Quality.csv

Procesando Ciudad: Brasilia
  Evaluando escenario: RandomRows_20percent
  Evaluando escenario: BlockRows_12h_20percent
  Evaluando escenario: BlockRows_24h_20percent
  Evaluando escenario: BlockRows_36h_20percent
  Evaluando escenario: BlockRows_48h_20percent
  Evaluando escenario: BlockRows_60h_20percent
  Procesamiento de escenarios para Brasilia completado.

Procesando Ciudad: Cairo
  Evaluando escenario: RandomRows_20percent
  Evaluando escenario: BlockRows_12h_20percent
  Evaluando escenario: BlockRows_24h_20percent
  Evaluando escenario: BlockRows_36h_20percent
  Evaluando escenario: BlockRows_48h_20percent
  Evaluando escenario: BlockRows_60h_20percent
  Procesamiento de escenarios para Cairo completado.

Procesando Ciudad: Dubai
  Evaluando escenario: RandomRows_20percent
  Evaluando escenario: BlockRows_12h_20percent
  Evaluando escenario: BlockRows_24h_20percent
  Evaluando escenario: BlockRows_36h_20percent
  Evaluan

SARIMA ORDERS

In [ ]:
# CELDA 1: BÚSQUEDA DE ÓRDENES ÓPTIMOS ARIMA/SARIMA (EJECUTAR SOLO UNA VEZ O CUANDO SEA NECESARIO)
import pandas as pd
import numpy as np
import os
import pmdarima as pm
import json
import warnings
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

# --- 0. Carga de Datos (Asegúrate de que df_air_quality esté cargado) ---
try:
    file_path_load = os.path.join('CSVs', 'Air_Quality.csv')
    if not os.path.exists(file_path_load):
        script_dir_parent = os.path.dirname(os.getcwd())
        alt_file_path_load = os.path.join(script_dir_parent, 'CSVs', 'Air_Quality.csv')
        if os.path.exists(alt_file_path_load): file_path_load = alt_file_path_load
        else: raise FileNotFoundError("Air_Quality.csv no encontrado.")
    df_air_quality = pd.read_csv(file_path_load)
    df_air_quality['Date'] = pd.to_datetime(df_air_quality['Date'])
    if 'CO2' in df_air_quality.columns: df_air_quality = df_air_quality.drop(columns=['CO2'])
    print(f"Datos cargados desde: {file_path_load} para Celda 1")
except FileNotFoundError:
    print("Error: df_air_quality no encontrado en Celda 1.")
    _example_data = [] # Datos de ejemplo mínimos
    _cities_example = ['Brasilia', 'Cairo']
    _start_date = pd.to_datetime('2024-01-01')
    for _city_ex in _cities_example:
        for i in range(24*7*4): # 4 semanas de datos horarios
            _date = _start_date + pd.Timedelta(hours=i)
            _example_data.append([_date, _city_ex, 200+np.random.rand()*100, 20+np.random.rand()*10, 40+np.random.rand()*20 ])
    df_air_quality = pd.DataFrame(_example_data, columns=['Date', 'City', 'CO', 'NO2', 'AQI'])
    print("DataFrame de ejemplo creado para Celda 1.")


# --- Parámetros ---
COLUMNS_FOR_ARIMA = ['CO', 'NO2', 'SO2', 'O3', 'PM2.5', 'PM10', 'AQI']
SEASONAL_PERIOD = 24

PROJECT_ROOT_DIR_ARIMA_SEARCH = os.getcwd()
# Directorio donde se guardarán los JSONs directamente
ORDERS_OUTPUT_DIR = os.path.join(PROJECT_ROOT_DIR_ARIMA_SEARCH, 'imputaciones', 'SARIMA_auto', 'model_orders_v2')
os.makedirs(ORDERS_OUTPUT_DIR, exist_ok=True) # Solo se crea la carpeta base 'model_orders'

print(f"--- INICIO DE BÚSQUEDA DE ÓRDENES ARIMA/SARIMA ---")
print(f"Los órdenes se guardarán directamente en: {ORDERS_OUTPUT_DIR}")

if 'df_air_quality' not in locals() or df_air_quality.empty:
    print("Error: df_air_quality no está definido o está vacío en Celda 1.")
    exit()

for city_name_original in df_air_quality['City'].unique():
    city_name_safe = city_name_original.replace(" ", "_").replace("-", "_")
    print(f"\nProcesando ciudad: {city_name_original}")

    df_city = df_air_quality[df_air_quality['City'] == city_name_original].copy()
    if df_city.empty:
        print(f"  No hay datos para {city_name_original}, omitiendo.")
        continue

    for col_name in COLUMNS_FOR_ARIMA:
        if col_name not in df_city.columns:
            print(f"  Variable {col_name} no encontrada en los datos de {city_name_original}, omitiendo.")
            continue

        print(f"  Buscando orden para variable: {col_name}...")
        series_data = df_city[col_name].dropna()

        if len(series_data) < 2 * SEASONAL_PERIOD:
            print(f"    Datos insuficientes para SARIMA ({len(series_data)} puntos). Intentando ARIMA no estacional.")
            try:
                auto_model = pm.auto_arima(series_data, start_p=1, start_q=1, max_p=5, max_q=5, d=None, seasonal=False, trace=False, error_action='ignore', suppress_warnings=True, stepwise=True)
                order = auto_model.order; seasonal_order = (0,0,0,0)
            except Exception as e:
                print(f"    Error en auto_arima no estacional para {col_name}: {e}. Usando (1,0,0).")
                order = (1,0,0); seasonal_order = (0,0,0,0)
        elif len(series_data) < 10:
             print(f"    Datos muy insuficientes (<10 puntos). Usando (1,0,0).")
             order = (1,0,0); seasonal_order = (0,0,0,0)
        else:
            try:
                auto_model = pm.auto_arima(series_data, start_p=1, start_q=1, max_p=5, max_q=5, m=SEASONAL_PERIOD, start_P=0, D=None, max_P=2, max_Q=2, seasonal=True, trace=False, error_action='ignore', suppress_warnings=True, stepwise=True)     
                order = auto_model.order; seasonal_order = auto_model.seasonal_order
                print(f"    Orden encontrado para {col_name}: {order}, Estacional: {seasonal_order}")
            except Exception as e:
                print(f"    Error en auto_arima para {col_name}: {e}. Usando (1,0,1)(1,0,1,{SEASONAL_PERIOD}).")
                order = (1,0,1); seasonal_order = (1,0,1,SEASONAL_PERIOD) if SEASONAL_PERIOD > 0 else (0,0,0,0)
        
        order_data = {'order': list(order), 'seasonal_order': list(seasonal_order)}
        # Guardar directamente en ORDERS_OUTPUT_DIR, sin subcarpeta de ciudad para el JSON
        file_path_order = os.path.join(ORDERS_OUTPUT_DIR, f'orders_{city_name_safe}_{col_name}.json')
        with open(file_path_order, 'w') as f:
            json.dump(order_data, f, indent=4)
        print(f"    Órdenes guardados en: {file_path_order}")

print(f"--- FIN DE BÚSQUEDA DE ÓRDENES ARIMA/SARIMA ---")

Datos originales (ground truth) cargados desde: CSVs\Air_Quality.csv
--- Fase 1: Encontrando/Cargando Órdenes SARIMA Óptimos ---
  Procesando ciudad para encontrar órdenes: Brasilia
    Procesando variable para auto_arima: CO
      Ejecutando auto_arima para Brasilia-CO (esto puede tardar)...
        Usando subconjunto de 2160 filas para auto_arima.
      auto_arima completado para Brasilia-CO: Order=(3, 0, 0), Seasonal=(1, 0, 1, 24)
    Procesando variable para auto_arima: NO2
      Ejecutando auto_arima para Brasilia-NO2 (esto puede tardar)...
        Usando subconjunto de 2160 filas para auto_arima.
      auto_arima completado para Brasilia-NO2: Order=(1, 1, 0), Seasonal=(1, 0, 0, 24)
    Procesando variable para auto_arima: SO2
      Ejecutando auto_arima para Brasilia-SO2 (esto puede tardar)...
        Usando subconjunto de 2160 filas para auto_arima.
      auto_arima completado para Brasilia-SO2: Order=(3, 1, 1), Seasonal=(1, 0, 0, 24)
    Procesando variable para auto_arima: O3


KeyboardInterrupt: 

Imputación SARIMA

In [5]:
# =============================================================================
# 02_imputacion_sarima.py (Versión con Target City Opcional)
# =============================================================================

import pandas as pd
import numpy as np
import os
import json
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error, mean_squared_error
import time
import warnings
import glob

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# --- Parámetros de Imputación y Evaluación ---
# CORRECCIÓN: Añadir parámetro de ciudad objetivo
TARGET_CITY_SARIMA = "Brasilia"  # Poner a None para ejecutar para TODAS las ciudades

COLUMNS_TO_IMPUTE_SARIMA = ['CO', 'NO2', 'SO2', 'O3', 'PM2.5', 'PM10', 'AQI']
N_REPETITIONS_SARIMA = 1

NAN_SIMULATION_SCENARIOS_SARIMA = [
    {'name': 'RandomRows_20percent'},
    {'name': 'BlockRows_12h_20percent'},
    {'name': 'BlockRows_24h_20percent'},
    {'name': 'BlockRows_36h_20percent'},
    {'name': 'BlockRows_48h_20percent'},
    {'name': 'BlockRows_60h_20percent'}
]

# --- Rutas de Entrada y Salida ---
PROJECT_ROOT_DIR_SARIMA_IMPUTE = os.getcwd()
SIMULATED_DATA_INPUT_ROOT_DIR = os.path.join(PROJECT_ROOT_DIR_SARIMA_IMPUTE, 'CSVs', 'simulados')
ORDERS_JSON_INPUT_DIR = os.path.join(PROJECT_ROOT_DIR_SARIMA_IMPUTE, 'imputaciones', 'SARIMA', 'model_orders')
SARIMA_METHOD_NAME = 'SARIMA'
IMPUTATIONS_SARIMA_BASE_DIR = os.path.join(PROJECT_ROOT_DIR_SARIMA_IMPUTE, 'imputaciones', SARIMA_METHOD_NAME)
METRICS_SARIMA_OUTPUT_DIR = os.path.join(IMPUTATIONS_SARIMA_BASE_DIR, 'metricas')
IMPUTED_CSVS_SARIMA_OUTPUT_DIR = os.path.join(IMPUTATIONS_SARIMA_BASE_DIR, 'CSVs')
os.makedirs(METRICS_SARIMA_OUTPUT_DIR, exist_ok=True)
os.makedirs(IMPUTED_CSVS_SARIMA_OUTPUT_DIR, exist_ok=True)

# --- Función de Imputación SARIMA (sin cambios) ---
def impute_sarima_kalman_smoother(series_with_nans, order_tuple, seasonal_order_tuple, original_series_for_bounds=None):
    imputed_series = series_with_nans.copy()
    original_nan_mask = imputed_series.isnull()
    if not original_nan_mask.any(): return series_with_nans
    min_len_heuristic = max(sum(order_tuple), sum(seasonal_order_tuple[:3])) + seasonal_order_tuple[3] if seasonal_order_tuple[3] > 0 else max(sum(order_tuple), sum(seasonal_order_tuple[:3])); min_len_heuristic = max(min_len_heuristic, 10)
    if len(imputed_series.dropna()) < min_len_heuristic + 5: return series_with_nans.interpolate(method='linear', limit_direction='both').ffill().bfill()
    sarima_failed_or_unstable = False
    try:
        model = SARIMAX(imputed_series, order=order_tuple, seasonal_order=seasonal_order_tuple, enforce_stationarity=False, enforce_invertibility=False, initialization='approximate_diffuse')
        results = model.fit(disp=False, maxiter=200)
        predictions_all = results.predict(start=imputed_series.index[0], end=imputed_series.index[-1])
        imputed_series.loc[original_nan_mask] = predictions_all[original_nan_mask]
        if imputed_series.isnull().any():
            temp_linear_fill = series_with_nans.interpolate(method='linear', limit_direction='both'); imputed_series = imputed_series.fillna(temp_linear_fill).ffill().bfill()
    except Exception as e: sarima_failed_or_unstable = True
    if sarima_failed_or_unstable: imputed_series = series_with_nans.interpolate(method='linear', limit_direction='both').ffill().bfill()
    if original_series_for_bounds is not None and original_nan_mask.any():
        clean_original_data = original_series_for_bounds.dropna()
        if not clean_original_data.empty:
            q01, q99 = clean_original_data.quantile(0.001), clean_original_data.quantile(0.999); iqr_val = clean_original_data.quantile(0.75) - clean_original_data.quantile(0.25)
            padding = max(1.0, 0.1 * abs(q99 - q01)) if (q99 - q01) != 0 else 1.0
            lower_bound, upper_bound = q01 - 3 * iqr_val - padding, q99 + 3 * iqr_val + padding
            extreme_values_mask = original_nan_mask & ((imputed_series < lower_bound) | (imputed_series > upper_bound))
            if extreme_values_mask.any():
                temp_linear_imputation = series_with_nans.interpolate(method='linear', limit_direction='both').ffill().bfill(); imputed_series.loc[extreme_values_mask] = temp_linear_imputation[extreme_values_mask]
    return imputed_series

# --- Función para calcular métricas (sin cambios) ---
def calculate_metrics_sarima_full(y_true_df_reset, y_pred_df_reset, nan_mask_for_metrics_df_reset):
    results = {}
    for col in COLUMNS_TO_IMPUTE_SARIMA:
        results[col] = {'MAE': np.nan, 'RMSE': np.nan, 'NRMAE_range': np.nan, 'NRMSE_range': np.nan}
        if col not in y_true_df_reset.columns or col not in y_pred_df_reset.columns: continue
        true_col_vals, pred_col_vals, mask_col_flags = y_true_df_reset[col].values, y_pred_df_reset[col].values, nan_mask_for_metrics_df_reset[col].values
        true_at_nans, pred_at_nans = true_col_vals[mask_col_flags], pred_col_vals[mask_col_flags]
        if len(true_at_nans) == 0: continue
        valid_pred_mask = ~np.isnan(pred_at_nans); true_at_nans_valid, pred_at_nans_valid = true_at_nans[valid_pred_mask], pred_at_nans[valid_pred_mask]
        if len(true_at_nans_valid) == 0: continue
        try:
            mae, rmse = mean_absolute_error(true_at_nans_valid, pred_at_nans_valid), np.sqrt(mean_squared_error(true_at_nans_valid, pred_at_nans_valid))
            original_col_range = y_true_df_reset[col].max() - y_true_df_reset[col].min()
            nrmae_range = np.nan if original_col_range < 1e-9 else mae / original_col_range; nrmse_range = np.nan if original_col_range < 1e-9 else rmse / original_col_range
            results[col].update({'MAE': mae, 'RMSE': rmse, 'NRMAE_range': nrmae_range, 'NRMSE_range': nrmse_range})
        except Exception: pass
    return results

# --- Script Principal de Imputación SARIMA ---
all_detailed_metrics_sarima_results = []
start_time_script_sarima = time.time()
try:
    file_path_load_orig = os.path.join('CSVs', 'Air_Quality.csv')
    if not os.path.exists(file_path_load_orig):
        script_dir_parent = os.path.dirname(os.getcwd()); alt_file_path_load_orig = os.path.join(script_dir_parent, 'CSVs', 'Air_Quality.csv')
        file_path_load_orig = alt_file_path_load_orig if os.path.exists(alt_file_path_load_orig) else ''
    if not file_path_load_orig: raise FileNotFoundError("Air_Quality.csv (original) no encontrado.")
    df_air_quality_original_gt = pd.read_csv(file_path_load_orig); df_air_quality_original_gt['Date'] = pd.to_datetime(df_air_quality_original_gt['Date'])
    if 'CO2' in df_air_quality_original_gt.columns: df_air_quality_original_gt = df_air_quality_original_gt.drop(columns=['CO2'])
    print(f"Datos originales GT cargados desde: {file_path_load_orig}")
except FileNotFoundError:
    print("Error crítico: Archivo 'Air_Quality.csv' de ground truth no encontrado."); exit()

# CORRECCIÓN: Lógica para seleccionar la(s) ciudad(es) a procesar
if TARGET_CITY_SARIMA:
    cities_to_process = [TARGET_CITY_SARIMA]
    if TARGET_CITY_SARIMA not in df_air_quality_original_gt['City'].unique():
        print(f"Advertencia: La ciudad objetivo '{TARGET_CITY_SARIMA}' no se encuentra en el dataset. No se procesará nada.")
        cities_to_process = []
else:
    cities_to_process = df_air_quality_original_gt['City'].unique()

for city_name_original_sarima in cities_to_process:
    city_name_safe_sarima = city_name_original_sarima.replace(" ", "_").replace("-", "_")
    print(f"\nProcesando Ciudad para Imputación SARIMA: {city_name_original_sarima}")

    city_sim_data_input_dir_root = os.path.join(SIMULATED_DATA_INPUT_ROOT_DIR, city_name_safe_sarima)
    city_imputed_csv_sarima_dir_root = os.path.join(IMPUTED_CSVS_SARIMA_OUTPUT_DIR, city_name_safe_sarima)
    os.makedirs(city_imputed_csv_sarima_dir_root, exist_ok=True)

    df_city_gt_for_metrics = df_air_quality_original_gt[df_air_quality_original_gt['City'] == city_name_original_sarima].copy()
    if df_city_gt_for_metrics.empty: print(f"No hay datos para {city_name_original_sarima}. Omitiendo."); continue
    if df_city_gt_for_metrics[COLUMNS_TO_IMPUTE_SARIMA].isnull().any().any():
        for col_fill_gt in COLUMNS_TO_IMPUTE_SARIMA:
            if col_fill_gt in df_city_gt_for_metrics.columns: df_city_gt_for_metrics[col_fill_gt] = df_city_gt_for_metrics[col_fill_gt].ffill().bfill()
    df_city_gt_for_metrics_reset = df_city_gt_for_metrics.reset_index(drop=True)

    for scenario_sarima in NAN_SIMULATION_SCENARIOS_SARIMA:
        scenario_name_safe_sarima = scenario_sarima['name'].replace("%", "percent").replace(" ", "_")
        print(f"  Escenario: {scenario_name_safe_sarima}")
        scenario_city_sim_data_input_dir = os.path.join(city_sim_data_input_dir_root, scenario_name_safe_sarima)
        scenario_city_imputed_csv_sarima_dir = os.path.join(city_imputed_csv_sarima_dir_root, scenario_name_safe_sarima); os.makedirs(scenario_city_imputed_csv_sarima_dir, exist_ok=True)

        for rep_s in range(1, N_REPETITIONS_SARIMA + 1):
            sim_files_pattern = os.path.join(scenario_city_sim_data_input_dir, f"{city_name_safe_sarima}_{scenario_name_safe_sarima}_rep{rep_s}_*_with_nans.csv")
            sim_files_in_dir = sorted(glob.glob(sim_files_pattern))
            if not sim_files_in_dir: continue
            
            sim_file_path_to_load = sim_files_in_dir[0]; current_seed_loaded = os.path.basename(sim_file_path_to_load).split('_seed')[-1].split('_with_nans.csv')[0]
            try: df_city_with_nans_loaded = pd.read_csv(sim_file_path_to_load)
            except FileNotFoundError: print(f"    Error: Archivo simulado no encontrado: {sim_file_path_to_load}"); continue
            
            df_city_imputed_sarima = df_city_with_nans_loaded.copy()
            mask_for_metrics_s = pd.DataFrame(False, index=df_city_gt_for_metrics_reset.index, columns=COLUMNS_TO_IMPUTE_SARIMA)
            for col_mask in COLUMNS_TO_IMPUTE_SARIMA:
                 if col_mask in df_city_with_nans_loaded.columns and col_mask in df_city_gt_for_metrics_reset.columns:
                    mask_for_metrics_s[col_mask] = df_city_with_nans_loaded[col_mask].isnull() & ~df_city_gt_for_metrics_reset[col_mask].isnull()

            for col_impute_s in COLUMNS_TO_IMPUTE_SARIMA:
                print(f"    Imputando columna: {col_impute_s}...")
                order_file_path = os.path.join(ORDERS_JSON_INPUT_DIR, f'orders_{city_name_safe_sarima}_{col_impute_s}.json')
                try:
                    with open(order_file_path, 'r') as f: orders = json.load(f)
                    order_tuple, seasonal_order_tuple = tuple(orders['order']), tuple(orders['seasonal_order'])
                except (FileNotFoundError, json.JSONDecodeError):
                    print(f"      Archivo de órdenes no encontrado/corrupto para {col_impute_s}. Usando (1,0,1)(0,0,0,0)."); order_tuple, seasonal_order_tuple = (1,0,1), (0,0,0,0)
                series_to_impute, original_clean_series_for_bounds = df_city_with_nans_loaded[col_impute_s].copy(), df_city_gt_for_metrics_reset[col_impute_s].copy()
                imputed_values_col = impute_sarima_kalman_smoother(series_to_impute, order_tuple, seasonal_order_tuple, original_clean_series_for_bounds)
                df_city_imputed_sarima[col_impute_s] = imputed_values_col
            
            imputed_sarima_filename = f"{city_name_safe_sarima}_{scenario_name_safe_sarima}_rep{rep_s}_seed{current_seed_loaded}_imputed_sarima.csv"
            df_city_imputed_sarima.to_csv(os.path.join(scenario_city_imputed_csv_sarima_dir, imputed_sarima_filename), index=False)
            metrics_scores = calculate_metrics_sarima_full(df_city_gt_for_metrics_reset, df_city_imputed_sarima, mask_for_metrics_s)
            
            for var_name in COLUMNS_TO_IMPUTE_SARIMA:
                all_detailed_metrics_sarima_results.append({'City': city_name_original_sarima, 'Scenario': scenario_name_safe_sarima, 'Repetition': rep_s, 'Seed': current_seed_loaded, 'Imputation_Method': SARIMA_METHOD_NAME, 'Variable': var_name, 'MAE': metrics_scores[var_name]['MAE'], 'RMSE': metrics_scores[var_name]['RMSE'], 'NRMAE_range': metrics_scores[var_name]['NRMAE_range'], 'NRMSE_range': metrics_scores[var_name]['NRMSE_range']})

# --- Guardado de Métricas Resumidas ---
if all_detailed_metrics_sarima_results:
    df_all_detailed_metrics = pd.DataFrame(all_detailed_metrics_sarima_results)
    
    print("\nGenerando resúmenes de métricas por variable y ciudad...")
    grouped_by_city_var = df_all_detailed_metrics.groupby(['City', 'Variable'])[['MAE', 'RMSE', 'NRMAE_range', 'NRMSE_range']].mean().reset_index()
    grouped_by_city_var.rename(columns={'MAE': 'Avg_MAE', 'RMSE': 'Avg_RMSE', 'NRMAE_range': 'Avg_NRMAE_range', 'NRMSE_range': 'Avg_NRMSE_range'}, inplace=True)
    for city_name in grouped_by_city_var['City'].unique():
        city_name_safe = city_name.replace(" ", "_").replace("-", "_")
        city_metrics_df = grouped_by_city_var[grouped_by_city_var['City'] == city_name]
        variables_summary_path = os.path.join(METRICS_SARIMA_OUTPUT_DIR, f'metrics_variables_summary_{city_name_safe}.csv')
        city_metrics_df.to_csv(variables_summary_path, index=False); print(f"  Resumen de variables para {city_name} guardado en: {variables_summary_path}")

    print("\nGenerando resumen combinado de contaminantes...")
    # CORRECCIÓN: Filtrar correctamente las columnas de contaminantes
    contaminant_cols = [c for c in COLUMNS_TO_IMPUTE_SARIMA if c != 'AQI']
    df_contaminants_only = df_all_detailed_metrics[df_all_detailed_metrics['Variable'].isin(contaminant_cols)]
    contaminants_combined_summary = df_contaminants_only.groupby('City')[['MAE', 'RMSE', 'NRMAE_range', 'NRMSE_range']].mean().reset_index()
    contaminants_combined_summary.rename(columns={'MAE': 'Avg_MAE_Conts', 'RMSE': 'Avg_RMSE_Conts', 'NRMAE_range': 'Avg_NRMAE_range_Conts', 'NRMSE_range': 'Avg_NRMSE_range_Conts'}, inplace=True)
    contaminants_summary_path = os.path.join(METRICS_SARIMA_OUTPUT_DIR, 'metrics_contaminants_combined_summary.csv')
    contaminants_combined_summary.to_csv(contaminants_summary_path, index=False); print(f"  Resumen contaminantes guardado: {contaminants_summary_path}"); print(contaminants_combined_summary)

    print("\nGenerando resumen combinado de AQI...")
    df_aqi_only = df_all_detailed_metrics[df_all_detailed_metrics['Variable'] == 'AQI']
    aqi_combined_summary = df_aqi_only.groupby('City')[['MAE', 'RMSE', 'NRMAE_range', 'NRMSE_range']].mean().reset_index()
    aqi_combined_summary.rename(columns={'MAE': 'Avg_MAE_AQI', 'RMSE': 'Avg_RMSE_AQI', 'NRMAE_range': 'Avg_NRMAE_range_AQI', 'NRMSE_range': 'Avg_NRMSE_range_AQI'}, inplace=True)
    aqi_summary_path = os.path.join(METRICS_SARIMA_OUTPUT_DIR, 'metrics_aqi_combined_summary.csv')
    aqi_combined_summary.to_csv(aqi_summary_path, index=False); print(f"  Resumen AQI guardado: {aqi_summary_path}"); print(aqi_combined_summary)

    global_detailed_s_file = os.path.join(METRICS_SARIMA_OUTPUT_DIR, f'ALL_metrics_detailed_{SARIMA_METHOD_NAME}.csv')
    df_all_detailed_metrics.to_csv(global_detailed_s_file, index=False); print(f"\nMétricas detalladas globales de SARIMA guardadas en: {global_detailed_s_file}")
else:
    print("\nNo se generaron resultados de métricas detalladas.")

end_time_script_sarima = time.time()
print(f"\nProceso SARIMA completado. CSVs: {IMPUTED_CSVS_SARIMA_OUTPUT_DIR}, Métricas: {METRICS_SARIMA_OUTPUT_DIR}")
print(f"Tiempo SARIMA: {(end_time_script_sarima - start_time_script_sarima) / 60:.2f} min.")

Datos originales GT cargados desde: CSVs\Air_Quality.csv

Procesando Ciudad para Imputación SARIMA: Brasilia
  Escenario: RandomRows_20percent
    Imputando columna: CO...
    Imputando columna: NO2...
    Imputando columna: SO2...
    Imputando columna: O3...
    Imputando columna: PM2.5...
    Imputando columna: PM10...
    Imputando columna: AQI...
  Escenario: BlockRows_12h_20percent
    Imputando columna: CO...
    Imputando columna: NO2...
    Imputando columna: SO2...
    Imputando columna: O3...
    Imputando columna: PM2.5...
    Imputando columna: PM10...
    Imputando columna: AQI...
  Escenario: BlockRows_24h_20percent
    Imputando columna: CO...
    Imputando columna: NO2...
    Imputando columna: SO2...
    Imputando columna: O3...
    Imputando columna: PM2.5...
    Imputando columna: PM10...
    Imputando columna: AQI...
  Escenario: BlockRows_36h_20percent
    Imputando columna: CO...
    Imputando columna: NO2...
    Imputando columna: SO2...
    Imputando columna: O

RANDOM FOREST

In [6]:
import pandas as pd
import numpy as np
import os
import random
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import time
import warnings
import glob

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# --- 0. Carga de Datos y Preparación Inicial ---
try:
    file_path_load = os.path.join('CSVs', 'Air_Quality.csv')
    if not os.path.exists(file_path_load):
        script_dir_parent = os.path.dirname(os.getcwd())
        alt_file_path_load = os.path.join(script_dir_parent, 'CSVs', 'Air_Quality.csv')
        if os.path.exists(alt_file_path_load): file_path_load = alt_file_path_load
        else: raise FileNotFoundError("Air_Quality.csv no encontrado.")
    df_air_quality_orig_load = pd.read_csv(file_path_load)
    print(f"Datos cargados desde: {file_path_load}")
except FileNotFoundError:
    print("Error: Air_Quality.csv no encontrado. Creando DataFrame de ejemplo...")
    _example_data = []
    _cities_example = ['Brasilia', 'Cairo', 'Dubai', 'London', 'New_York', 'Sydney']
    _start_date = pd.to_datetime('2024-01-01')
    for _city_ex in _cities_example:
        for i in range(200):
            _date = _start_date + pd.Timedelta(hours=i)
            _example_data.append([_date, _city_ex, 200+np.random.rand()*100, 20+np.random.rand()*10, 5+np.random.rand()*5, 50+np.random.rand()*30, 15+np.random.rand()*10, 30+np.random.rand()*20, 40+np.random.rand()*20])
    df_air_quality_orig_load = pd.DataFrame(_example_data, columns=['Date', 'City', 'CO', 'NO2', 'SO2', 'O3', 'PM2.5', 'PM10', 'AQI'])
    print("DataFrame de ejemplo creado.")

df_air_quality_orig_load['Date'] = pd.to_datetime(df_air_quality_orig_load['Date'])
if 'CO2' in df_air_quality_orig_load.columns:
    df_air_quality_orig_load = df_air_quality_orig_load.drop(columns=['CO2'])

# --- Parámetros ---
TARGET_CITY_RF = "Brasilia" # Poner a None para procesar todas las ciudades

CONTAMINANT_COLS = ['CO', 'NO2', 'SO2', 'O3', 'PM2.5', 'PM10']
AQI_COL = ['AQI']
ALL_TARGET_COLS = CONTAMINANT_COLS + AQI_COL

df_air_quality_orig_load['Hour'] = df_air_quality_orig_load['Date'].dt.hour
df_air_quality_orig_load['DayOfWeek'] = df_air_quality_orig_load['Date'].dt.dayofweek
df_air_quality_orig_load['Month'] = df_air_quality_orig_load['Date'].dt.month
TIME_FEATURES = ['Hour', 'DayOfWeek', 'Month']

N_LAGS = [1, 2, 24, 48]
LAG_FEATURES = []
df_with_features = df_air_quality_orig_load.copy()
for col_to_lag in ALL_TARGET_COLS:
    for lag_val in N_LAGS:
        lag_col_name = f"{col_to_lag}_lag{lag_val}"
        df_with_features[lag_col_name] = df_with_features.groupby('City')[col_to_lag].shift(lag_val)
        LAG_FEATURES.append(lag_col_name)

COLUMNS_FOR_IMPUTER_TRAINING_AND_TRANSFORM = ALL_TARGET_COLS + TIME_FEATURES + LAG_FEATURES

N_REPETITIONS = 1 # Ajustar según sea necesario
# CORRECCIÓN: Usar la nueva lista de escenarios
NAN_SIMULATION_SCENARIOS = [
    {'name': 'RandomRows_20percent'},
    {'name': 'BlockRows_12h_20percent'},
    {'name': 'BlockRows_24h_20percent'},
    {'name': 'BlockRows_36h_20percent'},
    {'name': 'BlockRows_48h_20percent'},
    {'name': 'BlockRows_60h_20percent'}
]

PROJECT_ROOT_DIR = os.getcwd()
SIMULATED_DATA_ROOT_DIR = os.path.join(PROJECT_ROOT_DIR, 'CSVs', 'simulados')

RF_IMPUTER_METHOD_NAME = 'RandomForestImputer'
IMPUTATIONS_RF_BASE_DIR = os.path.join(PROJECT_ROOT_DIR, 'imputaciones', RF_IMPUTER_METHOD_NAME)
METRICS_RF_OUTPUT_DIR = os.path.join(IMPUTATIONS_RF_BASE_DIR, 'metricas')
IMPUTED_CSVS_RF_OUTPUT_DIR = os.path.join(IMPUTATIONS_RF_BASE_DIR, 'CSVs')

os.makedirs(METRICS_RF_OUTPUT_DIR, exist_ok=True)
os.makedirs(IMPUTED_CSVS_RF_OUTPUT_DIR, exist_ok=True)

# --- Funciones Auxiliares (sin cambios) ---
def calculate_metrics_with_nmae_nrmse(y_true_df_reset, y_pred_df_reset, nan_mask_for_metrics_df_reset):
    results = {}
    for col in ALL_TARGET_COLS:
        results[col] = {'MAE': np.nan, 'RMSE': np.nan, 'NRMAE_range': np.nan, 'NRMSE_range': np.nan}
        if col not in y_true_df_reset.columns or col not in y_pred_df_reset.columns or col not in nan_mask_for_metrics_df_reset.columns: continue
        true_col_vals, pred_col_vals, mask_col_flags = y_true_df_reset[col].values, y_pred_df_reset[col].values, nan_mask_for_metrics_df_reset[col].values
        true_at_nans, pred_at_nans = true_col_vals[mask_col_flags], pred_col_vals[mask_col_flags]
        if len(true_at_nans) == 0: continue
        valid_pred_mask = ~np.isnan(pred_at_nans); true_at_nans_valid, pred_at_nans_valid = true_at_nans[valid_pred_mask], pred_at_nans[valid_pred_mask]
        if len(true_at_nans_valid) == 0: continue
        try:
            mae, rmse = mean_absolute_error(true_at_nans_valid, pred_at_nans_valid), np.sqrt(mean_squared_error(true_at_nans_valid, pred_at_nans_valid))
            original_col_full_series = y_true_df_reset[col].dropna()
            original_col_range = original_col_full_series.max() - original_col_full_series.min() if not original_col_full_series.empty else 0
            nrmae_range, nrmse_range = np.nan, np.nan
            if original_col_range > 1e-9: nrmae_range, nrmse_range = mae / original_col_range, rmse / original_col_range
            results[col].update({'MAE': mae, 'RMSE': rmse, 'NRMAE_range': nrmae_range, 'NRMSE_range': nrmse_range})
        except Exception: pass
    return results

# --- Script Principal ---
all_detailed_metrics_rf_results = []
script_start_time_rf = time.time()

if 'df_with_features' not in locals() or df_with_features.empty:
    print("Error: df_with_features no está definido. Saliendo."); exit()

# CORRECCIÓN: Lógica para seleccionar ciudades a procesar
if TARGET_CITY_RF:
    cities_to_process = [TARGET_CITY_RF]
    if TARGET_CITY_RF not in df_with_features['City'].unique():
        print(f"Advertencia: La ciudad objetivo '{TARGET_CITY_RF}' no se encuentra en el dataset. No se procesará nada.")
        cities_to_process = []
else:
    cities_to_process = df_with_features['City'].unique()

for city_name_original in cities_to_process:
    city_name_safe = city_name_original.replace(" ", "_").replace("-", "_")
    print(f"\nProcesando Ciudad para Imputación RF: {city_name_original}")

    df_city_original_features = df_with_features[df_with_features['City'] == city_name_original].copy()
    if df_city_original_features.empty: continue

    df_city_gt_eval = df_air_quality_orig_load[df_air_quality_orig_load['City'] == city_name_original].copy()
    if df_city_gt_eval[ALL_TARGET_COLS].isnull().any().any():
        for col_fill in ALL_TARGET_COLS:
             if col_fill in df_city_gt_eval.columns: df_city_gt_eval[col_fill] = df_city_gt_eval[col_fill].ffill().bfill()
    if df_city_gt_eval[ALL_TARGET_COLS].isnull().any().any():
        print(f"  Error: No se pudieron rellenar NaNs en el ground truth de {city_name_original}. Omitiendo."); continue
    df_city_original_ground_truth_reset = df_city_gt_eval.reset_index(drop=True)

    city_imputed_csv_rf_dir_root = os.path.join(IMPUTED_CSVS_RF_OUTPUT_DIR, city_name_safe)
    os.makedirs(city_imputed_csv_rf_dir_root, exist_ok=True)



    rf_estimator = RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1, max_depth=15, min_samples_split=5, min_samples_leaf=2, max_features='sqrt')
    iterative_imputer_rf = IterativeImputer(estimator=rf_estimator, max_iter=15, tol=1e-3, random_state=42, initial_strategy='mean', verbose=1)
    scaler = StandardScaler()
    
    
    
    data_for_fitting_imputer = df_city_original_features[COLUMNS_FOR_IMPUTER_TRAINING_AND_TRANSFORM].copy()
    print(f"  Ajustando Scaler e IterativeImputer para {city_name_original}...")
    start_fit_time = time.time()
    try:
        scaled_data_for_fitting = scaler.fit_transform(data_for_fitting_imputer)
        iterative_imputer_rf.fit(scaled_data_for_fitting)
        print(f"  Scaler e Imputer ajustados en {time.time() - start_fit_time:.2f} segundos.")
    except Exception as e_fit:
        print(f"  Error ajustando Scaler/Imputer: {e_fit}. Omitiendo ciudad."); continue

    for scenario in NAN_SIMULATION_SCENARIOS:
        scenario_name_safe = scenario['name'].replace("%", "percent").replace(" ", "_")
        print(f"    Escenario: {scenario_name_safe}")
        
        sim_data_scenario_dir = os.path.join(SIMULATED_DATA_ROOT_DIR, city_name_safe, scenario_name_safe)
        imputed_rf_scenario_dir = os.path.join(city_imputed_csv_rf_dir_root, scenario_name_safe); os.makedirs(imputed_rf_scenario_dir, exist_ok=True)
        if not os.path.exists(sim_data_scenario_dir):
            print(f"      Directorio de datos simulados no encontrado: {sim_data_scenario_dir}. Omitiendo."); continue

        for rep in range(1, N_REPETITIONS + 1):
            matching_files = sorted(glob.glob(os.path.join(sim_data_scenario_dir, f"{city_name_safe}_{scenario_name_safe}_rep{rep}_*_with_nans.csv")))
            if not matching_files: continue
            sim_file_path_to_load = matching_files[0]
            current_seed_loaded = os.path.basename(sim_file_path_to_load).split('_seed')[-1].split('_with_nans.csv')[0]

            try:
                df_city_with_nans_loaded_raw = pd.read_csv(sim_file_path_to_load)
                df_for_imputation_base = df_city_original_features.copy().reset_index(drop=True)
                df_city_with_nans_loaded_reset = df_city_with_nans_loaded_raw.reset_index(drop=True)
                for col_target in ALL_TARGET_COLS:
                    if col_target in df_city_with_nans_loaded_reset.columns:
                        df_for_imputation_base[col_target] = df_city_with_nans_loaded_reset[col_target]
            except Exception as e_load:
                print(f"      Error cargando/procesando {sim_file_path_to_load}: {e_load}."); continue
            
            temp_introduced_nan_mask_df = pd.DataFrame(False, index=df_city_original_ground_truth_reset.index, columns=ALL_TARGET_COLS)
            for col_mask_check in ALL_TARGET_COLS:
                 if col_mask_check in df_city_with_nans_loaded_reset.columns and col_mask_check in df_city_original_ground_truth_reset.columns:
                    temp_introduced_nan_mask_df[col_mask_check] = df_city_with_nans_loaded_reset[col_mask_check].isnull() & ~df_city_original_ground_truth_reset[col_mask_check].isnull()

            df_to_impute_for_rf = df_for_imputation_base[COLUMNS_FOR_IMPUTER_TRAINING_AND_TRANSFORM].copy()
            
            try:
                scaled_data_to_impute = scaler.transform(df_to_impute_for_rf)
                imputed_scaled_data_rf_array = iterative_imputer_rf.transform(scaled_data_to_impute)
                imputed_data_rf_array = scaler.inverse_transform(imputed_scaled_data_rf_array)
                df_imputed_features = pd.DataFrame(imputed_data_rf_array, columns=COLUMNS_FOR_IMPUTER_TRAINING_AND_TRANSFORM)
                df_final_imputed_rf = df_for_imputation_base[['Date', 'City']].copy()
                for col_final in ALL_TARGET_COLS + TIME_FEATURES:
                    if col_final in df_imputed_features.columns: df_final_imputed_rf[col_final] = df_imputed_features[col_final]
            except Exception as e_impute:
                print(f"      Error durante la imputación RF para rep {rep}: {e_impute}."); continue

            imputed_filename_rf = f"{city_name_safe}_{scenario_name_safe}_rep{rep}_seed{current_seed_loaded}_imputed_rf.csv"
            df_final_imputed_rf.to_csv(os.path.join(imputed_rf_scenario_dir, imputed_filename_rf), index=False)

            metrics_scores_rf = calculate_metrics_with_nmae_nrmse(df_city_original_ground_truth_reset[ALL_TARGET_COLS], df_final_imputed_rf[ALL_TARGET_COLS], temp_introduced_nan_mask_df)
            for var_name in ALL_TARGET_COLS:
                all_detailed_metrics_rf_results.append({'City': city_name_original, 'Scenario': scenario_name_safe, 'Repetition': rep, 'Seed': current_seed_loaded, 'Imputation_Method': RF_IMPUTER_METHOD_NAME, 'Variable': var_name, 'MAE': metrics_scores_rf[var_name]['MAE'], 'RMSE': metrics_scores_rf[var_name]['RMSE'], 'NRMAE_range': metrics_scores_rf[var_name]['NRMAE_range'], 'NRMSE_range': metrics_scores_rf[var_name]['NRMSE_range']})
    print(f"  Procesamiento de escenarios RF para {city_name_original} completado.")

# --- Guardado de Métricas Resumidas ---
if all_detailed_metrics_rf_results:
    df_all_detailed_metrics_rf = pd.DataFrame(all_detailed_metrics_rf_results)
    print("\nGenerando resúmenes de métricas RF por variable y ciudad...")
    grouped_by_city_var_rf = df_all_detailed_metrics_rf.groupby(['City', 'Variable'])[['MAE', 'RMSE', 'NRMAE_range', 'NRMSE_range']].mean().reset_index()
    grouped_by_city_var_rf.rename(columns={'MAE': 'Avg_MAE', 'RMSE': 'Avg_RMSE', 'NRMAE_range': 'Avg_NRMAE_range', 'NRMSE_range': 'Avg_NRMSE_range'}, inplace=True)
    for city_name_original_summary in grouped_by_city_var_rf['City'].unique():
        city_name_safe_rf_summary = city_name_original_summary.replace(" ", "_").replace("-", "_"); city_metrics_df_rf = grouped_by_city_var_rf[grouped_by_city_var_rf['City'] == city_name_original_summary]
        variables_summary_path_rf = os.path.join(METRICS_RF_OUTPUT_DIR, f'metrics_variables_summary_{city_name_safe_rf_summary}.csv')
        city_metrics_df_rf.to_csv(variables_summary_path_rf, index=False); print(f"  Resumen RF de variables para {city_name_original_summary} guardado en: {variables_summary_path_rf}")
        if city_name_original_summary == cities_to_process[0]: print(city_metrics_df_rf)

    print("\nGenerando resumen RF combinado de contaminantes...")
    df_contaminants_only_rf = df_all_detailed_metrics_rf[df_all_detailed_metrics_rf['Variable'].isin(CONTAMINANT_COLS)]
    contaminants_combined_summary_rf = df_contaminants_only_rf.groupby('City')[['MAE', 'RMSE', 'NRMAE_range', 'NRMSE_range']].mean().reset_index()
    contaminants_combined_summary_rf.rename(columns={'MAE': 'Avg_MAE_Conts', 'RMSE': 'Avg_RMSE_Conts', 'NRMAE_range': 'Avg_NRMAE_range_Conts', 'NRMSE_range': 'Avg_NRMSE_range_Conts'}, inplace=True)
    contaminants_summary_path_rf = os.path.join(METRICS_RF_OUTPUT_DIR, 'metrics_contaminants_combined_summary.csv')
    contaminants_combined_summary_rf.to_csv(contaminants_summary_path_rf, index=False); print(f"  Resumen RF combinado de contaminantes guardado en: {contaminants_summary_path_rf}"); print(contaminants_combined_summary_rf)
    
    print("\nGenerando resumen RF combinado de AQI...")
    df_aqi_only_rf = df_all_detailed_metrics_rf[df_all_detailed_metrics_rf['Variable'] == 'AQI']
    aqi_combined_summary_rf = df_aqi_only_rf.groupby('City')[['MAE', 'RMSE', 'NRMAE_range', 'NRMSE_range']].mean().reset_index()
    aqi_combined_summary_rf.rename(columns={'MAE': 'Avg_MAE_AQI', 'RMSE': 'Avg_RMSE_AQI', 'NRMAE_range': 'Avg_NRMAE_range_AQI', 'NRMSE_range': 'Avg_NRMSE_range_AQI'}, inplace=True)
    aqi_summary_path_rf = os.path.join(METRICS_RF_OUTPUT_DIR, 'metrics_aqi_combined_summary.csv')
    aqi_combined_summary_rf.to_csv(aqi_summary_path_rf, index=False); print(f"  Resumen RF combinado de AQI guardado en: {aqi_summary_path_rf}"); print(aqi_combined_summary_rf)
else:
    print("No se generaron resultados de métricas detalladas para Random Forest Imputer.")

script_end_time_rf = time.time()
print(f"\nProceso de Imputación con Random Forest completado. Tiempo total: {(script_end_time_rf - script_start_time_rf) / 60:.2f} minutos.")

Datos cargados desde: CSVs\Air_Quality.csv

Procesando Ciudad para Imputación RF: Brasilia
  Ajustando Scaler e IterativeImputer para Brasilia...
[IterativeImputer] Completing matrix with shape (8784, 38)
[IterativeImputer] Change: 6.442078832853756, scaled tolerance: 0.008004094305323934 
[IterativeImputer] Change: 3.7969897000559465, scaled tolerance: 0.008004094305323934 
[IterativeImputer] Change: 2.484989515146827, scaled tolerance: 0.008004094305323934 
[IterativeImputer] Change: 3.0486658070600754, scaled tolerance: 0.008004094305323934 
[IterativeImputer] Change: 2.9925322609138094, scaled tolerance: 0.008004094305323934 
[IterativeImputer] Change: 2.4537382930967846, scaled tolerance: 0.008004094305323934 
[IterativeImputer] Change: 2.349269439012307, scaled tolerance: 0.008004094305323934 
[IterativeImputer] Change: 1.6335229209468896, scaled tolerance: 0.008004094305323934 
[IterativeImputer] Change: 1.2187169412200536, scaled tolerance: 0.008004094305323934 
[IterativeImput

LSTM-VAE

In [10]:
import pandas as pd
import numpy as np
import os
import json
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import tensorflow as tf
import keras # Keras 3 de alto nivel
from keras import layers, Model, optimizers
from keras import backend as K
import time
import glob
import warnings
import shutil
import re
from unicodedata import normalize
import joblib
import random

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# --- Helper Function for Sanitizing TF Names ---
def sanitize_tf_name(name_str):
    s = normalize('NFKD', str(name_str)); s = s.encode('ascii', 'ignore'); s = s.decode('ascii') # Added str() for safety
    s = re.sub(r'[^a-zA-Z0-9_]', '_', s) # Reemplazar con _ en lugar de eliminar
    if s and s[0].isdigit() and not s.isdigit(): s = "n_" + s
    if not s: s = "sanitized_default_name"
    # Ensure name doesn't start or end with underscore if TF has strict rules, or just limit consecutive underscores
    s = re.sub(r'_+', '_', s) # Replace multiple underscores with single
    if s.startswith('_') and len(s) > 1: s = s[1:]
    if s.endswith('_') and len(s) > 1: s = s[:-1]
    if not s: s = "sanitized_default_name_v2" # If sanitization made it empty
    return s

# --- 0. Carga de Datos Originales ---
try:
    file_path_load = os.path.join('CSVs', 'Air_Quality.csv')
    if not os.path.exists(file_path_load):
        script_dir_parent = os.path.dirname(os.getcwd())
        alt_file_path_load = os.path.join(script_dir_parent, 'CSVs', 'Air_Quality.csv')
        if os.path.exists(alt_file_path_load): file_path_load = alt_file_path_load
        else: raise FileNotFoundError("Air_Quality.csv no encontrado.")
    df_air_quality_raw = pd.read_csv(file_path_load)
    print(f"Datos originales cargados desde: {file_path_load}")
except FileNotFoundError:
    print("Error: Air_Quality.csv no encontrado. Creando DataFrame de ejemplo...")
    _example_data = []
    _cities_example = ['Brasilia', 'Cairo', 'Dubai', 'London', 'New York', 'Sydney'] # 'New_York' a 'New York'
    _start_date = pd.to_datetime('2024-01-01')
    for _city_ex in _cities_example:
        for i in range(550): # Longitud similar a tus datos
            _date = _start_date + pd.Timedelta(hours=i)
            _example_data.append([_date, _city_ex, 200+np.random.rand()*100, 20+np.random.rand()*10, 5+np.random.rand()*5, 50+np.random.rand()*30, 15+np.random.rand()*10, 30+np.random.rand()*20, 40+np.random.rand()*20])
    df_air_quality_raw = pd.DataFrame(_example_data, columns=['Date', 'City', 'CO', 'NO2', 'SO2', 'O3', 'PM2.5', 'PM10', 'AQI'])
    print("DataFrame de ejemplo creado.")

# Asegurar que la columna 'Date' es de tipo datetime y está localizada en UTC
df_air_quality_raw['Date'] = pd.to_datetime(df_air_quality_raw['Date'], utc=True)
if 'CO2' in df_air_quality_raw.columns:
    df_air_quality_raw = df_air_quality_raw.drop(columns=['CO2'])

# Reemplazar "New_York" por "New York" si existe en los datos cargados
df_air_quality_raw['City'] = df_air_quality_raw['City'].replace('New_York', 'New York')

# --- 1. Conversión de Zona Horaria y Extracción de Características de Tiempo ---
print("\nConvirtiendo fechas de UTC a la hora local de cada ciudad y extrayendo características de tiempo...")
city_timezones = {
    'Brasilia': 'America/Sao_Paulo', 'Cairo': 'Africa/Cairo', 'Dubai': 'Asia/Dubai',
    'London': 'Europe/London', 'New York': 'America/New_York', 'Sydney': 'Australia/Sydney'
}
df_parts = []
TIME_FEATURES = ['Hour', 'DayOfWeek', 'Month'] # Define TIME_FEATURES before the loop

for city, tz in city_timezones.items():
    df_city_part = df_air_quality_raw[df_air_quality_raw['City'] == city].copy()
    if not df_city_part.empty:
        # Convert to local timezone
        df_city_part['Date'] = df_city_part['Date'].dt.tz_convert(tz)
        # Extract time features from the localized 'Date' column
        df_city_part['Hour'] = df_city_part['Date'].dt.hour
        df_city_part['DayOfWeek'] = df_city_part['Date'].dt.dayofweek
        df_city_part['Month'] = df_city_part['Date'].dt.month
        df_parts.append(df_city_part)

if not df_parts: # Handle case where no cities match or df_air_quality_raw was empty for specified cities
    print("Advertencia: No se procesaron datos para ninguna ciudad. `df_air_quality_full_original` estará vacío o no se creará como se esperaba.")
    # Create an empty DataFrame with expected columns to prevent downstream errors if possible
    # Or, exit/raise an error if this state is critical
    expected_cols = list(df_air_quality_raw.columns) + TIME_FEATURES
    df_air_quality_full_original = pd.DataFrame(columns=expected_cols)
else:
    df_air_quality_full_original = pd.concat(df_parts).sort_values(by=['City', 'Date']).reset_index(drop=True)

print("Conversión de zona horaria y extracción de características de tiempo completada.")
# The following lines are no longer needed here as features are created in the loop:
# df_air_quality_full_original['Hour'] = df_air_quality_full_original['Date'].dt.hour
# df_air_quality_full_original['DayOfWeek'] = df_air_quality_full_original['Date'].dt.dayofweek
# df_air_quality_full_original['Month'] = df_air_quality_full_original['Date'].dt.month

# --- Parámetros ---
TARGET_CITY_VAE = "Brasilia" # Poner a None para procesar todas las ciudades
CONTAMINANT_COLS = ['CO', 'NO2', 'SO2', 'O3', 'PM2.5', 'PM10']
AQI_COL = ['AQI']
ALL_TARGET_COLS_VAE = CONTAMINANT_COLS + AQI_COL

N_LAGS_VAE = 1
LATENT_DIM_VAE = 4
ENCODER_NEURONS_L1_VAE = 8
DECODER_NEURONS_L1_VAE = 8
VAE_INITIAL_LR = 5e-4; KL_LOSS_WEIGHT = 0.0001
VAE_BATCH_SIZE = 16
WFV_INITIAL_TRAIN_POINTS_MIN_RATIO = 0.1
WFV_VALIDATION_POINTS_MIN_RATIO = 0.05
WFV_STEP_POINTS_MIN_RATIO = 0.05
WFV_EPOCHS_PER_FOLD = 5 # Reducido para pruebas rápidas
WFV_ES_PATIENCE = 5
WFV_LR_PATIENCE = 3
MAX_WFV_FOLDS = 1
N_REPETITIONS_FOR_EVAL = 1 # Una repetición para pruebas

# Escenarios de NaNs a usar (deben coincidir con los generados por el script 01)
NAN_SIMULATION_SCENARIOS = [
    {'name': 'RandomRows_20percent'},
    #{'name': 'BlockRows_12h_20percent'},
    #{'name': 'BlockRows_24h_20percent'},
    #{'name': 'BlockRows_36h_20percent'},
    #{'name': 'BlockRows_48h_20percent'},
    #{'name': 'BlockRows_60h_20percent'}
]

PROJECT_ROOT_DIR = os.getcwd()
SIMULATED_DATA_ROOT_DIR = os.path.join(PROJECT_ROOT_DIR, 'CSVs', 'simulados') # LEER DE AQUÍ

LSTM_VAE_METHOD_NAME = 'LSTM_VAE_imputer_v4'
IMPUTATIONS_LSTM_VAE_BASE_DIR = os.path.join(PROJECT_ROOT_DIR, 'imputaciones', LSTM_VAE_METHOD_NAME)
METRICS_LSTM_VAE_OUTPUT_DIR = os.path.join(IMPUTATIONS_LSTM_VAE_BASE_DIR, 'metricas')
IMPUTED_CSVS_LSTM_VAE_OUTPUT_DIR = os.path.join(IMPUTATIONS_LSTM_VAE_BASE_DIR, 'CSVs')
MODELS_LSTM_VAE_OUTPUT_DIR = os.path.join(IMPUTATIONS_LSTM_VAE_BASE_DIR, 'model_weights')
os.makedirs(METRICS_LSTM_VAE_OUTPUT_DIR, exist_ok=True)
os.makedirs(IMPUTED_CSVS_LSTM_VAE_OUTPUT_DIR, exist_ok=True)
os.makedirs(MODELS_LSTM_VAE_OUTPUT_DIR, exist_ok=True)

# --- Funciones VAE (como las proporcionaste) ---
def sampling(args):
    z_mean, z_log_var = args; batch = tf.shape(z_mean)[0]; dim = tf.shape(z_mean)[1]
    epsilon = tf.random.normal(shape=(batch, dim), mean=0., stddev=1.0)
    return z_mean + keras.ops.exp(0.5 * z_log_var) * epsilon
class VAE(keras.Model):
    def __init__(self, encoder, decoder, kl_loss_weight_param, name="vae_model", **kwargs): # Corregido: __init__
        super().__init__(name=name, **kwargs)
        self.encoder_model = encoder; self.decoder_model = decoder
        self.kl_loss_weight = kl_loss_weight_param
        self.total_loss_tracker = keras.metrics.Mean(name="total_loss")
        self.reconstruction_loss_tracker = keras.metrics.Mean(name="reconstruction_loss")
        self.kl_loss_tracker = keras.metrics.Mean(name="kl_loss")
        self.mse_loss_fn = keras.losses.MeanSquaredError(reduction="sum_over_batch_size") # Changed from tf.keras to keras
    @property
    def metrics(self): return [self.total_loss_tracker, self.reconstruction_loss_tracker, self.kl_loss_tracker]
    def call(self, inputs, training=False):
        z_mean, z_log_var, z = self.encoder_model(inputs, training=training)
        return self.decoder_model(z, training=training)
    def train_step(self, data):
        x_input = data[0] if isinstance(data, tuple) else data; target_for_loss = x_input[:, :, 0:1] # Assumes target is the first feature
        with tf.GradientTape() as tape:
            z_mean_val, z_log_var_val, z_val = self.encoder_model(x_input, training=True)
            reconstruction = self.decoder_model(z_val, training=True)
            reconstruction_loss_val = self.mse_loss_fn(target_for_loss, reconstruction)
            kl_loss_val = -0.5 * keras.ops.mean(keras.ops.sum(1 + z_log_var_val - keras.ops.square(z_mean_val) - keras.ops.exp(z_log_var_val), axis=1))
            total_loss_val = reconstruction_loss_val + self.kl_loss_weight * kl_loss_val
        grads = tape.gradient(total_loss_val, self.trainable_weights); self.optimizer.apply_gradients(zip(grads, self.trainable_weights))
        self.total_loss_tracker.update_state(total_loss_val); self.reconstruction_loss_tracker.update_state(reconstruction_loss_val); self.kl_loss_tracker.update_state(kl_loss_val)
        return {m.name: m.result() for m in self.metrics}
    def test_step(self, data):
        x_input = data[0] if isinstance(data, tuple) else data; target_for_loss = x_input[:, :, 0:1] # Assumes target is the first feature
        z_mean_val, z_log_var_val, z_val = self.encoder_model(x_input, training=False)
        reconstruction = self.decoder_model(z_val, training=False)
        reconstruction_loss_val = self.mse_loss_fn(target_for_loss, reconstruction)
        kl_loss_val = -0.5 * keras.ops.mean(keras.ops.sum(1 + z_log_var_val - keras.ops.square(z_mean_val) - keras.ops.exp(z_log_var_val), axis=1))
        total_loss_val = reconstruction_loss_val + self.kl_loss_weight * kl_loss_val
        self.total_loss_tracker.update_state(total_loss_val); self.reconstruction_loss_tracker.update_state(reconstruction_loss_val); self.kl_loss_tracker.update_state(kl_loss_val)
        return {m.name: m.result() for m in self.metrics}
def create_vae_model_architecture(n_lags, n_features_input, latent_dim, enc_neurons, dec_neurons, var_name_for_layer_names="var"):
    safe_var_prefix = sanitize_tf_name(var_name_for_layer_names)
    enc_in = layers.Input(shape=(n_lags, n_features_input), name=f"{safe_var_prefix}_encoder_input"); enc_lstm = layers.LSTM(enc_neurons, activation='tanh', name=f"{safe_var_prefix}_encoder_lstm")(enc_in)
    z_m = layers.Dense(latent_dim, name=f"{safe_var_prefix}_z_mean")(enc_lstm); z_lv = layers.Dense(latent_dim, name=f"{safe_var_prefix}_z_log_var")(enc_lstm)
    z = layers.Lambda(sampling, name=f"{safe_var_prefix}_z_sampling_layer")([z_m, z_lv]); encoder = Model(enc_in, [z_m, z_lv, z], name=f"{safe_var_prefix}_encoder")
    dec_in = layers.Input(shape=(latent_dim,), name=f"{safe_var_prefix}_decoder_latent_input"); dec_rep = layers.RepeatVector(n_lags, name=f"{safe_var_prefix}_decoder_repeat_vector")(dec_in)
    dec_lstm = layers.LSTM(dec_neurons, activation='tanh', return_sequences=True, name=f"{safe_var_prefix}_decoder_lstm")(dec_rep)
    dec_out = layers.TimeDistributed(layers.Dense(1), name=f"{safe_var_prefix}_decoder_reconstruction_output")(dec_lstm); decoder = Model(dec_in, dec_out, name=f"{safe_var_prefix}_decoder")
    return encoder, decoder
def create_sequences_for_vae_training(target_series_1d_scaled, exog_df_scaled_nullable, n_lags_vae):
    X_seq_list = []; num_target_features = 1; num_exog_features = exog_df_scaled_nullable.shape[1] if exog_df_scaled_nullable is not None and not exog_df_scaled_nullable.empty else 0
    for i in range(len(target_series_1d_scaled) - n_lags_vae + 1):
        target_window_reshaped = target_series_1d_scaled[i : i + n_lags_vae].reshape(-1, num_target_features)
        if num_exog_features > 0:
            exog_window = exog_df_scaled_nullable.iloc[i : i + n_lags_vae].values
            if exog_window.shape[0] != n_lags_vae: continue
            combined_window = np.concatenate((target_window_reshaped, exog_window), axis=1)
        else: combined_window = target_window_reshaped
        X_seq_list.append(combined_window)
    if not X_seq_list: return np.array([]).reshape(0, n_lags_vae, num_target_features + num_exog_features)
    return np.array(X_seq_list)
def impute_segment_with_lstm_vae(target_series_1d_with_nans, exog_df_for_segment_scaled, vae_model, scaler_target, n_lags, n_total_features_vae_input, fallback_value_unscaled):
    imputed_series_unscaled = target_series_1d_with_nans.copy().flatten(); nan_indices_in_segment = np.where(np.isnan(imputed_series_unscaled))[0]
    num_target_features, num_exog_features_expected = 1, n_total_features_vae_input - 1
    for idx_nan in nan_indices_in_segment:
        if idx_nan < n_lags: imputed_series_unscaled[idx_nan] = imputed_series_unscaled[idx_nan-1] if idx_nan > 0 and not np.isnan(imputed_series_unscaled[idx_nan-1]) else fallback_value_unscaled; continue
        current_target_window_unscaled = imputed_series_unscaled[idx_nan - n_lags : idx_nan].copy(); s_temp_target = pd.Series(current_target_window_unscaled).ffill().bfill()
        if s_temp_target.isnull().any(): s_temp_target.fillna(fallback_value_unscaled, inplace=True)
        current_target_window_scaled = scaler_target.transform(s_temp_target.values.reshape(-1, 1)); current_exog_window_scaled = np.array([]).reshape(n_lags,0)
        if num_exog_features_expected > 0 and exog_df_for_segment_scaled is not None and not exog_df_for_segment_scaled.empty: # Added check for not empty
            start_exog_window, end_exog_window = idx_nan - n_lags, idx_nan
            if start_exog_window >=0 and end_exog_window <= len(exog_df_for_segment_scaled): current_exog_window_scaled = exog_df_for_segment_scaled.iloc[start_exog_window:end_exog_window].values
            else: current_exog_window_scaled = np.zeros((n_lags, num_exog_features_expected)) # Fallback if exog window is out of bounds
        elif num_exog_features_expected > 0: current_exog_window_scaled = np.zeros((n_lags, num_exog_features_expected))
        vae_input_sequence_scaled = np.concatenate((current_target_window_scaled, current_exog_window_scaled), axis=1) if num_exog_features_expected > 0 else current_target_window_scaled
        if vae_input_sequence_scaled.shape != (n_lags, n_total_features_vae_input): imputed_series_unscaled[idx_nan] = imputed_series_unscaled[idx_nan-1] if idx_nan > 0 and not np.isnan(imputed_series_unscaled[idx_nan-1]) else fallback_value_unscaled; continue
        vae_input_reshaped = vae_input_sequence_scaled.reshape(1, n_lags, n_total_features_vae_input)
        try:
            reconstructed_scaled_sequence = vae_model.predict(vae_input_reshaped, verbose=0); imputed_value_scaled = reconstructed_scaled_sequence[0, -1, 0]
            if np.isnan(imputed_value_scaled) or np.isinf(imputed_value_scaled): raise ValueError("Pred VAE NaN/Inf")
            imputed_series_unscaled[idx_nan] = scaler_target.inverse_transform(np.array([[imputed_value_scaled]]))[0,0]
        except Exception: imputed_series_unscaled[idx_nan] = imputed_series_unscaled[idx_nan-1] if idx_nan > 0 and not np.isnan(imputed_series_unscaled[idx_nan-1]) else fallback_value_unscaled
    return imputed_series_unscaled.reshape(-1,1)
def calculate_metrics_with_nmae_nrmse(y_true_df_reset, y_pred_df_reset, nan_mask_for_metrics_df_reset, cols_to_eval):
    results = {}
    for col in cols_to_eval:
        results[col] = {'MAE': np.nan, 'RMSE': np.nan, 'NRMAE_range': np.nan, 'NRMSE_range': np.nan}
        if col not in y_true_df_reset.columns or col not in y_pred_df_reset.columns or col not in nan_mask_for_metrics_df_reset.columns: continue
        true_col_vals, pred_col_vals, mask_col_flags = y_true_df_reset[col].values, y_pred_df_reset[col].values, nan_mask_for_metrics_df_reset[col].values
        # Ensure mask_col_flags is boolean
        mask_col_flags = mask_col_flags.astype(bool)
        true_at_nans, pred_at_nans = true_col_vals[mask_col_flags], pred_col_vals[mask_col_flags]
        if len(true_at_nans) == 0: continue
        valid_pred_mask = ~np.isnan(pred_at_nans); true_at_nans_valid, pred_at_nans_valid = true_at_nans[valid_pred_mask], pred_at_nans[valid_pred_mask]
        if len(true_at_nans_valid) == 0: continue
        try:
            mae, rmse = mean_absolute_error(true_at_nans_valid, pred_at_nans_valid), np.sqrt(mean_squared_error(true_at_nans_valid, pred_at_nans_valid))
            original_col_full_series = y_true_df_reset[col].dropna(); original_col_range = original_col_full_series.max() - original_col_full_series.min() if not original_col_full_series.empty else 0
            nrmae_range, nrmse_range = (mae / original_col_range if original_col_range > 1e-9 else np.nan), (rmse / original_col_range if original_col_range > 1e-9 else np.nan)
            results[col].update({'MAE': mae, 'RMSE': rmse, 'NRMAE_range': nrmae_range, 'NRMSE_range': nrmse_range})
        except Exception: pass
    return results

# --- Script Principal de Imputación con LSTM-VAE ---
all_detailed_metrics_lstm_vae_results = []
script_start_time_lstm_vae = time.time()

# CORRECCIÓN: Lógica para seleccionar ciudades a procesar
if TARGET_CITY_VAE:
    cities_to_process_vae = [TARGET_CITY_VAE]
    if TARGET_CITY_VAE not in df_air_quality_full_original['City'].unique():
        print(f"Advertencia: La ciudad objetivo '{TARGET_CITY_VAE}' no se encuentra en el dataset. No se procesará nada."); cities_to_process_vae = []
else:
    cities_to_process_vae = df_air_quality_full_original['City'].unique() if not df_air_quality_full_original.empty else []


for current_city_name_vae in cities_to_process_vae:
    print(f"\n=== PROCESANDO CIUDAD: {current_city_name_vae} ===")
    # df_city_original_for_wfv_unfiltered should use df_air_quality_full_original which now has TIME_FEATURES
    df_city_original_for_wfv_unfiltered = df_air_quality_full_original[df_air_quality_full_original['City'] == current_city_name_vae].copy()
    if df_city_original_for_wfv_unfiltered.empty: print(f"No hay datos para {current_city_name_vae}. Omitiendo."); continue

    trained_vaes = {}; scalers_target = {}
    # TIME_FEATURES are already in df_city_original_for_wfv_unfiltered
    df_exog_base_city = df_city_original_for_wfv_unfiltered[TIME_FEATURES].copy()
    for tf_col in TIME_FEATURES: df_exog_base_city[tf_col] = df_exog_base_city[tf_col].ffill().bfill() # ffill/bfill just in case, though they should be complete
    if df_exog_base_city.isnull().any().any(): print(f"Error: NaNs en TFs globales para {current_city_name_vae} después de ffill/bfill."); continue # Should not happen if data prep is correct

    scaler_exog_city = MinMaxScaler(feature_range=(0,1))
    # Ensure df_exog_base_city has data before fitting scaler
    if df_exog_base_city.empty:
        print(f"Advertencia: df_exog_base_city está vacía para {current_city_name_vae}. Omitiendo entrenamiento de VAEs para esta ciudad.");
        continue
    scaled_exog_global_city_df = pd.DataFrame(scaler_exog_city.fit_transform(df_exog_base_city), index=df_exog_base_city.index, columns=df_exog_base_city.columns)
    
    city_model_weights_root_dir = os.path.join(MODELS_LSTM_VAE_OUTPUT_DIR, sanitize_tf_name(current_city_name_vae))
    os.makedirs(city_model_weights_root_dir, exist_ok=True)
    joblib.dump(scaler_exog_city, os.path.join(city_model_weights_root_dir, f'scaler_exog_GLOBAL_{sanitize_tf_name(current_city_name_vae)}.joblib'))
    n_exog_features_global = scaled_exog_global_city_df.shape[1] if not scaled_exog_global_city_df.empty else 0
    n_features_input_global_vae = 1 + n_exog_features_global # 1 for target variable

    for target_var_to_impute in ALL_TARGET_COLS_VAE:
        K.clear_session(); base_seed_val = int(time.time_ns() / 1000) + hash(target_var_to_impute + current_city_name_vae)
        tf.random.set_seed(abs(base_seed_val + os.getpid() + 1) % (2**32 - 1)); np.random.seed(abs(base_seed_val + os.getpid() + 2) % (2**32 - 1)); random.seed(abs(base_seed_val + os.getpid() + 3) % (2**32 - 1))
        print(f"\n--- Preparando VAE para VARIABLE: {target_var_to_impute} (Ciudad: {current_city_name_vae}) ---")
        
        if target_var_to_impute not in df_city_original_for_wfv_unfiltered.columns:
            print(f"  Variable {target_var_to_impute} no encontrada en los datos de la ciudad. Omitiendo."); continue
        series_clean_target_wfv = df_city_original_for_wfv_unfiltered[target_var_to_impute].ffill().bfill()
        if series_clean_target_wfv.isnull().any(): print(f"  Serie {target_var_to_impute} con NaNs irreparables. Omitiendo."); continue
        if series_clean_target_wfv.empty: print(f"  Serie {target_var_to_impute} vacía. Omitiendo."); continue

        current_scaler_target_var = MinMaxScaler(feature_range=(0,1)); scaled_target_wfv_full_for_var = current_scaler_target_var.fit_transform(series_clean_target_wfv.values.reshape(-1,1)); scalers_target[target_var_to_impute] = current_scaler_target_var
        var_model_weights_dir = os.path.join(city_model_weights_root_dir, sanitize_tf_name(target_var_to_impute)); os.makedirs(var_model_weights_dir, exist_ok=True)
        joblib.dump(current_scaler_target_var, os.path.join(var_model_weights_dir, f'scaler_target_{sanitize_tf_name(target_var_to_impute)}.joblib'))
        print(f"  Iniciando WFV para VAE de {target_var_to_impute}...")
        total_clean_points_var = len(series_clean_target_wfv); wfv_init_train_len = int(total_clean_points_var * WFV_INITIAL_TRAIN_POINTS_MIN_RATIO)
        wfv_val_len = int(total_clean_points_var * WFV_VALIDATION_POINTS_MIN_RATIO); wfv_step_len = max(1, int(total_clean_points_var * WFV_STEP_POINTS_MIN_RATIO))
        if total_clean_points_var < wfv_init_train_len + wfv_val_len + N_LAGS_VAE + 10 or wfv_init_train_len < N_LAGS_VAE: # Added check for init_train_len
            print(f"  Datos insuficientes para WFV de {target_var_to_impute} (total: {total_clean_points_var}, train_init: {wfv_init_train_len}, val: {wfv_val_len}, lags: {N_LAGS_VAE})."); trained_vaes[target_var_to_impute] = None; continue
        
        current_train_end_idx_wfv, fold_num_wfv, best_val_loss_overall_wfv = wfv_init_train_len, 0, float('inf')
        vae_model_weights_path_final_var = os.path.join(var_model_weights_dir, f'vae_weights_final_{sanitize_tf_name(target_var_to_impute)}.weights.h5')
        fold_best_weights_path_temp_var = os.path.join(var_model_weights_dir, f'temp_fold_best_{sanitize_tf_name(target_var_to_impute)}.weights.h5')
        current_encoder_arch_vae, current_decoder_arch_vae = create_vae_model_architecture(N_LAGS_VAE, n_features_input_global_vae, LATENT_DIM_VAE, ENCODER_NEURONS_L1_VAE, DECODER_NEURONS_L1_VAE, var_name_for_layer_names=target_var_to_impute)
        
        # Check if pre-trained model exists and try to load it
        final_model_to_load = None
        if os.path.exists(vae_model_weights_path_final_var):
            print(f"  Intentando cargar VAE pre-entrenado para {target_var_to_impute}...")
            final_model_to_load = VAE(current_encoder_arch_vae, current_decoder_arch_vae, KL_LOSS_WEIGHT, name=f"vae_loaded_{sanitize_tf_name(target_var_to_impute)}")
            final_model_to_load.compile(optimizer=keras.optimizers.AdamW(learning_rate=VAE_INITIAL_LR))
            # Build the model by passing a dummy input, required before loading weights if model wasn't compiled with input_shape
            dummy_input_shape = (None, N_LAGS_VAE, n_features_input_global_vae)
            if not final_model_to_load.built:
                 try:
                    final_model_to_load.build(input_shape=dummy_input_shape)
                 except Exception as e_build: # Keras 3 might require build through a call
                    try:
                        final_model_to_load(tf.zeros((1, N_LAGS_VAE, n_features_input_global_vae)))
                    except Exception as e_call_build:
                         print(f"    Error building model for loading weights: {e_call_build}. Will retrain.")
                         final_model_to_load = None # Force retraining

            if final_model_to_load and final_model_to_load.built: # Check if built successfully
                try:
                    final_model_to_load.load_weights(vae_model_weights_path_final_var)
                    trained_vaes[target_var_to_impute] = final_model_to_load
                    print(f"  Modelo VAE para {target_var_to_impute} cargado exitosamente.")
                    continue # Skip WFV training
                except Exception as e_loadw_f:
                    print(f"  Error cargando VAE final para {target_var_to_impute}: {e_loadw_f}. Re-entrenando.")
                    final_model_to_load = None # Ensure it's None so WFV proceeds
            elif final_model_to_load and not final_model_to_load.built:
                 print(f"   Modelo VAE para {target_var_to_impute} no se pudo construir para cargar pesos. Re-entrenando.")
                 final_model_to_load = None


        # WFV loop
        while (current_train_end_idx_wfv + wfv_val_len <= total_clean_points_var) and (fold_num_wfv < MAX_WFV_FOLDS):
            fold_num_wfv += 1
            train_target_scaled_fold = scaled_target_wfv_full_for_var[:current_train_end_idx_wfv]
            val_target_scaled_fold = scaled_target_wfv_full_for_var[current_train_end_idx_wfv : current_train_end_idx_wfv + wfv_val_len]
            
            # Ensure exog features align with target series length for the fold
            train_exog_scaled_fold_df = scaled_exog_global_city_df.iloc[:current_train_end_idx_wfv] if not scaled_exog_global_city_df.empty else pd.DataFrame()
            val_exog_scaled_fold_df = scaled_exog_global_city_df.iloc[current_train_end_idx_wfv : current_train_end_idx_wfv + wfv_val_len] if not scaled_exog_global_city_df.empty else pd.DataFrame()

            print(f"    WFV Fold {fold_num_wfv} ({target_var_to_impute}): Train target {len(train_target_scaled_fold)}, Val target {len(val_target_scaled_fold)}")
            if not train_exog_scaled_fold_df.empty: print(f"    Train exog {train_exog_scaled_fold_df.shape}, Val exog {val_exog_scaled_fold_df.shape if not val_exog_scaled_fold_df.empty else 'empty'}")

            X_train_fold_vae = create_sequences_for_vae_training(train_target_scaled_fold.flatten(), train_exog_scaled_fold_df, N_LAGS_VAE)
            X_val_fold_vae = create_sequences_for_vae_training(val_target_scaled_fold.flatten(), val_exog_scaled_fold_df, N_LAGS_VAE)
            
            if X_train_fold_vae.shape[0] < VAE_BATCH_SIZE or X_val_fold_vae.shape[0] == 0:
                print(f"      Fold {fold_num_wfv} insuf. sec. (Train: {X_train_fold_vae.shape[0]}, Val: {X_val_fold_vae.shape[0]}). Avanzando fold o finalizando WFV.")
                current_train_end_idx_wfv += wfv_step_len
                if not (current_train_end_idx_wfv + wfv_val_len <= total_clean_points_var): # If next step is too large
                    break # Exit WFV loop if no more data for next fold
                continue

            fold_vae_model = VAE(current_encoder_arch_vae, current_decoder_arch_vae, KL_LOSS_WEIGHT, name=f"vae_fold{fold_num_wfv}_{sanitize_tf_name(target_var_to_impute)}")
            fold_vae_model.compile(optimizer=keras.optimizers.AdamW(learning_rate=VAE_INITIAL_LR, weight_decay=0.004, clipnorm=1.0))
            # Build model before loading weights or training
            dummy_input_shape_fold = (None, N_LAGS_VAE, n_features_input_global_vae)
            if not fold_vae_model.built:
                try: fold_vae_model.build(input_shape=dummy_input_shape_fold)
                except: fold_vae_model(tf.zeros((1, N_LAGS_VAE, n_features_input_global_vae)))


            if fold_num_wfv > 1 and os.path.exists(vae_model_weights_path_final_var): # Load best weights from previous fold/overall best
                try:
                    print(f"      Cargando pesos de {vae_model_weights_path_final_var} en fold_vae_model.")
                    fold_vae_model.load_weights(vae_model_weights_path_final_var)
                except Exception as e_lw: print(f"      Error cargando pesos globales en fold: {e_lw}")

            early_stopping_cb_fold_vae = keras.callbacks.EarlyStopping(monitor='val_total_loss', mode='min', patience=WFV_ES_PATIENCE, verbose=0, restore_best_weights=True)
            reduce_lr_cb = keras.callbacks.ReduceLROnPlateau(monitor='val_total_loss', mode='min', factor=0.5, patience=WFV_LR_PATIENCE, min_lr=1e-7, verbose=0)
            model_checkpoint_cb = keras.callbacks.ModelCheckpoint(filepath=fold_best_weights_path_temp_var, monitor='val_total_loss', mode='min', save_best_only=True, save_weights_only=True, verbose=0)
            callbacks_list = [early_stopping_cb_fold_vae, reduce_lr_cb, model_checkpoint_cb]

            fold_history_vae = fold_vae_model.fit(X_train_fold_vae, X_train_fold_vae, epochs=WFV_EPOCHS_PER_FOLD, batch_size=VAE_BATCH_SIZE, validation_data=(X_val_fold_vae, X_val_fold_vae), callbacks=callbacks_list, verbose=0)
            
            current_fold_val_loss = min(fold_history_vae.history.get('val_total_loss', [float('inf')])) # Get best loss from this fit
            print(f"      Fold {fold_num_wfv} Val Total Loss (best during this fit): {current_fold_val_loss:.4f}")

            # Check if the current fold's best model (restored by EarlyStopping or saved by ModelCheckpoint) is better than overall best
            if current_fold_val_loss < best_val_loss_overall_wfv:
                best_val_loss_overall_wfv = current_fold_val_loss
                print(f"      NUEVO MEJOR VAE ({target_var_to_impute}, loss: {best_val_loss_overall_wfv:.4f}). Guardando pesos globales.")
                # Save weights from the model (ES should have restored best) or from temp checkpoint
                if os.path.exists(fold_best_weights_path_temp_var) and early_stopping_cb_fold_vae.best == current_fold_val_loss:
                     # If ModelCheckpoint saved something and ES restored the same best, use the checkpointed one
                     # This logic can be tricky; simpler to just save fold_vae_model.get_weights() if ES restore_best_weights=True
                    try:
                        shutil.copyfile(fold_best_weights_path_temp_var, vae_model_weights_path_final_var)
                    except Exception as e_copy:
                        print(f"      Error copiando pesos temporales: {e_copy}. Guardando directamente del modelo.")
                        fold_vae_model.save_weights(vae_model_weights_path_final_var)
                else: # If ES restored weights, or MC didn't save, save current model weights
                    fold_vae_model.save_weights(vae_model_weights_path_final_var)

            if os.path.exists(fold_best_weights_path_temp_var): os.remove(fold_best_weights_path_temp_var)
            current_train_end_idx_wfv += wfv_step_len
            K.clear_session() # Clear session after each fold

        # After WFV, load the best overall weights into a final model instance
        final_vae_model_for_var = VAE(current_encoder_arch_vae, current_decoder_arch_vae, KL_LOSS_WEIGHT, name=f"vae_final_{sanitize_tf_name(target_var_to_impute)}")
        final_vae_model_for_var.compile(optimizer=keras.optimizers.AdamW(learning_rate=VAE_INITIAL_LR)) # Compile is needed
        dummy_input_shape_final = (None, N_LAGS_VAE, n_features_input_global_vae)
        if not final_vae_model_for_var.built:
            try: final_vae_model_for_var.build(input_shape=dummy_input_shape_final)
            except: final_vae_model_for_var(tf.zeros((1, N_LAGS_VAE, n_features_input_global_vae)))


        if os.path.exists(vae_model_weights_path_final_var):
            try:
                final_vae_model_for_var.load_weights(vae_model_weights_path_final_var)
                print(f"  Modelo VAE final para {target_var_to_impute} cargado/actualizado con mejores pesos de WFV.")
                trained_vaes[target_var_to_impute] = final_vae_model_for_var
            except Exception as e_lfw_final:
                print(f"  Error cargando pesos finales VAE {target_var_to_impute}: {e_lfw_final}. Modelo no disponible.")
                trained_vaes[target_var_to_impute] = None
        else:
            print(f"  Pesos VAE finales no encontrados para {target_var_to_impute} (ningún fold completado o guardado?). Modelo no disponible.")
            trained_vaes[target_var_to_impute] = None
        # End of WFV logic for one variable

    # --- Imputation Phase for the current city ---
    print(f"\n--- Iniciando Imputación para {current_city_name_vae} ---")
    df_city_original_gt_reset_for_metrics = df_air_quality_full_original[df_air_quality_full_original['City'] == current_city_name_vae].reset_index(drop=True)
    if df_city_original_gt_reset_for_metrics.empty :
        print(f"  No hay datos originales GT para {current_city_name_vae} para calcular métricas. Omitiendo imputación para esta ciudad.")
        continue
    city_name_safe_for_files = sanitize_tf_name(current_city_name_vae)

    for scenario_vae in NAN_SIMULATION_SCENARIOS:
        scenario_name_safe_vae = sanitize_tf_name(scenario_vae['name']) # Sanitize scenario name
        print(f"  Escenario: {scenario_name_safe_vae}")
        imputed_unified_scenario_dir = os.path.join(IMPUTED_CSVS_LSTM_VAE_OUTPUT_DIR, city_name_safe_for_files, scenario_name_safe_vae)
        os.makedirs(imputed_unified_scenario_dir, exist_ok=True)
        for rep_vae in range(1, N_REPETITIONS_FOR_EVAL + 1):
            sim_data_scenario_dir_for_rep = os.path.join(SIMULATED_DATA_ROOT_DIR, city_name_safe_for_files, scenario_name_safe_vae)
            # Glob pattern needs to match the actual filenames saved by script 01
            # Assuming script 01 saves with sanitized scenario name:
            matching_files_sim = sorted(glob.glob(os.path.join(sim_data_scenario_dir_for_rep, f"{city_name_safe_for_files}_{scenario_name_safe_vae}_rep{rep_vae}_*_with_nans.csv")))
            if not matching_files_sim:
                # Try original scenario name if sanitization caused mismatch
                original_scenario_name_from_list = scenario_vae['name'].replace("%", "percent") # As it was before sanitizing for dir name
                sim_data_scenario_dir_for_rep_orig_name = os.path.join(SIMULATED_DATA_ROOT_DIR, city_name_safe_for_files, original_scenario_name_from_list)
                matching_files_sim = sorted(glob.glob(os.path.join(sim_data_scenario_dir_for_rep_orig_name, f"{city_name_safe_for_files}_{original_scenario_name_from_list}_rep{rep_vae}_*_with_nans.csv")))

            if not matching_files_sim:
                print(f"    Archivo _with_nans.csv no encontrado para Ciudad: {city_name_safe_for_files}, Escenario: {scenario_name_safe_vae} (o {original_scenario_name_from_list}), Rep: {rep_vae}. Omitiendo.")
                continue

            sim_file_path_to_load = matching_files_sim[0]
            try:
                current_seed_loaded_vae = os.path.basename(sim_file_path_to_load).split('_seed')[-1].split('_with_nans.csv')[0]
            except IndexError:
                print(f"    Error extrayendo seed de nombre de archivo: {sim_file_path_to_load}. Usando 'unknown_seed'.")
                current_seed_loaded_vae = "unknown_seed"

            try:
                df_with_nans_all_vars_loaded = pd.read_csv(sim_file_path_to_load)
                # Convert 'Date' column to datetime, as it's read as string from CSV
                if 'Date' in df_with_nans_all_vars_loaded.columns:
                    df_with_nans_all_vars_loaded['Date'] = pd.to_datetime(df_with_nans_all_vars_loaded['Date'])
                else:
                    print(f"    Advertencia: Columna 'Date' no encontrada en {sim_file_path_to_load}.")
                    # If 'Date' is critical and missing, might need to skip or handle
            except Exception as e_load_rep:
                print(f"    Error cargando {sim_file_path_to_load}: {e_load_rep}."); continue

            df_to_save_imputed_all_vars_rep = df_with_nans_all_vars_loaded.copy()
            exog_for_this_rep_df = pd.DataFrame(index=range(len(df_with_nans_all_vars_loaded)))
            current_scaled_exog_for_rep_df = None

            if n_exog_features_global > 0 and scaler_exog_city is not None and 'Date' in df_with_nans_all_vars_loaded.columns:
                temp_dates_rep = df_with_nans_all_vars_loaded['Date'] # Already datetime objects
                # Ensure temp_dates_rep is indeed datetime and not object or strings
                if not pd.api.types.is_datetime64_any_dtype(temp_dates_rep) and not pd.api.types.is_period_dtype(temp_dates_rep) :
                     temp_dates_rep = pd.to_datetime(temp_dates_rep)


                # Check if temp_dates_rep has .dt accessor
                if hasattr(temp_dates_rep, 'dt'):
                    for tf_col_rep in TIME_FEATURES:
                        if tf_col_rep == 'Hour': exog_for_this_rep_df[tf_col_rep] = temp_dates_rep.dt.hour
                        elif tf_col_rep == 'DayOfWeek': exog_for_this_rep_df[tf_col_rep] = temp_dates_rep.dt.dayofweek
                        elif tf_col_rep == 'Month': exog_for_this_rep_df[tf_col_rep] = temp_dates_rep.dt.month
                    # Ensure all TIME_FEATURES columns are present before transform
                    if all(col in exog_for_this_rep_df.columns for col in TIME_FEATURES):
                         current_scaled_exog_for_rep_df = pd.DataFrame(scaler_exog_city.transform(exog_for_this_rep_df[TIME_FEATURES]), columns=TIME_FEATURES)
                    else:
                        print(f"    Advertencia: No todas las TIME_FEATURES se pudieron generar para la repetición. current_scaled_exog_for_rep_df será None.")
                else:
                    print(f"    Advertencia: temp_dates_rep no tiene accessor .dt. No se pueden generar exog features para repetición.")
            
            # Prepare NaN mask for metrics calculation
            # Align indices with df_city_original_gt_reset_for_metrics
            nan_mask_for_metrics_rep = pd.DataFrame(False, index=df_city_original_gt_reset_for_metrics.index, columns=ALL_TARGET_COLS_VAE)
            # Ensure df_with_nans_all_vars_loaded is also indexed from 0 to len-1 for direct comparison
            df_with_nans_all_vars_loaded_reset = df_with_nans_all_vars_loaded.reset_index(drop=True)

            for col_mask_rep in ALL_TARGET_COLS_VAE:
                if col_mask_rep in df_with_nans_all_vars_loaded_reset.columns and col_mask_rep in df_city_original_gt_reset_for_metrics.columns:
                    # Make sure lengths match for direct boolean indexing, or take min length
                    len_min = min(len(df_with_nans_all_vars_loaded_reset), len(df_city_original_gt_reset_for_metrics))
                    mask_series = df_with_nans_all_vars_loaded_reset[col_mask_rep].iloc[:len_min].isnull() & \
                                  ~df_city_original_gt_reset_for_metrics[col_mask_rep].iloc[:len_min].isnull()
                    nan_mask_for_metrics_rep[col_mask_rep].iloc[:len_min] = mask_series


            for target_var_impute_loop in ALL_TARGET_COLS_VAE:
                if target_var_impute_loop not in df_with_nans_all_vars_loaded.columns:
                    print(f"      Variable {target_var_impute_loop} no encontrada en datos cargados con NaNs. Omitiendo imputación para esta variable."); continue

                vae_model_for_var, scaler_target_for_var = trained_vaes.get(target_var_impute_loop), scalers_target.get(target_var_impute_loop)
                if vae_model_for_var is None or scaler_target_for_var is None:
                    print(f"      VAE/scaler no listo para {target_var_impute_loop}. Usando ffill/bfill.");
                    df_to_save_imputed_all_vars_rep[target_var_impute_loop] = df_to_save_imputed_all_vars_rep[target_var_impute_loop].ffill().bfill()
                    if df_to_save_imputed_all_vars_rep[target_var_impute_loop].isnull().any() and target_var_impute_loop in df_city_original_gt_reset_for_metrics.columns:
                        fallback_mean = df_city_original_gt_reset_for_metrics[target_var_impute_loop].mean()
                        if pd.notna(fallback_mean):
                            df_to_save_imputed_all_vars_rep[target_var_impute_loop].fillna(fallback_mean, inplace=True)
                        else: # If mean is also NaN (e.g. all NaNs in original)
                            df_to_save_imputed_all_vars_rep[target_var_impute_loop].fillna(0, inplace=True) # Ultimate fallback
                    continue
                
                target_series_with_nans_for_impute = df_with_nans_all_vars_loaded[target_var_impute_loop].values
                fallback_value_for_var = df_city_original_gt_reset_for_metrics[target_var_impute_loop].mean() if target_var_impute_loop in df_city_original_gt_reset_for_metrics.columns else 0
                if pd.isna(fallback_value_for_var): fallback_value_for_var = 0 # Ensure fallback is not NaN

                imputed_values_for_var_2d = impute_segment_with_lstm_vae(
                    target_series_with_nans_for_impute,
                    current_scaled_exog_for_rep_df, # This might be None or empty
                    vae_model_for_var,
                    scaler_target_for_var,
                    N_LAGS_VAE,
                    n_features_input_global_vae,
                    fallback_value_unscaled=fallback_value_for_var
                )
                df_to_save_imputed_all_vars_rep[target_var_impute_loop] = imputed_values_for_var_2d.flatten()

            # Add back TIME_FEATURES if they were dropped or not present, from original data
            # Ensure df_to_save_imputed_all_vars_rep has the same length as original for this
            max_len = len(df_to_save_imputed_all_vars_rep)
            for tf_col_final in TIME_FEATURES:
                if tf_col_final not in df_to_save_imputed_all_vars_rep.columns and tf_col_final in df_city_original_gt_reset_for_metrics.columns:
                     df_to_save_imputed_all_vars_rep[tf_col_final] = df_city_original_gt_reset_for_metrics[tf_col_final].values[:max_len]
                elif tf_col_final in df_to_save_imputed_all_vars_rep.columns and tf_col_final in df_city_original_gt_reset_for_metrics.columns: # If present, ensure it's from original non-scaled data
                     df_to_save_imputed_all_vars_rep[tf_col_final] = df_city_original_gt_reset_for_metrics[tf_col_final].values[:max_len]


            unified_imputed_filename = f"{city_name_safe_for_files}_{scenario_name_safe_vae}_rep{rep_vae}_seed{current_seed_loaded_vae}_imputed_vae_ALL.csv"
            unified_imputed_file_save_path = os.path.join(imputed_unified_scenario_dir, unified_imputed_filename)
            
            # Define columns to save, ensuring they exist in the dataframe
            cols_to_save_final = ['Date', 'City'] + ALL_TARGET_COLS_VAE + [tf for tf in TIME_FEATURES if tf in df_to_save_imputed_all_vars_rep.columns]
            cols_to_save_final = [c for c in cols_to_save_final if c in df_to_save_imputed_all_vars_rep.columns] # Filter out non-existing cols
            df_to_save_final_subset = df_to_save_imputed_all_vars_rep[cols_to_save_final].copy() # Create subset
            
            # Ensure 'Date' is string format for CSV if it was converted to datetime
            if 'Date' in df_to_save_final_subset.columns and pd.api.types.is_datetime64_any_dtype(df_to_save_final_subset['Date']):
                df_to_save_final_subset['Date'] = df_to_save_final_subset['Date'].dt.strftime('%Y-%m-%d %H:%M:%S%z')


            df_to_save_final_subset.to_csv(unified_imputed_file_save_path, index=False)
            print(f"      Archivo unificado imputado guardado: {unified_imputed_filename}")

            # Metrics calculation
            # Ensure y_pred_df_reset has same length and index as y_true_df_reset for comparison
            df_imputed_for_metrics = df_to_save_imputed_all_vars_rep.reset_index(drop=True)
            len_true = len(df_city_original_gt_reset_for_metrics)
            len_pred = len(df_imputed_for_metrics)
            
            if len_pred < len_true:
                print(f"    Advertencia: Longitud de datos imputados ({len_pred}) es menor que original ({len_true}). Métricas pueden ser incompletas.")
                # Pad df_imputed_for_metrics or truncate df_city_original_gt_reset_for_metrics for calc
                # For now, metrics will be calculated on the common length, implicitly handled by calculate_metrics if columns are sliced
            
            metrics_scores_vae_rep = calculate_metrics_with_nmae_nrmse(
                df_city_original_gt_reset_for_metrics.iloc[:len_pred] if len_pred < len_true else df_city_original_gt_reset_for_metrics,
                df_imputed_for_metrics,
                nan_mask_for_metrics_rep.iloc[:len_pred] if len_pred < len_true else nan_mask_for_metrics_rep, # Align mask length
                ALL_TARGET_COLS_VAE
            )

            for var_name_metric in ALL_TARGET_COLS_VAE:
                all_detailed_metrics_lstm_vae_results.append({
                    'City': current_city_name_vae,
                    'Scenario': scenario_name_safe_vae,
                    'Repetition': rep_vae,
                    'Seed': current_seed_loaded_vae,
                    'Imputation_Method': LSTM_VAE_METHOD_NAME,
                    'Variable': var_name_metric,
                    'MAE': metrics_scores_vae_rep.get(var_name_metric, {}).get('MAE', np.nan),
                    'RMSE': metrics_scores_vae_rep.get(var_name_metric, {}).get('RMSE', np.nan),
                    'NRMAE_range': metrics_scores_vae_rep.get(var_name_metric, {}).get('NRMAE_range', np.nan),
                    'NRMSE_range': metrics_scores_vae_rep.get(var_name_metric, {}).get('NRMSE_range', np.nan)
                })
            print(f"    Métricas calculadas para rep {rep_vae}, Escenario {scenario_name_safe_vae}")
    print(f"  Procesamiento de escenarios para {current_city_name_vae} completado.")

# --- Guardado de Métricas Resumidas ---
if all_detailed_metrics_lstm_vae_results:
    df_all_detailed_metrics_vae = pd.DataFrame(all_detailed_metrics_lstm_vae_results)
    print("\nGenerando resúmenes de métricas LSTM-VAE...")
    sufijo_nombre_archivo = f"_{sanitize_tf_name(TARGET_CITY_VAE)}" if TARGET_CITY_VAE else "_ALL_CITIES" # More descriptive
    detailed_metrics_path_vae = os.path.join(METRICS_LSTM_VAE_OUTPUT_DIR, f'ALL_metrics_detailed_{LSTM_VAE_METHOD_NAME}{sufijo_nombre_archivo}.csv')
    df_all_detailed_metrics_vae.to_csv(detailed_metrics_path_vae, index=False)
    print(f"  Métricas detalladas LSTM-VAE guardadas en: {detailed_metrics_path_vae}")
    if not df_all_detailed_metrics_vae.empty: print(df_all_detailed_metrics_vae.head())

    # Resumen por variable y ciudad
    if not df_all_detailed_metrics_vae.empty:
        grouped_by_city_var_vae = df_all_detailed_metrics_vae.groupby(['City', 'Variable'])[['MAE', 'RMSE', 'NRMAE_range', 'NRMSE_range']].mean().reset_index()
        grouped_by_city_var_vae.rename(columns={'MAE': 'Avg_MAE', 'RMSE': 'Avg_RMSE', 'NRMAE_range': 'Avg_NRMAE_range', 'NRMSE_range': 'Avg_NRMSE_range'}, inplace=True)
        for city_name_summary_val in grouped_by_city_var_vae['City'].unique(): # Changed var name to avoid conflict
            city_metrics_df_summary = grouped_by_city_var_vae[grouped_by_city_var_vae['City'] == city_name_summary_val]
            variables_summary_path_vae = os.path.join(METRICS_LSTM_VAE_OUTPUT_DIR, f'metrics_variables_summary_{LSTM_VAE_METHOD_NAME}_{sanitize_tf_name(city_name_summary_val)}.csv')
            city_metrics_df_summary.to_csv(variables_summary_path_vae, index=False)
            print(f"  Resumen LSTM-VAE de variables para {city_name_summary_val} guardado en: {variables_summary_path_vae}")

        # Resumen combinado de contaminantes para todas las ciudades procesadas
        contaminant_cols_for_summary = [c for c in ALL_TARGET_COLS_VAE if c != 'AQI']
        df_contaminants_only_vae = df_all_detailed_metrics_vae[df_all_detailed_metrics_vae['Variable'].isin(contaminant_cols_for_summary)]
        if not df_contaminants_only_vae.empty:
            contaminants_combined_summary_vae = df_contaminants_only_vae.groupby('City')[['MAE', 'RMSE', 'NRMAE_range', 'NRMSE_range']].mean().reset_index()
            contaminants_combined_summary_vae.rename(columns={'MAE': 'Avg_MAE_Conts', 'RMSE': 'Avg_RMSE_Conts', 'NRMAE_range': 'Avg_NRMAE_range_Conts', 'NRMSE_range': 'Avg_NRMSE_range_Conts'}, inplace=True)
            contaminants_summary_path_vae = os.path.join(METRICS_LSTM_VAE_OUTPUT_DIR, f'metrics_contaminants_combined_summary_{LSTM_VAE_METHOD_NAME}{sufijo_nombre_archivo}.csv')
            contaminants_combined_summary_vae.to_csv(contaminants_summary_path_vae, index=False); print(f"  Resumen LSTM-VAE contaminantes guardado: {contaminants_summary_path_vae}"); print(contaminants_combined_summary_vae)

        # Resumen combinado de AQI para todas las ciudades procesadas
        df_aqi_only_vae = df_all_detailed_metrics_vae[df_all_detailed_metrics_vae['Variable'] == 'AQI']
        if not df_aqi_only_vae.empty:
            aqi_combined_summary_vae = df_aqi_only_vae.groupby('City')[['MAE', 'RMSE', 'NRMAE_range', 'NRMSE_range']].mean().reset_index()
            aqi_combined_summary_vae.rename(columns={'MAE': 'Avg_MAE_AQI', 'RMSE': 'Avg_RMSE_AQI', 'NRMAE_range': 'Avg_NRMAE_range_AQI', 'NRMSE_range': 'Avg_NRMSE_range_AQI'}, inplace=True)
            aqi_summary_path_vae = os.path.join(METRICS_LSTM_VAE_OUTPUT_DIR, f'metrics_aqi_combined_summary_{LSTM_VAE_METHOD_NAME}{sufijo_nombre_archivo}.csv')
            aqi_combined_summary_vae.to_csv(aqi_summary_path_vae, index=False); print(f"  Resumen LSTM-VAE AQI guardado: {aqi_summary_path_vae}"); print(aqi_combined_summary_vae)
else: print("No se generaron métricas detalladas para LSTM-VAE.")

script_end_time_lstm_vae = time.time()
print(f"\nProceso LSTM-VAE completado. Tiempo: {(script_end_time_lstm_vae - script_start_time_lstm_vae) / 60:.2f} min.")

Datos originales cargados desde: CSVs\Air_Quality.csv

Convirtiendo fechas de UTC a la hora local de cada ciudad y extrayendo características de tiempo...
Conversión de zona horaria y extracción de características de tiempo completada.

=== PROCESANDO CIUDAD: Brasilia ===

--- Preparando VAE para VARIABLE: CO (Ciudad: Brasilia) ---
  Iniciando WFV para VAE de CO...
    WFV Fold 1 (CO): Train target 878, Val target 439
    Train exog (878, 3), Val exog (439, 3)
      Fold 1 Val Total Loss (best during this fit): 0.0032
      NUEVO MEJOR VAE (CO, loss: 0.0032). Guardando pesos globales.
  Modelo VAE final para CO cargado/actualizado con mejores pesos de WFV.

--- Preparando VAE para VARIABLE: NO2 (Ciudad: Brasilia) ---
  Iniciando WFV para VAE de NO2...
    WFV Fold 1 (NO2): Train target 878, Val target 439
    Train exog (878, 3), Val exog (439, 3)
      Fold 1 Val Total Loss (best during this fit): 0.0094
      NUEVO MEJOR VAE (NO2, loss: 0.0094). Guardando pesos globales.
  Modelo VAE

MOMENT-PRE

In [ ]:
import pandas as pd
import numpy as np
import os
import torch
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
import joblib
import traceback
import time
import random
import warnings

# Attempt to import MOMENT, handle if not found
try:
    from momentfm import MOMENTPipeline
except ImportError:
    print("MOMENTfm library not found. Please install it: pip install momentfm")
    MOMENTPipeline = None # Placeholder

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error


warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# --- Helper Function for Sanitizing File/Path Names ---
def sanitize_name(name_str):
    import re
    # Convert to string, remove non-alphanumeric (keep underscores), replace spaces
    s = str(name_str)
    s = re.sub(r'[^\w\s-]', '', s).strip() # Remove special chars except underscore, hyphen
    s = re.sub(r'[-\s]+', '_', s)       # Replace spaces/hyphens with underscore
    s = re.sub(r'_+', '_', s)           # Replace multiple underscores with single
    return s if s else "default_sanitized_name"


# --- 0. Carga de Datos Originales (Ground Truth) ---
TARGET_CITY_MOMENT = "Brasilia" # Define early for GT loading
try:
    file_path_load_orig = os.path.join('CSVs', 'Air_Quality.csv')
    if not os.path.exists(file_path_load_orig):
        script_dir_parent_orig = os.path.dirname(os.getcwd())
        alt_file_path_load_orig = os.path.join(script_dir_parent_orig, 'CSVs', 'Air_Quality.csv')
        if os.path.exists(alt_file_path_load_orig): file_path_load_orig = alt_file_path_load_orig
        else: raise FileNotFoundError("Air_Quality.csv (original) no encontrado.")
    df_air_quality_ground_truth_raw = pd.read_csv(file_path_load_orig)
    print(f"Datos originales (ground truth) cargados desde: {file_path_load_orig}")
except FileNotFoundError:
    print("Error: Air_Quality.csv (original) no encontrado. Creando DataFrame de ejemplo...")
    _example_data = []
    _cities_example = [TARGET_CITY_MOMENT]
    _start_date = pd.to_datetime('2024-01-01')
    for _city_ex in _cities_example:
        num_rows_example = 550
        for i in range(num_rows_example):
            _date = _start_date + pd.Timedelta(hours=i)
            _example_data.append([_date, _city_ex, 200+np.random.rand()*100, 20+np.random.rand()*10, 5+np.random.rand()*5, 50+np.random.rand()*30, 15+np.random.rand()*10, 30+np.random.rand()*20, 40+np.random.rand()*20])
    df_air_quality_ground_truth_raw = pd.DataFrame(_example_data, columns=['Date', 'City', 'CO', 'NO2', 'SO2', 'O3', 'PM2.5', 'PM10', 'AQI'])
    print("DataFrame de ejemplo creado.")

# Ensure 'Date' is datetime and UTC, then localize for the TARGET_CITY_MOMENT
df_air_quality_ground_truth_raw['Date'] = pd.to_datetime(df_air_quality_ground_truth_raw['Date'], utc=True)
if 'CO2' in df_air_quality_ground_truth_raw.columns:
    df_air_quality_ground_truth_raw = df_air_quality_ground_truth_raw.drop(columns=['CO2'])

# Reemplazar "New_York" por "New York" si existe en los datos cargados
df_air_quality_ground_truth_raw['City'] = df_air_quality_ground_truth_raw['City'].replace('New_York', 'New York')


TIME_FEATURES = ['Hour', 'DayOfWeek', 'Month']
city_timezones = {
    'Brasilia': 'America/Sao_Paulo', 'Cairo': 'Africa/Cairo', 'Dubai': 'Asia/Dubai',
    'London': 'Europe/London', 'New York': 'America/New_York', 'Sydney': 'Australia/Sydney'
}

df_air_quality_ground_truth_localized = pd.DataFrame()
if TARGET_CITY_MOMENT in city_timezones:
    tz = city_timezones[TARGET_CITY_MOMENT]
    df_city_part_gt = df_air_quality_ground_truth_raw[df_air_quality_ground_truth_raw['City'] == TARGET_CITY_MOMENT].copy()
    if not df_city_part_gt.empty:
        df_city_part_gt['Date'] = df_city_part_gt['Date'].dt.tz_convert(tz)
        df_city_part_gt['Hour'] = df_city_part_gt['Date'].dt.hour
        df_city_part_gt['DayOfWeek'] = df_city_part_gt['Date'].dt.dayofweek
        df_city_part_gt['Month'] = df_city_part_gt['Date'].dt.month
        df_air_quality_ground_truth_localized = df_city_part_gt
        print(f"Ground truth for {TARGET_CITY_MOMENT} localized to {tz} and time features extracted.")
    else:
        print(f"No data for {TARGET_CITY_MOMENT} in ground truth for timezone conversion.")
else:
    print(f"Timezone not defined for {TARGET_CITY_MOMENT}. Using original dates for GT and attempting to extract time features.")
    df_air_quality_ground_truth_localized = df_air_quality_ground_truth_raw[df_air_quality_ground_truth_raw['City'] == TARGET_CITY_MOMENT].copy()
    if not df_air_quality_ground_truth_localized.empty and 'Date' in df_air_quality_ground_truth_localized.columns:
        df_air_quality_ground_truth_localized['Date'] = pd.to_datetime(df_air_quality_ground_truth_localized['Date']) # Ensure datetime
        df_air_quality_ground_truth_localized['Hour'] = df_air_quality_ground_truth_localized['Date'].dt.hour
        df_air_quality_ground_truth_localized['DayOfWeek'] = df_air_quality_ground_truth_localized['Date'].dt.dayofweek
        df_air_quality_ground_truth_localized['Month'] = df_air_quality_ground_truth_localized['Date'].dt.month
        print(f"Time features extracted for {TARGET_CITY_MOMENT} from non-localized dates.")

if df_air_quality_ground_truth_localized.empty:
    print(f"Error: Ground truth for {TARGET_CITY_MOMENT} is empty. Exiting.")
    exit()


# --- Parámetros Configurables ---
CONTAMINANT_COLS_MOMENT = ['CO', 'NO2', 'SO2', 'O3', 'PM2.5', 'PM10']
AQI_COL_MOMENT = ['AQI']
ALL_TARGET_COLS_MOMENT = CONTAMINANT_COLS_MOMENT + AQI_COL_MOMENT

SEQ_LEN_MODEL_MOMENT = 512
IMPUTATION_STEP_MOMENT = SEQ_LEN_MODEL_MOMENT # Non-overlapping windows for simplicity
BATCH_SIZE_IMPUTE_MOMENT = 16
MODEL_NAME_MOMENT_PRETRAINED = "AutonLab/MOMENT-1-base" # Changed to base model

N_REPETITIONS_MOMENT_EVAL = 1 # Changed to 1 repetition
# Scenarios should match those generated by the VAE simulation script (01_simular_nulos_corregido_v2.py)
NAN_SIMULATION_SCENARIOS_MOMENT = [
    {'name': 'RandomRows_20percent'},
    {'name': 'BlockRows_12h_20percent'},
    {'name': 'BlockRows_24h_20percent'},
    {'name': 'BlockRows_36h_20percent'}, # Add more if needed and available
    {'name': 'BlockRows_48h_20percent'},
    {'name': 'BlockRows_60h_20percent'}
]

PROJECT_ROOT_DIR_MOMENT = os.getcwd()
# INPUT Directory: Point to where the VAE script saved its simulated data
SIMULATED_DATA_INPUT_ROOT_DIR_MOMENT = os.path.join(PROJECT_ROOT_DIR_MOMENT, 'CSVs', 'simulados')

MOMENT_METHOD_NAME = f'MOMENT_PRE_{sanitize_name(MODEL_NAME_MOMENT_PRETRAINED.split("/")[-1])}'
IMPUTATIONS_MOMENT_BASE_DIR = os.path.join(PROJECT_ROOT_DIR_MOMENT, 'imputaciones', MOMENT_METHOD_NAME)
METRICS_MOMENT_OUTPUT_DIR = os.path.join(IMPUTATIONS_MOMENT_BASE_DIR, 'metricas')
IMPUTED_CSVS_MOMENT_OUTPUT_DIR = os.path.join(IMPUTATIONS_MOMENT_BASE_DIR, 'CSVs')
SCALERS_MOMENT_OUTPUT_DIR = os.path.join(IMPUTATIONS_MOMENT_BASE_DIR, 'scalers')
os.makedirs(METRICS_MOMENT_OUTPUT_DIR, exist_ok=True)
os.makedirs(IMPUTED_CSVS_MOMENT_OUTPUT_DIR, exist_ok=True)
os.makedirs(SCALERS_MOMENT_OUTPUT_DIR, exist_ok=True)

# ======================================================================
# --- Dataset Personalizado para MOMENT (CON LA CORRECCIÓN FINAL) ---
# ======================================================================
class MomentImputationDataset(Dataset):
    def __init__(self, data_array_2d_multivar_scaled_zeros, nan_mask_original_bool_2d, seq_len, step):
        self.data_array_scaled_zeros = data_array_2d_multivar_scaled_zeros # (total_len, n_channels)
        self.nan_mask_original = nan_mask_original_bool_2d         # (total_len, n_channels), True where original was NaN
        self.seq_len = seq_len
        self.step = step

        if len(self.data_array_scaled_zeros) == 0:
            self.n_samples, self.indices = 0, []
        elif len(self.data_array_scaled_zeros) < self.seq_len :
            self.n_samples, self.indices = 1, [0] # Will pad one sample
        else:
            self.indices = list(range(0, len(self.data_array_scaled_zeros) - self.seq_len + 1, self.step))
            last_possible_start = len(self.data_array_scaled_zeros) - self.seq_len
            # Ensure the last segment is included if step doesn't align perfectly
            if not self.indices or self.indices[-1] < last_possible_start :
                if not self.indices or last_possible_start > self.indices[-1]: # Add if not present or if last_possible_start is further
                    self.indices.append(last_possible_start)
            self.n_samples = len(self.indices)
            if self.n_samples == 0 and len(self.data_array_scaled_zeros) >= self.seq_len: # Should not happen with above logic
                 self.indices, self.n_samples = [0],1


    def __len__(self):
        return self.n_samples

    def __getitem__(self, idx):
        if self.n_samples == 0:
            raise IndexError("Dataset is empty")
            
        actual_start_index = self.indices[idx]
        seq_data_scaled_zeros_orig_shape = self.data_array_scaled_zeros[actual_start_index : actual_start_index + self.seq_len]
        # nan_mask_original is True for NaN. We need a mask that's True for OBSERVED.
        seq_observed_mask_bool_per_channel = ~self.nan_mask_original[actual_start_index : actual_start_index + self.seq_len] # (seq_len, n_channels)

        if len(seq_data_scaled_zeros_orig_shape) < self.seq_len:
            pad_width = self.seq_len - len(seq_data_scaled_zeros_orig_shape)
            # Pad data with 0.0
            seq_data_scaled_zeros_orig_shape = np.pad(seq_data_scaled_zeros_orig_shape, ((0, pad_width), (0, 0)), 'constant', constant_values=0.0)
            # Pad observed mask with False (these padded values are not observed)
            seq_observed_mask_bool_per_channel = np.pad(seq_observed_mask_bool_per_channel, ((0, pad_width), (0,0)), 'constant', constant_values=False)

        # timeseries: (n_channels, seq_len)
        timeseries_out = torch.tensor(seq_data_scaled_zeros_orig_shape, dtype=torch.float32).transpose(0, 1)

        # input_mask for MOMENT: (seq_len,). True if the time point is considered "observed enough".
        # Let's define "observed enough" as: True if ALL channels at that timestep were observed in the original _with_nans data.
        input_mask_per_channel_tensor = torch.tensor(seq_observed_mask_bool_per_channel, dtype=torch.bool)
        input_mask_out = input_mask_per_channel_tensor.all(dim=1) # Shape: (seq_len,)

        return {
            'timeseries': timeseries_out,    # (n_channels, seq_len)
            'input_mask': input_mask_out     # (seq_len,)
        }

# --- Función de Imputación con MOMENT ---
def impute_multivar_with_moment_pipeline(df_with_nans_city_scaled_zeros, original_nan_mask_df_bool, moment_pipeline_obj, device_str, scaler_dict_vars_city, seq_len_moment, imputation_step_moment, batch_size_moment):
    """
    Imputes missing values in a multivariate DataFrame using MOMENT.
    Args:
        df_with_nans_city_scaled_zeros (pd.DataFrame): DataFrame with target columns scaled, and NaNs filled with 0.0. (num_rows, num_target_cols)
        original_nan_mask_df_bool (pd.DataFrame): Boolean mask DataFrame, True where original data had NaNs. (num_rows, num_target_cols)
        moment_pipeline_obj: Initialized MOMENT pipeline.
        device_str (str): 'cuda' or 'cpu'.
        scaler_dict_vars_city (dict): Dictionary of scalers, {col_name: scaler_obj}.
        seq_len_moment (int): Sequence length for MOMENT.
        imputation_step_moment (int): Step size for creating sequences.
        batch_size_moment (int): Batch size for DataLoader.
    Returns:
        pd.DataFrame: DataFrame with imputed values, in original scale. Only NaNs are filled.
                      Returns None if imputation fails or dataset is too short.
    """
    if len(df_with_nans_city_scaled_zeros) < 1: # Basic check
        print("  Dataset for MOMENT imputation is too short or empty.")
        return None

    imputation_dataset = MomentImputationDataset(
        df_with_nans_city_scaled_zeros.values, # (total_len, n_channels)
        original_nan_mask_df_bool.values,    # (total_len, n_channels)
        seq_len_moment,
        imputation_step_moment
    )
    if len(imputation_dataset) == 0:
        print("  MomentImputationDataset resulted in 0 samples.")
        return None

    imputation_loader = DataLoader(imputation_dataset, batch_size=batch_size_moment, shuffle=False, num_workers=0)

    n_rows_total, n_features_total = df_with_nans_city_scaled_zeros.shape
    accumulated_preds_scaled = np.zeros((n_rows_total, n_features_total), dtype=float)
    accumulated_counts = np.zeros((n_rows_total, n_features_total), dtype=float)

    with torch.no_grad():
        for batch_idx, batch_data in enumerate(imputation_loader):
            timeseries_batch_tensor = batch_data['timeseries'].to(device_str) # (batch, n_channels, seq_len)
            input_mask_batch_tensor = batch_data['input_mask'].to(device_str) # (batch, seq_len)
            preds_batch_scaled_np = None
            try:
                outputs_obj = moment_pipeline_obj(x_enc=timeseries_batch_tensor, input_mask=input_mask_batch_tensor)
                # MOMENT output reconstruction is (batch_size, n_channels, seq_len)
                preds_batch_scaled_np = outputs_obj.reconstruction.cpu().numpy()
            except Exception as e:
                print(f"    Error during MOMENT prediction for batch {batch_idx}: {e}")
                # traceback.print_exc() # Optional: for more detailed error
                continue # Skip this batch

            if preds_batch_scaled_np is None:
                continue

            for i_in_batch_loop in range(preds_batch_scaled_np.shape[0]): # Iterate through samples in the batch
                current_dataset_idx = batch_idx * batch_size_moment + i_in_batch_loop
                if current_dataset_idx >= len(imputation_loader.dataset.indices):
                    continue # Should not happen if dataloader and dataset are correct

                start_idx_in_series = imputation_loader.dataset.indices[current_dataset_idx]
                end_idx_in_series = min(start_idx_in_series + seq_len_moment, n_rows_total)
                len_window_actual = end_idx_in_series - start_idx_in_series

                if len_window_actual > 0:
                    # pred_window_scaled_loop needs to be (len_window_actual, n_features_total)
                    # preds_batch_scaled_np[i_in_batch_loop, :, :len_window_actual] is (n_features_total, len_window_actual)
                    pred_window_scaled_loop = preds_batch_scaled_np[i_in_batch_loop, :, :len_window_actual].T
                    accumulated_preds_scaled[start_idx_in_series:end_idx_in_series, :] += pred_window_scaled_loop
                    accumulated_counts[start_idx_in_series:end_idx_in_series, :] += 1

    # Avoid division by zero for parts of the series not covered by any window
    accumulated_counts[accumulated_counts == 0] = 1
    imputed_full_series_scaled = accumulated_preds_scaled / accumulated_counts # Element-wise division

    # Create a DataFrame for unscaled imputed values
    imputed_full_series_unscaled_df = pd.DataFrame(columns=df_with_nans_city_scaled_zeros.columns, index=df_with_nans_city_scaled_zeros.index, dtype=float)

    for col_idx, col_name_unscale in enumerate(df_with_nans_city_scaled_zeros.columns):
        scaler_var_unscale = scaler_dict_vars_city.get(col_name_unscale)
        if scaler_var_unscale:
            # imputed_full_series_scaled[:, col_idx] is 1D array for the current column
            imputed_values_for_col_scaled = imputed_full_series_scaled[:, col_idx].reshape(-1, 1)
            imputed_full_series_unscaled_df[col_name_unscale] = scaler_var_unscale.inverse_transform(imputed_values_for_col_scaled).flatten()
        else: # Should not happen if all target cols have scalers
            imputed_full_series_unscaled_df[col_name_unscale] = imputed_full_series_scaled[:, col_idx]

    # Start with the original data (unscaled, with NaNs)
    df_final_imputed_unscaled = df_with_nans_city_scaled_zeros.copy() # This is scaled with 0s for NaNs
    # First, unscale the observed values in df_final_imputed_unscaled
    for col_name_inv_transform in df_final_imputed_unscaled.columns:
        if col_name_inv_transform in scaler_dict_vars_city:
            scaler = scaler_dict_vars_city[col_name_inv_transform]
            # Inverse transform only the observed values (where original_nan_mask is False)
            observed_mask_col = ~original_nan_mask_df_bool[col_name_inv_transform].values
            if np.any(observed_mask_col): # If there are any observed values
                 scaled_observed_values = df_final_imputed_unscaled.loc[observed_mask_col, col_name_inv_transform].values.reshape(-1, 1)
                 df_final_imputed_unscaled.loc[observed_mask_col, col_name_inv_transform] = scaler.inverse_transform(scaled_observed_values).flatten()
            # For values that were originally NaN (now 0.0), they will be overwritten next
    
    # Now, fill in the imputed values (unscaled) only at the positions that were originally NaN
    for col_final_fill in df_final_imputed_unscaled.columns:
        if col_final_fill in imputed_full_series_unscaled_df.columns:
            # Get mask for original NaNs for this column
            nan_mask_this_col = original_nan_mask_df_bool[col_final_fill].values
            # Update these NaN positions with unscaled imputations from MOMENT
            df_final_imputed_unscaled.loc[nan_mask_this_col, col_final_fill] = imputed_full_series_unscaled_df.loc[nan_mask_this_col, col_final_fill]

    return df_final_imputed_unscaled


# --- Función de Métricas ---
def calculate_metrics_moment(y_true_df_reset, y_pred_df_reset, nan_mask_for_metrics_df_reset, cols_to_eval):
    results = {}
    for col in cols_to_eval:
        results[col] = {'MAE': np.nan, 'RMSE': np.nan, 'NRMAE_range': np.nan, 'NRMSE_range': np.nan}
        if col not in y_true_df_reset.columns or col not in y_pred_df_reset.columns or col not in nan_mask_for_metrics_df_reset.columns:
            # print(f"Warning: Column {col} missing in one of the DFs for metrics.")
            continue

        true_col_vals = y_true_df_reset[col].values
        pred_col_vals = y_pred_df_reset[col].values
        mask_col_flags = nan_mask_for_metrics_df_reset[col].values.astype(bool) # Ensure boolean

        # Align lengths if necessary, though they should be from reset_index DFs
        min_len = min(len(true_col_vals), len(pred_col_vals), len(mask_col_flags))
        true_col_vals = true_col_vals[:min_len]
        pred_col_vals = pred_col_vals[:min_len]
        mask_col_flags = mask_col_flags[:min_len]

        true_at_nans = true_col_vals[mask_col_flags]
        pred_at_nans = pred_col_vals[mask_col_flags]

        if len(true_at_nans) == 0:
            # print(f"Warning: No NaNs to evaluate for column {col}.")
            continue

        valid_pred_mask = ~np.isnan(pred_at_nans) # Filter out NaNs if MOMENT produced any
        true_at_nans_valid = true_at_nans[valid_pred_mask]
        pred_at_nans_valid = pred_at_nans[valid_pred_mask]

        if len(true_at_nans_valid) == 0:
            # print(f"Warning: No valid predictions at NaN locations for column {col}.")
            continue
        try:
            mae = mean_absolute_error(true_at_nans_valid, pred_at_nans_valid)
            rmse = np.sqrt(mean_squared_error(true_at_nans_valid, pred_at_nans_valid))

            original_col_full_series = y_true_df_reset[col].dropna()
            original_col_range = 0
            if not original_col_full_series.empty:
                original_col_range = original_col_full_series.max() - original_col_full_series.min()

            nrmae_range = (mae / original_col_range if original_col_range > 1e-9 else np.nan)
            nrmse_range = (rmse / original_col_range if original_col_range > 1e-9 else np.nan)
            results[col].update({'MAE': mae, 'RMSE': rmse, 'NRMAE_range': nrmae_range, 'NRMSE_range': nrmse_range})
        except Exception as e_metrics:
            print(f"Error calculating metrics for {col}: {e_metrics}")
            pass # Keep NaNs
    return results

# --- Script Principal ---
if MOMENTPipeline is None:
    print("MOMENTPipeline could not be imported. Exiting script.")
    exit()

all_detailed_metrics_moment_results = []
script_start_time_moment = time.time()
device = "cuda" if torch.cuda.is_available() else "cpu"; print(f"Usando dispositivo: {device}")
print(f"Cargando modelo MOMENT pre-entrenado: {MODEL_NAME_MOMENT_PRETRAINED}")
moment_pipeline_global = None
try:
    moment_pipeline_global = MOMENTPipeline.from_pretrained(
        MODEL_NAME_MOMENT_PRETRAINED, model_kwargs={'task_name': 'reconstruction'}
    )
    moment_pipeline_global.init() # Initialize the model with the task
    moment_pipeline_global = moment_pipeline_global.to(device)
    print("Modelo MOMENT pre-entrenado cargado, INICIALIZADO y movido al dispositivo.")
except Exception as e_model_load:
    print(f"Error crítico cargando/inicializando modelo MOMENT: {e_model_load}"); traceback.print_exc(); exit()

# Use the localized ground truth for the target city
df_city_ground_truth_moment_reset = df_air_quality_ground_truth_localized.reset_index(drop=True)

# Fill any remaining NaNs in GT (e.g., at start/end) for robust scaler fitting
for col_gt_fill in ALL_TARGET_COLS_MOMENT:
    if col_gt_fill in df_city_ground_truth_moment_reset.columns:
        df_city_ground_truth_moment_reset[col_gt_fill] = df_city_ground_truth_moment_reset[col_gt_fill].ffill().bfill()
# Final check for NaNs in GT after fill
if df_city_ground_truth_moment_reset[ALL_TARGET_COLS_MOMENT].isnull().any().any():
    print(f"Error: No se pudo rellenar completamente el Ground Truth para {TARGET_CITY_MOMENT} para el entrenamiento de scalers. Omitiendo columnas con NaNs persistentes o saliendo.");
    # Option: fill with mean or 0, or exit
    for col_final_check in ALL_TARGET_COLS_MOMENT:
        if df_city_ground_truth_moment_reset[col_final_check].isnull().any():
            mean_val = df_city_ground_truth_moment_reset[col_final_check].mean()
            df_city_ground_truth_moment_reset[col_final_check].fillna(mean_val if pd.notna(mean_val) else 0, inplace=True)
            print(f"  Columna GT {col_final_check} rellenada con media ({mean_val}) o 0.")
    # exit()


# Fit scalers based on the (localized and filled) ground truth data
scalers_for_city_moment = {}
city_name_safe_gt = sanitize_name(TARGET_CITY_MOMENT)
city_scalers_dir = os.path.join(SCALERS_MOMENT_OUTPUT_DIR, city_name_safe_gt)
os.makedirs(city_scalers_dir, exist_ok=True)

for col_scale in ALL_TARGET_COLS_MOMENT:
    scaler_var = StandardScaler()
    # Ensure data for scaler is 2D and has no NaNs
    data_to_fit_scaler = df_city_ground_truth_moment_reset[[col_scale]].values # Already filled
    if len(data_to_fit_scaler) > 1 and not np.isnan(data_to_fit_scaler).all():
        scaler_var.fit(data_to_fit_scaler)
        scalers_for_city_moment[col_scale] = scaler_var
        joblib.dump(scaler_var, os.path.join(city_scalers_dir, f'scaler_{sanitize_name(col_scale)}.joblib'))
    else:
        print(f"  Advertencia: Datos insuficientes o todos NaN para escalar la columna {col_scale} en {TARGET_CITY_MOMENT}. Usando DummyScaler.")
        class DummyScaler: # Simple passthrough if scaling is not possible
            def fit(self, X): return self
            def transform(self, X): return np.array(X, dtype=float)
            def inverse_transform(self, X): return np.array(X, dtype=float)
            def __init__(self): self.mean_ = 0; self.scale_ = 1 # Mimic attributes
        scalers_for_city_moment[col_scale] = DummyScaler()
print(f"Scalers ajustados y guardados para {TARGET_CITY_MOMENT}.")


city_name_safe_for_files = sanitize_name(TARGET_CITY_MOMENT)
city_imputed_csv_moment_dir_root = os.path.join(IMPUTED_CSVS_MOMENT_OUTPUT_DIR, city_name_safe_for_files)
os.makedirs(city_imputed_csv_moment_dir_root, exist_ok=True)

for scenario_moment in NAN_SIMULATION_SCENARIOS_MOMENT:
    # Scenario name for directory (as created by VAE script: "RandomRows_20percent")
    scenario_name_for_dir = scenario_moment['name'].replace("%", "percent")
    # Scenario name part in filename (also "RandomRows_20percent")
    scenario_name_for_file_part = scenario_moment['name'].replace("%", "percent")
    
    print(f"\nProcesando Escenario: {scenario_name_for_dir}")
    # Path to the directory containing simulated files for this city and scenario
    sim_data_scenario_dir_moment = os.path.join(SIMULATED_DATA_INPUT_ROOT_DIR_MOMENT, city_name_safe_for_files, scenario_name_for_dir)
    
    imputed_moment_scenario_dir = os.path.join(city_imputed_csv_moment_dir_root, sanitize_name(scenario_name_for_dir))
    os.makedirs(imputed_moment_scenario_dir, exist_ok=True)

    if not os.path.exists(sim_data_scenario_dir_moment):
        print(f"    Directorio de datos simulados no encontrado: {sim_data_scenario_dir_moment}. Omitiendo escenario.")
        continue

    for rep_moment in range(1, N_REPETITIONS_MOMENT_EVAL + 1):
        print(f"  Repetición: {rep_moment}")
        
        # Construct file pattern based on VAE script's output naming
        file_pattern_to_find = f"{city_name_safe_for_files}_{scenario_name_for_file_part}_rep{rep_moment}_*_with_nans.csv"
        sim_files_in_dir_moment = sorted([
            f for f in os.listdir(sim_data_scenario_dir_moment)
            if f.startswith(f"{city_name_safe_for_files}_{scenario_name_for_file_part}_rep{rep_moment}") and f.endswith("_with_nans.csv")
        ])

        if not sim_files_in_dir_moment:
            print(f"    Archivo simulado no encontrado para patrón '{file_pattern_to_find}' en '{sim_data_scenario_dir_moment}'. Omitiendo repetición.")
            continue
        
        sim_file_path_to_load_moment = os.path.join(sim_data_scenario_dir_moment, sim_files_in_dir_moment[0])
        try:
            current_seed_loaded_moment = sim_files_in_dir_moment[0].split('_seed')[-1].split('_with_nans.csv')[0]
        except IndexError:
            current_seed_loaded_moment = "unknown" # Fallback if seed parsing fails
            print(f"    Advertencia: No se pudo extraer la semilla del nombre de archivo: {sim_files_in_dir_moment[0]}")

        try:
            df_city_with_nans_loaded_moment = pd.read_csv(sim_file_path_to_load_moment)
            # Ensure 'Date' is datetime for consistency, though not directly used by MOMENT core
            if 'Date' in df_city_with_nans_loaded_moment.columns:
                 df_city_with_nans_loaded_moment['Date'] = pd.to_datetime(df_city_with_nans_loaded_moment['Date'])
        except Exception as e_load_rep_m:
            print(f"    Error cargando archivo simulado {sim_file_path_to_load_moment}: {e_load_rep_m}. Omitiendo.")
            continue

        # --- Multivariate Imputation Preparations ---
        # Select only the target columns for imputation
        df_subset_for_imputation = df_city_with_nans_loaded_moment[ALL_TARGET_COLS_MOMENT].copy()
        
        # Scale the data for these columns
        df_scaled_multivar = pd.DataFrame(index=df_subset_for_imputation.index, columns=ALL_TARGET_COLS_MOMENT, dtype=float)
        for col_to_scale in ALL_TARGET_COLS_MOMENT:
            scaler_current = scalers_for_city_moment.get(col_to_scale)
            if scaler_current:
                col_data_unscaled = df_subset_for_imputation[[col_to_scale]].values
                # StandardScaler expects no NaNs for transform.
                # We will scale non-NaNs and keep NaNs as NaNs, then fill with 0.
                nan_mask_col = np.isnan(col_data_unscaled.flatten())
                scaled_values_col = np.full_like(col_data_unscaled.flatten(), np.nan, dtype=float)

                if not nan_mask_col.all(): # If there are some non-NaN values
                    non_nan_data = col_data_unscaled[~nan_mask_col].reshape(-1,1)
                    scaled_non_nan_data = scaler_current.transform(non_nan_data)
                    scaled_values_col[~nan_mask_col] = scaled_non_nan_data.flatten()
                df_scaled_multivar[col_to_scale] = scaled_values_col
            else: # Should not happen if scalers were created for all target cols
                df_scaled_multivar[col_to_scale] = df_subset_for_imputation[col_to_scale]

        original_nan_mask_multivar_df = df_subset_for_imputation.isnull() # Mask from unscaled subset (True for NaNs)
        df_scaled_filled_multivar = df_scaled_multivar.fillna(0.0) # Fill NaNs (original or from scaling) with 0 for MOMENT

        print(f"    Iniciando imputación multivariada para Rep {rep_moment}...")
        df_imputed_all_cols_unscaled_rep = impute_multivar_with_moment_pipeline(
            df_scaled_filled_multivar,      # (rows, num_target_cols), scaled, NaNs as 0
            original_nan_mask_multivar_df,  # (rows, num_target_cols), boolean, True for original NaNs
            moment_pipeline_global,
            device,
            scalers_for_city_moment,        # Dict of scalers for each target col
            SEQ_LEN_MODEL_MOMENT,
            IMPUTATION_STEP_MOMENT,
            BATCH_SIZE_IMPUTE_MOMENT
        )

        if df_imputed_all_cols_unscaled_rep is None or df_imputed_all_cols_unscaled_rep.isnull().all().all():
            print(f"    Falló la imputación MOMENT para rep {rep_moment} (resultado vacío o todos NaNs). Omitiendo.")
            continue
        
        # Prepare final DataFrame to save (Date, City, imputed target cols, time features)
        df_to_save_final = df_city_with_nans_loaded_moment[['Date', 'City']].copy()
        for col_target in ALL_TARGET_COLS_MOMENT:
            df_to_save_final[col_target] = df_imputed_all_cols_unscaled_rep[col_target]
        
        # Add back time features from ground truth (already localized and extracted)
        gt_cols_to_add = ['Date', 'City'] + ALL_TARGET_COLS_MOMENT + TIME_FEATURES
        final_cols_for_saving = ['Date', 'City'] + ALL_TARGET_COLS_MOMENT
        
        for tf_col in TIME_FEATURES:
            if tf_col in df_city_ground_truth_moment_reset.columns:
                # Align length with the imputed data
                df_to_save_final[tf_col] = df_city_ground_truth_moment_reset[tf_col].values[:len(df_to_save_final)]
                if tf_col not in final_cols_for_saving:
                    final_cols_for_saving.append(tf_col)
        
        df_to_save_final = df_to_save_final[final_cols_for_saving] # Ensure correct order and columns

        imputed_moment_filename = f"{city_name_safe_for_files}_{sanitize_name(scenario_name_for_file_part)}_rep{rep_moment}_seed{current_seed_loaded_moment}_imputed_momentALL.csv"
        imputed_moment_file_save_path = os.path.join(imputed_moment_scenario_dir, imputed_moment_filename)
        df_to_save_final.to_csv(imputed_moment_file_save_path, index=False)
        print(f"    Archivo imputado por MOMENT guardado: {imputed_moment_filename}")

        # --- Metrics Calculation ---
        # Mask for metrics: True where df_city_with_nans_loaded_moment had NaNs AND df_city_ground_truth_moment_reset did NOT have NaNs
        nan_mask_introduced_for_metrics_rep = pd.DataFrame(False, index=df_city_ground_truth_moment_reset.index, columns=ALL_TARGET_COLS_MOMENT)
        df_loaded_nans_reset = df_city_with_nans_loaded_moment.reset_index(drop=True)

        for col_m_mask in ALL_TARGET_COLS_MOMENT:
            if col_m_mask in df_loaded_nans_reset.columns and col_m_mask in df_city_ground_truth_moment_reset.columns:
                len_min_mask = min(len(df_loaded_nans_reset), len(df_city_ground_truth_moment_reset))
                mask_series_metric = (df_loaded_nans_reset[col_m_mask].iloc[:len_min_mask].isnull() &
                                     ~df_city_ground_truth_moment_reset[col_m_mask].iloc[:len_min_mask].isnull())
                nan_mask_introduced_for_metrics_rep[col_m_mask].iloc[:len_min_mask] = mask_series_metric


        metrics_scores_moment_rep = calculate_metrics_moment(
            df_city_ground_truth_moment_reset,  # Ground truth, reset index
            df_to_save_final.reset_index(drop=True), # Imputed data, reset index for alignment
            nan_mask_introduced_for_metrics_rep, # Mask of originally introduced NaNs
            ALL_TARGET_COLS_MOMENT
        )
        for var_name_metric_m in ALL_TARGET_COLS_MOMENT:
            all_detailed_metrics_moment_results.append({
                'City': TARGET_CITY_MOMENT,
                'Scenario': scenario_name_for_dir, # Use the dir name for consistency in reporting
                'Repetition': rep_moment,
                'Seed': current_seed_loaded_moment,
                'Imputation_Method': MOMENT_METHOD_NAME,
                'Variable': var_name_metric_m,
                'MAE': metrics_scores_moment_rep.get(var_name_metric_m, {}).get('MAE', np.nan),
                'RMSE': metrics_scores_moment_rep.get(var_name_metric_m, {}).get('RMSE', np.nan),
                'NRMAE_range': metrics_scores_moment_rep.get(var_name_metric_m, {}).get('NRMAE_range', np.nan),
                'NRMSE_range': metrics_scores_moment_rep.get(var_name_metric_m, {}).get('NRMSE_range', np.nan)
            })
        print(f"    Métricas calculadas para MOMENT, rep {rep_moment}")
    print(f"  Escenario {scenario_name_for_dir} completado.")

# --- Guardado de Métricas ---
if all_detailed_metrics_moment_results:
    df_all_detailed_metrics_moment = pd.DataFrame(all_detailed_metrics_moment_results)
    print("\nGenerando resúmenes de métricas MOMENT...")
    
    # Suffix for filenames based on model and city
    sufijo_test_m = f"_{sanitize_name(TARGET_CITY_MOMENT)}"
    if len(ALL_TARGET_COLS_MOMENT) == 1: # If testing only one variable (not the case here)
        sufijo_test_m += f"_SingleVar_{sanitize_name(ALL_TARGET_COLS_MOMENT[0])}"

    detailed_metrics_path_m = os.path.join(METRICS_MOMENT_OUTPUT_DIR, f'ALL_metrics_detailed_{MOMENT_METHOD_NAME}{sufijo_test_m}.csv')
    df_all_detailed_metrics_moment.to_csv(detailed_metrics_path_m, index=False)
    print(f"  Métricas detalladas MOMENT guardadas en: {detailed_metrics_path_m}")
    if not df_all_detailed_metrics_moment.empty:
        print(df_all_detailed_metrics_moment.head())

    # Summary by Variable
    if not df_all_detailed_metrics_moment.empty:
        grouped_by_var_m = df_all_detailed_metrics_moment.groupby(['City', 'Variable'])[['MAE', 'RMSE', 'NRMAE_range', 'NRMSE_range']].mean().reset_index()
        grouped_by_var_m.rename(columns={'MAE': 'Avg_MAE', 'RMSE': 'Avg_RMSE', 'NRMAE_range': 'Avg_NRMAE_range', 'NRMSE_range': 'Avg_NRMSE_range'}, inplace=True)
        variables_summary_path_m = os.path.join(METRICS_MOMENT_OUTPUT_DIR, f'metrics_variables_summary_{MOMENT_METHOD_NAME}{sufijo_test_m}.csv')
        grouped_by_var_m.to_csv(variables_summary_path_m, index=False)
        print(f"  Resumen MOMENT de variables guardado: {variables_summary_path_m}")
        print(grouped_by_var_m)

        # Summary for Contaminants (if multiple target cols)
        # Using original defs of CONTAMINANT_COLS_MOMENT and AQI_COL_MOMENT for clarity
        df_contaminants_only_m = df_all_detailed_metrics_moment[df_all_detailed_metrics_moment['Variable'].isin(CONTAMINANT_COLS_MOMENT)]
        if not df_contaminants_only_m.empty:
            contaminants_combined_summary_m = df_contaminants_only_m.groupby('City')[['MAE', 'RMSE', 'NRMAE_range', 'NRMSE_range']].mean().reset_index()
            contaminants_combined_summary_m.rename(columns={'MAE': 'Avg_MAE_Conts', 'RMSE': 'Avg_RMSE_Conts', 'NRMAE_range': 'Avg_NRMAE_range_Conts', 'NRMSE_range': 'Avg_NRMSE_range_Conts'}, inplace=True)
            contaminants_summary_path_m = os.path.join(METRICS_MOMENT_OUTPUT_DIR, f'metrics_contaminants_combined_summary_{MOMENT_METHOD_NAME}{sufijo_test_m}.csv')
            contaminants_combined_summary_m.to_csv(contaminants_summary_path_m, index=False)
            print(f"  Resumen MOMENT contaminantes guardado: {contaminants_summary_path_m}")
            print(contaminants_combined_summary_m)

        # Summary for AQI (if AQI is a target col)
        if AQI_COL_MOMENT[0] in ALL_TARGET_COLS_MOMENT:
            df_aqi_only_m = df_all_detailed_metrics_moment[df_all_detailed_metrics_moment['Variable'] == AQI_COL_MOMENT[0]]
            if not df_aqi_only_m.empty:
                aqi_combined_summary_m = df_aqi_only_m.groupby('City')[['MAE', 'RMSE', 'NRMAE_range', 'NRMSE_range']].mean().reset_index()
                aqi_combined_summary_m.rename(columns={'MAE': 'Avg_MAE_AQI', 'RMSE': 'Avg_RMSE_AQI', 'NRMAE_range': 'Avg_NRMAE_range_AQI', 'NRMSE_range': 'Avg_NRMSE_range_AQI'}, inplace=True)
                aqi_summary_path_m = os.path.join(METRICS_MOMENT_OUTPUT_DIR, f'metrics_aqi_combined_summary_{MOMENT_METHOD_NAME}{sufijo_test_m}.csv')
                aqi_combined_summary_m.to_csv(aqi_summary_path_m, index=False)
                print(f"  Resumen MOMENT AQI guardado: {aqi_summary_path_m}")
                print(aqi_combined_summary_m)
else:
    print("No se generaron métricas detalladas para MOMENT.")

script_end_time_moment = time.time()
print(f"\nProceso de Imputación con MOMENT Preentrenado completado. Tiempo total: {(script_end_time_moment - script_start_time_moment) / 60:.2f} minutos.")

Datos originales (ground truth) cargados desde: CSVs\Air_Quality.csv
Usando dispositivo: cpu
Cargando modelo MOMENT pre-entrenado: AutonLab/MOMENT-1-large
Modelo MOMENT pre-entrenado cargado, INICIALIZADO y movido al dispositivo.

Procesando Escenario: RandomRows_10percent
  Repetición: 1


  Imputando variables para Rep 1: 100%|██████████| 7/7 [00:25<00:00,  3.58s/var]


    Archivo imputado por MOMENT guardado: Brasilia_RandomRows_10percent_rep1_seed2377262672_imputed_momentALL.csv
    Métricas calculadas para MOMENT, rep 1
  Repetición: 2


  Imputando variables para Rep 2: 100%|██████████| 7/7 [00:19<00:00,  2.81s/var]


    Archivo imputado por MOMENT guardado: Brasilia_RandomRows_10percent_rep2_seed2500708800_imputed_momentALL.csv
    Métricas calculadas para MOMENT, rep 2
  Repetición: 3


...

    Archivo imputado por MOMENT guardado: Brasilia_BlockRows_24h_rep3_seed473312285_imputed_momentALL.csv
    Métricas calculadas para MOMENT, rep 3
  Repetición: 4


  Imputando variables para Rep 4: 100%|██████████| 7/7 [00:20<00:00,  2.94s/var]


    Archivo imputado por MOMENT guardado: Brasilia_BlockRows_24h_rep4_seed587731976_imputed_momentALL.csv
    Métricas calculadas para MOMENT, rep 4
  Repetición: 5


  Imputando variables para Rep 5: 100%|██████████| 7/7 [00:22<00:00,  3.19s/var]

    Archivo imputado por MOMENT guardado: Brasilia_BlockRows_24h_rep5_seed704623400_imputed_momentALL.csv
    Métricas calculadas para MOMENT, rep 5
  Escenario BlockRows_24h completado.

Generando resúmenes de métricas MOMENT...
  Métricas detalladas MOMENT guardadas en: c:\Users\felix\Documents\MÁSTER\TFM\TFM_Ambiental\AIR QUALITY\imputaciones\MOMENT_PRE_Fix7_FinalCorrected\metricas\ALL_metrics_detailed_MOMENT_PRE_Fix7_FinalCorrected_Brasilia.csv
       City              Scenario  Repetition        Seed  \
0  Brasilia  RandomRows_10percent           1  2377262672   
1  Brasilia  RandomRows_10percent           1  2377262672   
2  Brasilia  RandomRows_10percent           1  2377262672   
3  Brasilia  RandomRows_10percent           1  2377262672   
4  Brasilia  RandomRows_10percent           1  2377262672   

                Imputation_Method Variable        MAE       RMSE  NRMAE_range  \
0  MOMENT_PRE_Fix7_FinalCorrected       CO  53.365806  81.704448     0.060369   
1  MOMENT_PRE_Fix7

MOMENT_FT

In [ ]:
import pandas as pd
import numpy as np
import os
import torch
from torch.utils.data import DataLoader, Dataset
import torch.nn as nn
from torch.optim.lr_scheduler import ReduceLROnPlateau
from tqdm import tqdm
import joblib
import traceback
import time
# import shutil # Not strictly needed if using model_save_path directly
# import tempfile # Not strictly needed with current save logic
import random
import warnings

# Attempt to import MOMENT, handle if not found
try:
    from momentfm import MOMENTPipeline
except ImportError:
    print("MOMENTfm library not found. Please install it: pip install momentfm")
    MOMENTPipeline = None # Placeholder

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)


# --- Helper Function for Sanitizing File/Path Names ---
def sanitize_name(name_str):
    import re
    # Convert to string, remove non-alphanumeric (keep underscores), replace spaces
    s = str(name_str)
    s = re.sub(r'[^\w\s-]', '', s).strip() # Remove special chars except underscore, hyphen
    s = re.sub(r'[-\s]+', '_', s)       # Replace spaces/hyphens with underscore
    s = re.sub(r'_+', '_', s)           # Replace multiple underscores with single
    return s if s else "default_sanitized_name"


# --- 0. Carga de Datos Originales (Ground Truth) ---
TARGET_CITY_FT = "Brasilia" # Define early for GT loading
try:
    file_path_load_orig = os.path.join('CSVs', 'Air_Quality.csv')
    if not os.path.exists(file_path_load_orig):
        script_dir_parent_orig = os.path.dirname(os.getcwd())
        alt_file_path_load_orig = os.path.join(script_dir_parent_orig, 'CSVs', 'Air_Quality.csv')
        if os.path.exists(alt_file_path_load_orig): file_path_load_orig = alt_file_path_load_orig
        else: raise FileNotFoundError("Air_Quality.csv (original) no encontrado.")
    df_air_quality_ground_truth_raw = pd.read_csv(file_path_load_orig)
    print(f"Datos originales (ground truth) cargados desde: {file_path_load_orig}")
except FileNotFoundError:
    print("Error: Air_Quality.csv (original) no encontrado. Creando DataFrame de ejemplo...")
    _example_data = []
    _cities_example = [TARGET_CITY_FT]
    _start_date = pd.to_datetime('2024-01-01')
    for _city_ex in _cities_example:
        num_rows_example = 550
        for i in range(num_rows_example):
            _date = _start_date + pd.Timedelta(hours=i)
            _example_data.append([_date, _city_ex, 200+np.random.rand()*100, 20+np.random.rand()*10, 5+np.random.rand()*5, 50+np.random.rand()*30, 15+np.random.rand()*10, 30+np.random.rand()*20, 40+np.random.rand()*20])
    df_air_quality_ground_truth_raw = pd.DataFrame(_example_data, columns=['Date', 'City', 'CO', 'NO2', 'SO2', 'O3', 'PM2.5', 'PM10', 'AQI'])
    print("DataFrame de ejemplo creado.")

# Ensure 'Date' is datetime and UTC, then localize for the TARGET_CITY_FT
df_air_quality_ground_truth_raw['Date'] = pd.to_datetime(df_air_quality_ground_truth_raw['Date'], utc=True)
if 'CO2' in df_air_quality_ground_truth_raw.columns:
    df_air_quality_ground_truth_raw = df_air_quality_ground_truth_raw.drop(columns=['CO2'])

df_air_quality_ground_truth_raw['City'] = df_air_quality_ground_truth_raw['City'].replace('New_York', 'New York')

TIME_FEATURES = ['Hour', 'DayOfWeek', 'Month']
city_timezones = {
    'Brasilia': 'America/Sao_Paulo', 'Cairo': 'Africa/Cairo', 'Dubai': 'Asia/Dubai',
    'London': 'Europe/London', 'New York': 'America/New_York', 'Sydney': 'Australia/Sydney'
}

df_air_quality_ground_truth_localized = pd.DataFrame()
if TARGET_CITY_FT in city_timezones:
    tz = city_timezones[TARGET_CITY_FT]
    df_city_part_gt = df_air_quality_ground_truth_raw[df_air_quality_ground_truth_raw['City'] == TARGET_CITY_FT].copy()
    if not df_city_part_gt.empty:
        df_city_part_gt['Date'] = df_city_part_gt['Date'].dt.tz_convert(tz)
        df_city_part_gt['Hour'] = df_city_part_gt['Date'].dt.hour
        df_city_part_gt['DayOfWeek'] = df_city_part_gt['Date'].dt.dayofweek
        df_city_part_gt['Month'] = df_city_part_gt['Date'].dt.month
        df_air_quality_ground_truth_localized = df_city_part_gt
        print(f"Ground truth for {TARGET_CITY_FT} localized to {tz} and time features extracted.")
    else:
        print(f"No data for {TARGET_CITY_FT} in ground truth for timezone conversion.")
else:
    print(f"Timezone not defined for {TARGET_CITY_FT}. Using original dates for GT and attempting to extract time features.")
    df_air_quality_ground_truth_localized = df_air_quality_ground_truth_raw[df_air_quality_ground_truth_raw['City'] == TARGET_CITY_FT].copy()
    if not df_air_quality_ground_truth_localized.empty and 'Date' in df_air_quality_ground_truth_localized.columns:
        df_air_quality_ground_truth_localized['Date'] = pd.to_datetime(df_air_quality_ground_truth_localized['Date']) # Ensure datetime
        df_air_quality_ground_truth_localized['Hour'] = df_air_quality_ground_truth_localized['Date'].dt.hour
        df_air_quality_ground_truth_localized['DayOfWeek'] = df_air_quality_ground_truth_localized['Date'].dt.dayofweek
        df_air_quality_ground_truth_localized['Month'] = df_air_quality_ground_truth_localized['Date'].dt.month
        print(f"Time features extracted for {TARGET_CITY_FT} from non-localized dates.")

if df_air_quality_ground_truth_localized.empty:
    print(f"Error: Ground truth for {TARGET_CITY_FT} is empty. Exiting.")
    exit()


# --- Parámetros Configurables para Fine-Tuning ---
CONTAMINANT_COLS_FT = ['CO', 'NO2', 'SO2', 'O3', 'PM2.5', 'PM10']
AQI_COL_FT = ['AQI']
ALL_TARGET_COLS_FT = CONTAMINANT_COLS_FT + AQI_COL_FT

SEQ_LEN_MODEL_FT = 128 # Changed from 512, more common for fine-tuning
IMPUTATION_STEP_FT = SEQ_LEN_MODEL_FT # Non-overlapping for imputation
BATCH_SIZE_IMPUTE_FT = 16
MODEL_NAME_PRETRAINED_FT = "AutonLab/MOMENT-1-base" # Changed to base model

FINETUNE_EPOCHS = 10 # Reduced for faster run, adjust as needed
FINETUNE_STEP = 64   # Step for creating fine-tuning sequences (can overlap)
BATCH_SIZE_FINETUNE = 8
LEARNING_RATE_FINETUNE = 1e-5 # Often lower for fine-tuning
PATIENCE_EARLY_STOPPING = 3 # Reduced for faster run
MIN_DELTA_EARLY_STOPPING = 1e-5

N_REPETITIONS_FT_EVAL = 1 # Changed to 1 repetition
NAN_SIMULATION_SCENARIOS_FT = [ # Scenarios should match VAE script's output
    {'name': 'RandomRows_20percent'},
    {'name': 'BlockRows_12h_20percent'},
    {'name': 'BlockRows_24h_20percent'},
]

PROJECT_ROOT_DIR_FT = os.getcwd()
SIMULATED_DATA_INPUT_ROOT_DIR_FT = os.path.join(PROJECT_ROOT_DIR_FT, 'CSVs', 'simulados') # Input from VAE simulation

MOMENT_METHOD_NAME_FT = f'MOMENT_FT_{sanitize_name(MODEL_NAME_PRETRAINED_FT.split("/")[-1])}'
IMPUTATIONS_FT_BASE_DIR = os.path.join(PROJECT_ROOT_DIR_FT, 'imputaciones', MOMENT_METHOD_NAME_FT)
METRICS_FT_OUTPUT_DIR = os.path.join(IMPUTATIONS_FT_BASE_DIR, 'metricas')
IMPUTED_CSVS_FT_OUTPUT_DIR = os.path.join(IMPUTATIONS_FT_BASE_DIR, 'CSVs')
MODELS_FT_OUTPUT_DIR = os.path.join(IMPUTATIONS_FT_BASE_DIR, 'modelos') # For fine-tuned models
SCALERS_FT_OUTPUT_DIR = os.path.join(IMPUTATIONS_FT_BASE_DIR, 'scalers') # For scalers
os.makedirs(METRICS_FT_OUTPUT_DIR, exist_ok=True)
os.makedirs(IMPUTED_CSVS_FT_OUTPUT_DIR, exist_ok=True)
os.makedirs(MODELS_FT_OUTPUT_DIR, exist_ok=True)
os.makedirs(SCALERS_FT_OUTPUT_DIR, exist_ok=True)


# --- Dataset Personalizado para MOMENT (Fine-tuning and Imputation) ---
class MomentDataset(Dataset):
    def __init__(self, data_array_2d, observed_mask_2d, seq_len, step):
        """
        Args:
            data_array_2d (np.array): Data, (total_len, n_channels). For fine-tuning, this is clean, scaled data.
                                      For imputation, this is scaled data with NaNs filled (e.g., with 0).
            observed_mask_2d (np.array): Boolean mask, (total_len, n_channels). True where data is observed.
                                         For fine-tuning, this is all True.
                                         For imputation, True where original data (before filling NaNs) was observed.
            seq_len (int): Sequence length.
            step (int): Step for creating sequences.
        """
        self.data_array = data_array_2d
        self.observed_mask = observed_mask_2d # True for observed
        self.seq_len = seq_len
        self.step = step

        if len(self.data_array) == 0:
            self.n_samples, self.indices = 0, []
        elif len(self.data_array) < self.seq_len :
            self.n_samples, self.indices = 1, [0] # Will pad one sample
        else:
            self.indices = list(range(0, len(self.data_array) - self.seq_len + 1, self.step))
            last_possible_start = len(self.data_array) - self.seq_len
            if not self.indices or self.indices[-1] < last_possible_start :
                if not self.indices or last_possible_start > self.indices[-1]:
                    self.indices.append(last_possible_start)
            self.n_samples = len(self.indices)
            if self.n_samples == 0 and len(self.data_array) >= self.seq_len:
                 self.indices, self.n_samples = [0],1

    def __len__(self):
        return self.n_samples

    def __getitem__(self, idx):
        if self.n_samples == 0:
            raise IndexError("Dataset is empty")

        actual_start_index = self.indices[idx]
        seq_data_values = self.data_array[actual_start_index : actual_start_index + self.seq_len]
        seq_observed_status = self.observed_mask[actual_start_index : actual_start_index + self.seq_len] # (seq_len, n_channels)

        if len(seq_data_values) < self.seq_len:
            pad_width = self.seq_len - len(seq_data_values)
            seq_data_values = np.pad(seq_data_values, ((0, pad_width), (0, 0)), 'constant', constant_values=0.0)
            seq_observed_status = np.pad(seq_observed_status, ((0, pad_width), (0,0)), 'constant', constant_values=False) # Padded values are not observed

        # timeseries: (n_channels, seq_len)
        timeseries_out = torch.tensor(seq_data_values, dtype=torch.float32).transpose(0, 1)

        # input_mask for MOMENT: (seq_len,). True if the time point is considered "observed enough".
        # For fine-tuning with clean data, all are True.
        # For imputation, True if ALL channels at that timestep were originally observed.
        input_mask_per_channel_tensor = torch.tensor(seq_observed_status, dtype=torch.bool)
        input_mask_out = input_mask_per_channel_tensor.all(dim=1) # Shape: (seq_len,)

        return {
            'timeseries': timeseries_out,    # (n_channels, seq_len)
            'input_mask': input_mask_out     # (seq_len,)
        }


# --- Función de Entrenamiento (Fine-Tuning) ---
def train_moment_model_finetune(model, train_loader, device, epochs, variable_name, model_save_path_ft):
    model.train() # Set model to training mode
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE_FINETUNE)
    # Using MSE loss, as MOMENT's reconstruction is continuous
    criterion = nn.MSELoss()
    scheduler = ReduceLROnPlateau(optimizer, 'min', factor=0.5, patience=PATIENCE_EARLY_STOPPING // 2 + 1, verbose=False) # +1 to ensure patience > 0

    best_loss = float('inf')
    epochs_no_improve = 0
    print(f"Iniciando fine-tuning para '{variable_name}' ({epochs} épocas máx)...")

    for epoch in range(epochs):
        total_epoch_loss = 0
        batch_count = 0
        for batch_data in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} ({variable_name})", leave=False):
            timeseries_batch = batch_data['timeseries'].to(device) # (batch, n_channels, seq_len)
            input_mask_batch = batch_data['input_mask'].to(device) # (batch, seq_len)

            optimizer.zero_grad()
            try:
                # MOMENT's forward pass for reconstruction
                outputs = model(x_enc=timeseries_batch, input_mask=input_mask_batch)
                # Loss is calculated between the reconstruction and the original (clean) timeseries
                loss = criterion(outputs.reconstruction, timeseries_batch)
            except Exception as e_train:
                print(f"    Error durante forward/loss en epoch {epoch+1}, batch {batch_count} para {variable_name}: {e_train}")
                continue # Skip this batch

            if torch.isnan(loss) or torch.isinf(loss):
                print(f"    NaN/Inf loss detectado en epoch {epoch+1} para {variable_name}. Saltando batch.")
                continue

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0) # Gradient clipping
            optimizer.step()
            total_epoch_loss += loss.item()
            batch_count +=1

        if batch_count == 0:
            print(f"  Epoch {epoch+1} ({variable_name}): No batches procesados. Saltando época.")
            avg_epoch_loss = float('inf') # Or handle as error
        else:
            avg_epoch_loss = total_epoch_loss / batch_count

        if epoch == 0 or (epoch + 1) % 2 == 0 or epochs_no_improve > 0: # Print more often or if loss increases
            print(f"  Epoch {epoch+1} Avg Loss ({variable_name}): {avg_epoch_loss:.6f} (LR: {optimizer.param_groups[0]['lr']:.2e})")

        scheduler.step(avg_epoch_loss)

        if avg_epoch_loss < best_loss - MIN_DELTA_EARLY_STOPPING:
            best_loss = avg_epoch_loss
            epochs_no_improve = 0
            torch.save(model.state_dict(), model_save_path_ft) # Save best model
            print(f"    Nuevo mejor modelo guardado para {variable_name} con pérdida: {best_loss:.6f}")
        else:
            epochs_no_improve += 1

        if epochs_no_improve >= PATIENCE_EARLY_STOPPING:
            print(f"  Early stopping activado en epoch {epoch+1} para {variable_name}.")
            break

    print(f"Fine-tuning para '{variable_name}' completado. Mejor pérdida final: {best_loss:.6f}")
    # Load the best model saved
    if os.path.exists(model_save_path_ft):
        try:
            model.load_state_dict(torch.load(model_save_path_ft, map_location=device))
            print(f"  Mejor modelo para '{variable_name}' cargado desde {model_save_path_ft}")
        except Exception as e_load_best:
            print(f"  Error cargando el mejor modelo para {variable_name}: {e_load_best}. Usando el último estado.")
    else:
        print(f"  Advertencia: No se encontró el archivo del mejor modelo en {model_save_path_ft}. Usando el último estado del modelo.")
    return model


# --- Función de Imputación con Modelo Fine-tuneado (Univariado por ahora) ---
def impute_univar_with_finetuned_model(
    df_univar_with_nans_scaled_zeros, # DataFrame (n_rows, 1 col), scaled, NaNs filled with 0
    original_nan_mask_univar,      # Series/DataFrame (n_rows, 1 col), True where original was NaN
    model_ft_var,                     # Fine-tuned MOMENT model for this variable
    device_str,
    scaler_var,                       # Scaler for this variable
    seq_len_imp,
    step_imp,
    batch_size_imp
):
    model_ft_var.eval() # Set model to evaluation mode

    # Create dataset for imputation
    # observed_mask should be True where original data was present
    imputation_dataset = MomentDataset(
        df_univar_with_nans_scaled_zeros.values, # (total_len, 1 channel)
        ~original_nan_mask_univar.values,       # (total_len, 1 channel), True for observed
        seq_len_imp,
        step_imp
    )
    if len(imputation_dataset) == 0:
        print("  MomentDataset para imputación resultó en 0 samples.")
        return None # Or return original df filled with ffill/bfill

    imputation_loader = DataLoader(imputation_dataset, batch_size=batch_size_imp, shuffle=False, num_workers=0)

    n_rows_total, n_features_total = df_univar_with_nans_scaled_zeros.shape # n_features_total will be 1
    accumulated_preds_scaled = np.zeros((n_rows_total, n_features_total), dtype=float)
    accumulated_counts = np.zeros((n_rows_total, n_features_total), dtype=float)

    with torch.no_grad():
        for batch_idx, batch_data in enumerate(imputation_loader):
            timeseries_batch_tensor = batch_data['timeseries'].to(device_str) # (batch, 1, seq_len)
            input_mask_batch_tensor = batch_data['input_mask'].to(device_str) # (batch, seq_len)
            preds_batch_scaled_np = None
            try:
                outputs_obj = model_ft_var(x_enc=timeseries_batch_tensor, input_mask=input_mask_batch_tensor)
                preds_batch_scaled_np = outputs_obj.reconstruction.cpu().numpy() # (batch, 1, seq_len)
            except Exception as e_pred:
                print(f"    Error durante predicción de imputación para batch {batch_idx}: {e_pred}")
                continue

            if preds_batch_scaled_np is None:
                continue

            for i_in_batch_loop in range(preds_batch_scaled_np.shape[0]):
                current_dataset_idx = batch_idx * batch_size_imp + i_in_batch_loop
                if current_dataset_idx >= len(imputation_loader.dataset.indices):
                    continue

                start_idx_in_series = imputation_loader.dataset.indices[current_dataset_idx]
                end_idx_in_series = min(start_idx_in_series + seq_len_imp, n_rows_total)
                len_window_actual = end_idx_in_series - start_idx_in_series

                if len_window_actual > 0:
                    pred_window_scaled_loop = preds_batch_scaled_np[i_in_batch_loop, :, :len_window_actual].T # (len_actual, 1)
                    accumulated_preds_scaled[start_idx_in_series:end_idx_in_series, :] += pred_window_scaled_loop
                    accumulated_counts[start_idx_in_series:end_idx_in_series, :] += 1

    accumulated_counts[accumulated_counts == 0] = 1 # Avoid division by zero
    imputed_full_series_scaled = accumulated_preds_scaled / accumulated_counts

    # Inverse transform the imputed series
    imputed_full_series_unscaled = scaler_var.inverse_transform(imputed_full_series_scaled)

    # Start with a copy of the original unscaled data (with NaNs)
    df_final_imputed_unscaled = pd.DataFrame(
        scaler_var.inverse_transform(df_univar_with_nans_scaled_zeros.values), # Unscale the 0-filled data first
        columns=df_univar_with_nans_scaled_zeros.columns,
        index=df_univar_with_nans_scaled_zeros.index
    )

    # Fill in the imputed values (unscaled) only at the positions that were originally NaN
    # original_nan_mask_univar is a DataFrame (n_rows, 1 col)
    nan_mask_this_col_flat = original_nan_mask_univar.iloc[:,0].values # Flatten to 1D boolean array
    col_name = df_univar_with_nans_scaled_zeros.columns[0]
    df_final_imputed_unscaled.loc[nan_mask_this_col_flat, col_name] = imputed_full_series_unscaled[nan_mask_this_col_flat, 0]

    return df_final_imputed_unscaled


# --- Función de Métricas (re-using from previous script) ---
def calculate_metrics_moment(y_true_df_reset, y_pred_df_reset, nan_mask_for_metrics_df_reset, cols_to_eval):
    results = {}
    for col in cols_to_eval:
        results[col] = {'MAE': np.nan, 'RMSE': np.nan, 'NRMAE_range': np.nan, 'NRMSE_range': np.nan}
        if col not in y_true_df_reset.columns or col not in y_pred_df_reset.columns or col not in nan_mask_for_metrics_df_reset.columns:
            continue

        true_col_vals = y_true_df_reset[col].values
        pred_col_vals = y_pred_df_reset[col].values
        mask_col_flags = nan_mask_for_metrics_df_reset[col].values.astype(bool)

        min_len = min(len(true_col_vals), len(pred_col_vals), len(mask_col_flags))
        true_col_vals = true_col_vals[:min_len]
        pred_col_vals = pred_col_vals[:min_len]
        mask_col_flags = mask_col_flags[:min_len]

        true_at_nans = true_col_vals[mask_col_flags]
        pred_at_nans = pred_col_vals[mask_col_flags]

        if len(true_at_nans) == 0: continue
        valid_pred_mask = ~np.isnan(pred_at_nans)
        true_at_nans_valid = true_at_nans[valid_pred_mask]
        pred_at_nans_valid = pred_at_nans[valid_pred_mask]

        if len(true_at_nans_valid) == 0: continue
        try:
            mae = mean_absolute_error(true_at_nans_valid, pred_at_nans_valid)
            rmse = np.sqrt(mean_squared_error(true_at_nans_valid, pred_at_nans_valid))
            original_col_full_series = y_true_df_reset[col].dropna()
            original_col_range = 0
            if not original_col_full_series.empty:
                original_col_range = original_col_full_series.max() - original_col_full_series.min()
            nrmae_range = (mae / original_col_range if original_col_range > 1e-9 else np.nan)
            nrmse_range = (rmse / original_col_range if original_col_range > 1e-9 else np.nan)
            results[col].update({'MAE': mae, 'RMSE': rmse, 'NRMAE_range': nrmae_range, 'NRMSE_range': nrmse_range})
        except Exception: pass
    return results

# --- Script Principal ---
if MOMENTPipeline is None:
    print("MOMENTPipeline could not be imported. Exiting script.")
    exit()

all_detailed_metrics_ft_results = []
script_start_time_ft = time.time()
device = "cuda" if torch.cuda.is_available() else "cpu"; print(f"Usando dispositivo: {device}")

# Use the localized ground truth
df_city_ground_truth_ft_reset = df_air_quality_ground_truth_localized.reset_index(drop=True)

# Fill any remaining NaNs in GT (e.g., at start/end) for robust scaler fitting AND fine-tuning
for col_gt_fill in ALL_TARGET_COLS_FT:
    if col_gt_fill in df_city_ground_truth_ft_reset.columns:
        df_city_ground_truth_ft_reset[col_gt_fill] = df_city_ground_truth_ft_reset[col_gt_fill].ffill().bfill()
if df_city_ground_truth_ft_reset[ALL_TARGET_COLS_FT].isnull().any().any():
    print(f"Error: No se pudo rellenar completamente el Ground Truth para {TARGET_CITY_FT}. Revisar datos."); exit()


# --- Fine-Tuning Phase (Univariate per target column) ---
finetuned_models_dict = {} # Stores fine-tuned model for each variable
scalers_for_vars_dict = {}   # Stores scaler for each variable

city_name_safe_ft_models = sanitize_name(TARGET_CITY_FT)
city_models_dir = os.path.join(MODELS_FT_OUTPUT_DIR, city_name_safe_ft_models)
os.makedirs(city_models_dir, exist_ok=True)
city_scalers_dir = os.path.join(SCALERS_FT_OUTPUT_DIR, city_name_safe_ft_models) # For scalers
os.makedirs(city_scalers_dir, exist_ok=True)


for var_to_finetune in ALL_TARGET_COLS_FT:
    print(f"\n--- Iniciando Fine-Tuning para la variable: {var_to_finetune} ---")
    
    # 1. Prepare Scaler for this variable
    scaler_current_var = StandardScaler()
    # Use the clean, localized, and filled ground truth data for fitting the scaler
    data_for_scaler_fit = df_city_ground_truth_ft_reset[[var_to_finetune]].values
    if len(data_for_scaler_fit) < 1: # Should have data due to earlier checks
        print(f"  Datos insuficientes para el scaler de {var_to_finetune}. Saltando."); continue
    scaler_current_var.fit(data_for_scaler_fit)
    scalers_for_vars_dict[var_to_finetune] = scaler_current_var
    joblib.dump(scaler_current_var, os.path.join(city_scalers_dir, f'scaler_ft_{sanitize_name(var_to_finetune)}.joblib'))


    # 2. Prepare Data for Fine-tuning (scaled, no NaNs)
    # The data for fine-tuning should be the clean ground truth, scaled.
    train_data_unscaled_var = df_city_ground_truth_ft_reset[[var_to_finetune]].values # (n_rows, 1)
    train_data_scaled_var = scaler_current_var.transform(train_data_unscaled_var)    # (n_rows, 1)
    
    # For fine-tuning, the observed_mask is all True as we use clean data
    observed_mask_for_finetune = np.ones_like(train_data_scaled_var, dtype=bool) # (n_rows, 1)

    if len(train_data_scaled_var) < SEQ_LEN_MODEL_FT: # Check if enough data for at least one sequence
        print(f"  Datos insuficientes para crear secuencias de fine-tuning para {var_to_finetune} (len: {len(train_data_scaled_var)}, seq_len: {SEQ_LEN_MODEL_FT}). Saltando.")
        continue

    train_dataset_ft = MomentDataset(
        train_data_scaled_var,      # Data for training (scaled)
        observed_mask_for_finetune, # Mask (all True)
        SEQ_LEN_MODEL_FT,           # Sequence length for fine-tuning
        FINETUNE_STEP               # Step for creating sequences
    )
    if len(train_dataset_ft) == 0:
        print(f"  El Dataset de fine-tuning para {var_to_finetune} está vacío. Saltando."); continue
    train_loader_ft = DataLoader(train_dataset_ft, batch_size=BATCH_SIZE_FINETUNE, shuffle=True, num_workers=0)

    # 3. Load Pre-trained MOMENT model for fine-tuning
    model_to_finetune = None
    try:
        model_to_finetune = MOMENTPipeline.from_pretrained(MODEL_NAME_PRETRAINED_FT, model_kwargs={'task_name': 'reconstruction'})
        model_to_finetune.init()
        model_to_finetune = model_to_finetune.to(device)
    except Exception as e_load_ft:
        print(f"  Error cargando modelo MOMENT para fine-tuning de {var_to_finetune}: {e_load_ft}. Saltando variable.")
        continue

    # 4. Perform Fine-tuning
    model_save_path_current_var = os.path.join(city_models_dir, f'moment_ft_{sanitize_name(var_to_finetune)}.pth')
    
    finetuned_model_for_var = train_moment_model_finetune(
        model_to_finetune,
        train_loader_ft,
        device,
        FINETUNE_EPOCHS,
        var_to_finetune,
        model_save_path_current_var
    )
    finetuned_models_dict[var_to_finetune] = finetuned_model_for_var
    print(f"--- Fine-Tuning para {var_to_finetune} completado. ---")

print("\n\n--- Fine-Tuning completado para todas las variables objetivo. ---")

# --- Imputation Phase using Fine-tuned Models ---
city_name_safe_ft_impute = sanitize_name(TARGET_CITY_FT)
city_imputed_csv_ft_dir_root = os.path.join(IMPUTED_CSVS_FT_OUTPUT_DIR, city_name_safe_ft_impute)
os.makedirs(city_imputed_csv_ft_dir_root, exist_ok=True)

for scenario_ft in NAN_SIMULATION_SCENARIOS_FT:
    scenario_name_for_dir = scenario_ft['name'].replace("%", "percent")
    scenario_name_for_file_part = scenario_ft['name'].replace("%", "percent")
    print(f"\nProcesando Escenario de Imputación: {scenario_name_for_dir}")

    sim_data_scenario_dir_ft = os.path.join(SIMULATED_DATA_INPUT_ROOT_DIR_FT, city_name_safe_ft_impute, scenario_name_for_dir)
    imputed_ft_scenario_dir = os.path.join(city_imputed_csv_ft_dir_root, sanitize_name(scenario_name_for_dir))
    os.makedirs(imputed_ft_scenario_dir, exist_ok=True)

    if not os.path.exists(sim_data_scenario_dir_ft):
        print(f"    Directorio de datos simulados no encontrado: {sim_data_scenario_dir_ft}. Omitiendo escenario."); continue

    for rep_ft in range(1, N_REPETITIONS_FT_EVAL + 1):
        print(f"  Repetición de Imputación: {rep_ft}")
        file_pattern_to_find_imp = f"{city_name_safe_ft_impute}_{scenario_name_for_file_part}_rep{rep_ft}_*_with_nans.csv"
        sim_files_in_dir_ft = sorted([
            f for f in os.listdir(sim_data_scenario_dir_ft)
            if f.startswith(f"{city_name_safe_ft_impute}_{scenario_name_for_file_part}_rep{rep_ft}") and f.endswith("_with_nans.csv")
        ])

        if not sim_files_in_dir_ft:
            print(f"    Archivo simulado no encontrado para patrón '{file_pattern_to_find_imp}' en '{sim_data_scenario_dir_ft}'. Omitiendo."); continue
        
        sim_file_path_to_load_ft = os.path.join(sim_data_scenario_dir_ft, sim_files_in_dir_ft[0])
        current_seed_loaded_ft = "unknown"
        try:
            current_seed_loaded_ft = sim_files_in_dir_ft[0].split('_seed')[-1].split('_with_nans.csv')[0]
        except IndexError:
            print(f"    Advertencia: No se pudo extraer la semilla de: {sim_files_in_dir_ft[0]}")

        try:
            df_city_with_nans_loaded_ft = pd.read_csv(sim_file_path_to_load_ft)
            if 'Date' in df_city_with_nans_loaded_ft.columns:
                 df_city_with_nans_loaded_ft['Date'] = pd.to_datetime(df_city_with_nans_loaded_ft['Date'])
        except Exception as e_load_rep_ft:
            print(f"    Error cargando archivo simulado {sim_file_path_to_load_ft}: {e_load_rep_ft}. Omitiendo."); continue

        # This will hold the imputed columns, starting as a copy of the loaded data
        df_imputed_all_cols_rep_ft = df_city_with_nans_loaded_ft.copy()

        for var_to_impute in tqdm(ALL_TARGET_COLS_FT, desc=f"  Imputando variables para Rep {rep_ft}", unit="var", leave=False):
            if var_to_impute not in finetuned_models_dict or var_to_impute not in scalers_for_vars_dict:
                print(f"    Modelo fine-tuneado o scaler no disponible para {var_to_impute}. Saltando imputación para esta variable.")
                # Fallback: ffill/bfill for this variable if model/scaler is missing
                if var_to_impute in df_imputed_all_cols_rep_ft.columns:
                    df_imputed_all_cols_rep_ft[var_to_impute] = df_imputed_all_cols_rep_ft[var_to_impute].ffill().bfill()
                continue
            
            model_ft_to_use = finetuned_models_dict[var_to_impute]
            scaler_var_to_use = scalers_for_vars_dict[var_to_impute]

            # Prepare data for this specific variable for imputation
            df_univar_to_impute_with_nans = df_city_with_nans_loaded_ft[[var_to_impute]].copy() # (n_rows, 1)
            original_nan_mask_univar_df = df_univar_to_impute_with_nans.isnull() # (n_rows, 1), True for NaNs

            # Scale data, filling NaNs with 0 before scaling (as MOMENT expects non-NaN input)
            # This is a common approach: scale based on observed, then fill NaNs in scaled domain.
            # However, MOMENT dataset expects data_array_2d to be already filled.
            df_univar_filled_for_scaling = df_univar_to_impute_with_nans.fillna(0) # Fill NaNs with 0 in original scale first
            df_univar_scaled_zeros = pd.DataFrame(
                scaler_var_to_use.transform(df_univar_filled_for_scaling.values),
                columns=[var_to_impute],
                index=df_univar_to_impute_with_nans.index
            )

            df_imputed_univar_unscaled = impute_univar_with_finetuned_model(
                df_univar_scaled_zeros,       # Scaled, 0-filled data for MOMENT
                original_nan_mask_univar_df,  # Mask of original NaNs
                model_ft_to_use,
                device,
                scaler_var_to_use,
                SEQ_LEN_MODEL_FT,      # Use same seq_len as fine-tuning for imputation
                IMPUTATION_STEP_FT,    # Step for imputation
                BATCH_SIZE_IMPUTE_FT
            )
            
            if df_imputed_univar_unscaled is not None and var_to_impute in df_imputed_univar_unscaled.columns:
                # Update the main DataFrame with the imputed values for this variable
                df_imputed_all_cols_rep_ft[var_to_impute] = df_imputed_univar_unscaled[var_to_impute]
            else:
                print(f"    Imputación falló para {var_to_impute}. Usando ffill/bfill como fallback.")
                if var_to_impute in df_imputed_all_cols_rep_ft.columns: # Should exist
                    df_imputed_all_cols_rep_ft[var_to_impute] = df_imputed_all_cols_rep_ft[var_to_impute].ffill().bfill()


        # Save the imputed DataFrame
        df_to_save_final_ft = df_imputed_all_cols_rep_ft.copy()
        final_cols_for_saving_ft = ['Date', 'City'] + ALL_TARGET_COLS_FT
        for tf_col in TIME_FEATURES: # Add time features back from GT
            if tf_col in df_city_ground_truth_ft_reset.columns:
                df_to_save_final_ft[tf_col] = df_city_ground_truth_ft_reset[tf_col].values[:len(df_to_save_final_ft)]
                if tf_col not in final_cols_for_saving_ft:
                    final_cols_for_saving_ft.append(tf_col)
        df_to_save_final_ft = df_to_save_final_ft[[c for c in final_cols_for_saving_ft if c in df_to_save_final_ft.columns]]


        imputed_ft_filename = f"{city_name_safe_ft_impute}_{sanitize_name(scenario_name_for_file_part)}_rep{rep_ft}_seed{current_seed_loaded_ft}_imputed_momentFT.csv"
        imputed_ft_file_save_path = os.path.join(imputed_ft_scenario_dir, imputed_ft_filename)
        df_to_save_final_ft.to_csv(imputed_ft_file_save_path, index=False)
        print(f"    Archivo imputado por MOMENT-FT guardado: {imputed_ft_filename}")
        
        # --- Metrics Calculation ---
        nan_mask_introduced_for_metrics_rep_ft = pd.DataFrame(False, index=df_city_ground_truth_ft_reset.index, columns=ALL_TARGET_COLS_FT)
        df_loaded_nans_reset_ft = df_city_with_nans_loaded_ft.reset_index(drop=True)
        for col_m_mask in ALL_TARGET_COLS_FT:
            if col_m_mask in df_loaded_nans_reset_ft.columns and col_m_mask in df_city_ground_truth_ft_reset.columns:
                len_min_mask_ft = min(len(df_loaded_nans_reset_ft), len(df_city_ground_truth_ft_reset))
                mask_series_metric_ft = (df_loaded_nans_reset_ft[col_m_mask].iloc[:len_min_mask_ft].isnull() &
                                         ~df_city_ground_truth_ft_reset[col_m_mask].iloc[:len_min_mask_ft].isnull())
                nan_mask_introduced_for_metrics_rep_ft[col_m_mask].iloc[:len_min_mask_ft] = mask_series_metric_ft
        
        metrics_scores_ft_rep = calculate_metrics_moment(
            df_city_ground_truth_ft_reset,
            df_to_save_final_ft.reset_index(drop=True),
            nan_mask_introduced_for_metrics_rep_ft,
            ALL_TARGET_COLS_FT
        )
        
        for var_name_metric_ft in ALL_TARGET_COLS_FT:
            all_detailed_metrics_ft_results.append({
                'City': TARGET_CITY_FT,
                'Scenario': scenario_name_for_dir,
                'Repetition': rep_ft,
                'Seed': current_seed_loaded_ft,
                'Imputation_Method': MOMENT_METHOD_NAME_FT,
                'Variable': var_name_metric_ft,
                'MAE': metrics_scores_ft_rep.get(var_name_metric_ft, {}).get('MAE', np.nan),
                'RMSE': metrics_scores_ft_rep.get(var_name_metric_ft, {}).get('RMSE', np.nan),
                'NRMAE_range': metrics_scores_ft_rep.get(var_name_metric_ft, {}).get('NRMAE_range', np.nan),
                'NRMSE_range': metrics_scores_ft_rep.get(var_name_metric_ft, {}).get('NRMSE_range', np.nan)
            })
        print(f"    Métricas calculadas para MOMENT-FT, rep {rep_ft}")
    print(f"  Escenario de Imputación {scenario_name_for_dir} completado.")


# --- Guardado de Métricas Agregadas ---
if all_detailed_metrics_ft_results:
    df_all_detailed_metrics_ft = pd.DataFrame(all_detailed_metrics_ft_results)
    print("\nGenerando resúmenes de métricas MOMENT-FT...")
    sufijo_test_ft = f"_{sanitize_name(TARGET_CITY_FT)}"

    detailed_metrics_path_ft = os.path.join(METRICS_FT_OUTPUT_DIR, f'ALL_metrics_detailed_{MOMENT_METHOD_NAME_FT}{sufijo_test_ft}.csv')
    df_all_detailed_metrics_ft.to_csv(detailed_metrics_path_ft, index=False)
    print(f"  Métricas detalladas MOMENT-FT guardadas en: {detailed_metrics_path_ft}")
    if not df_all_detailed_metrics_ft.empty: print(df_all_detailed_metrics_ft.head())

    if not df_all_detailed_metrics_ft.empty:
        grouped_by_var_ft = df_all_detailed_metrics_ft.groupby(['City', 'Variable'])[['MAE', 'RMSE', 'NRMAE_range', 'NRMSE_range']].mean().reset_index()
        grouped_by_var_ft.rename(columns={'MAE': 'Avg_MAE', 'RMSE': 'Avg_RMSE', 'NRMAE_range': 'Avg_NRMAE_range', 'NRMSE_range': 'Avg_NRMSE_range'}, inplace=True)
        variables_summary_path_ft = os.path.join(METRICS_FT_OUTPUT_DIR, f'metrics_variables_summary_{MOMENT_METHOD_NAME_FT}{sufijo_test_ft}.csv')
        grouped_by_var_ft.to_csv(variables_summary_path_ft, index=False)
        print(f"  Resumen MOMENT-FT de variables guardado: {variables_summary_path_ft}"); print(grouped_by_var_ft)

        df_contaminants_only_ft = df_all_detailed_metrics_ft[df_all_detailed_metrics_ft['Variable'].isin(CONTAMINANT_COLS_FT)]
        if not df_contaminants_only_ft.empty:
            contaminants_combined_summary_ft = df_contaminants_only_ft.groupby('City')[['MAE', 'RMSE', 'NRMAE_range', 'NRMSE_range']].mean().reset_index()
            contaminants_combined_summary_ft.rename(columns={'MAE': 'Avg_MAE_Conts', 'RMSE': 'Avg_RMSE_Conts', 'NRMAE_range': 'Avg_NRMAE_range_Conts', 'NRMSE_range': 'Avg_NRMSE_range_Conts'}, inplace=True)
            contaminants_summary_path_ft = os.path.join(METRICS_FT_OUTPUT_DIR, f'metrics_contaminants_combined_summary_{MOMENT_METHOD_NAME_FT}{sufijo_test_ft}.csv')
            contaminants_combined_summary_ft.to_csv(contaminants_summary_path_ft, index=False)
            print(f"  Resumen MOMENT-FT contaminantes guardado: {contaminants_summary_path_ft}"); print(contaminants_combined_summary_ft)

        if AQI_COL_FT[0] in ALL_TARGET_COLS_FT:
            df_aqi_only_ft = df_all_detailed_metrics_ft[df_all_detailed_metrics_ft['Variable'] == AQI_COL_FT[0]]
            if not df_aqi_only_ft.empty:
                aqi_combined_summary_ft = df_aqi_only_ft.groupby('City')[['MAE', 'RMSE', 'NRMAE_range', 'NRMSE_range']].mean().reset_index()
                aqi_combined_summary_ft.rename(columns={'MAE': 'Avg_MAE_AQI', 'RMSE': 'Avg_RMSE_AQI', 'NRMAE_range': 'Avg_NRMAE_range_AQI', 'NRMSE_range': 'Avg_NRMSE_range_AQI'}, inplace=True)
                aqi_summary_path_ft = os.path.join(METRICS_FT_OUTPUT_DIR, f'metrics_aqi_combined_summary_{MOMENT_METHOD_NAME_FT}{sufijo_test_ft}.csv')
                aqi_combined_summary_ft.to_csv(aqi_summary_path_ft, index=False)
                print(f"  Resumen MOMENT-FT AQI guardado: {aqi_summary_path_ft}"); print(aqi_combined_summary_ft)
else:
    print("No se generaron métricas detalladas para MOMENT-FT.")

script_end_time_ft = time.time()
print(f"\nProceso de Fine-Tuning e Imputación con MOMENT completado. Tiempo total: {(script_end_time_ft - script_start_time_ft) / 60:.2f} minutos.")

Datos originales (ground truth) cargados desde: CSVs\Air_Quality.csv
Usando dispositivo: cpu

--- Preparando Fine-Tuning para la variable: CO ---
Iniciando fine-tuning para 'CO' (20 épocas máx)...


  Epoch 1 Loss (CO): 0.187409


  Epoch 5 Loss (CO): 0.190900


  Early stopping en epoch 6.
Fine-tuning para 'CO' completado. Mejor pérdida: 0.187409
  Mejor modelo para 'CO' cargado desde c:\Users\felix\Documents\MÁSTER\TFM\TFM_Ambiental\AIR QUALITY\imputaciones\MOMENT_FT_v2_Corrected\modelos\Brasilia\moment_ft_CO.pth

--- Preparando Fine-Tuning para la variable: NO2 ---
Iniciando fine-tuning para 'NO2' (20 épocas máx)...


  Epoch 1 Loss (NO2): 0.319694


...

    Archivo imputado por MOMENT-FT guardado: Brasilia_BlockRows_8h_rep3_seed2255836414_imputed_momentFT.csv
    Métricas calculadas para MOMENT-FT, rep 3
  Repetición: 4


  Imputando variables para Rep 4: 100%|██████████| 7/7 [00:23<00:00,  3.29s/var]


    Archivo imputado por MOMENT-FT guardado: Brasilia_BlockRows_8h_rep4_seed2385710956_imputed_momentFT.csv
    Métricas calculadas para MOMENT-FT, rep 4
  Repetición: 5


  Imputando variables para Rep 5: 100%|██████████| 7/7 [00:21<00:00,  3.12s/var]

    Archivo imputado por MOMENT-FT guardado: Brasilia_BlockRows_8h_rep5_seed2503372948_imputed_momentFT.csv
    Métricas calculadas para MOMENT-FT, rep 5
  Escenario BlockRows_8h completado.

Generando resúmenes de métricas MOMENT-FT...
  Métricas detalladas MOMENT-FT guardadas en: c:\Users\felix\Documents\MÁSTER\TFM\TFM_Ambiental\AIR QUALITY\imputaciones\MOMENT_FT_v2_Corrected\metricas\ALL_metrics_detailed_MOMENT_FT_v2_Corrected_Brasilia.csv
       City              Scenario  Repetition        Seed  \
0  Brasilia  RandomRows_10percent           1  2377262672   
1  Brasilia  RandomRows_10percent           1  2377262672   
2  Brasilia  RandomRows_10percent           1  2377262672   
3  Brasilia  RandomRows_10percent           1  2377262672   
4  Brasilia  RandomRows_10percent           1  2377262672   

        Imputation_Method Variable        MAE       RMSE  NRMAE_range  \
0  MOMENT_FT_v2_Corrected       CO  46.566176  73.870741     0.052677   
1  MOMENT_FT_v2_Corrected      NO2   4.6